In [ ]:
#preproc_P2_no_fnirs_filter

In [2]:
# MULTIMODAL PREPROCESSING: EEG + fNIRS 
# Set the paths based on your system before running the code
# you can adjust which subjects to process at the bottom of this cell

import os
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
from scipy.interpolate import CloughTocher2DInterpolator  # NEW: used by the vectorized spatial_mapping()
from scipy.spatial import Delaunay  # NEW: used by the vectorized spatial_mapping()
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt  # NEW: needed for before/after preprocessing plots
import time
import datetime
import json  # NEW: for per-subject metadata.json export (adopted from reviewing a third-party implementation, see chat history)
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
# BASE_DATASET_PATH = r"D:\Research\EEG-fNIRS\project\project\Dataset"
# OUTPUT_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"
# FIX (path update per user's actual folder layout): the user's dataset is laid out as
# DATA\VP001-EEG, DATA\VP001-NIRS, ... (i.e. WITHOUT an intermediate numeric subfolder like
# DATA\001\VP001-EEG that the original code assumed). Paths below updated accordingly; see the
# matching fix in load_and_preprocess_fnirs() and process_subject() further down where
# subject_path/subject_num are removed for the same reason.
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"  # NEW: matches user's actual dataset folder
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"  # NEW: matches user's actual output folder

# window parameters
EEG_WINDOW_LEN = 3.0      
EEG_STEP = 1.0            # 66% overlap
EEG_EPOCH_TMIN = 0.0
# EEG_EPOCH_TMAX = 8.0
# FIX (Problem 2): the n-back task period is 40 s long (2 s instruction + 40 s task + 20 s rest,
# Shin et al. 2018). The previous value of 8 s only let the sliding window traverse the first
# 8 s of each trial, discarding ~80% of the task period and, more importantly, was being applied
# on top of STIMULUS-level markers (every ~2 s) rather than TRIAL-level markers, which produced
# massive window overlap across different stimuli instead of covering the full cognitive-load trial.
EEG_EPOCH_TMAX = 40.0      # NEW: cover the full 40 s task period of a single n-back trial

# FNIRS_WINDOW_LEN = 10.0
# CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing, per chat history): matches the TSFNet
# paper's actual approach more closely than the single continuous 10s window used previously --
# instead of one 10s fNIRS window per EEG window, 11 overlapping 3s segments (stepped by
# FNIRS_STEP=1s each) are extracted and stacked, giving the model an explicit per-segment temporal
# axis (matching Table 1's fNIRS branch input) rather than one pre-averaged continuous block.
FNIRS_WINDOW_LEN = 3.0    # CHANGED: 10.0 -> 3.0 (length of EACH of the 11 segments, not the whole span)
FNIRS_NUM_SEGMENTS = 11   # NEW: number of overlapping 3s segments per EEG window, per the paper's approach
FNIRS_STEP = 1.0          # 90% overlap

# NEW (isolated baseline-correction test, per chat history): the paper's Sec. 2.2 also specifies
# "baseline correction was performed by subtracting the average signal within the -5 to -2 s
# interval before each trial" -- a step that was previously only ever tested BUNDLED together
# with the paper's bandpass filter (Run C/E, both rejected), so its effect was never isolated.
# Testing it alone here, with the current (already-validated) 0.2 Hz lowpass filter kept UNCHANGED.
BASELINE_TMIN = -5.0  # NEW: seconds before trial onset (start of baseline window)
BASELINE_TMAX = -2.0  # NEW: seconds before trial onset (end of baseline window)
FNIRS_EPOCH_TMIN = 0.0
FNIRS_EPOCH_TMAX = 18.0   # bacause of the hemodynamic delay
# NOTE: FNIRS_EPOCH_TMAX is no longer used directly to bound the slide range (the EEG trial length
# now governs how many windows are produced, see extract_synchronized_windows below), but it is left
# untouched here exactly as before so no other part of the code/structure changes.

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0      

# NEW (Problem 1 fix support): explicit trial-level marker maps based on the official dataset
# documentation (Shin et al. 2018, Dataset_description_MATLAB.pdf):
#   EEG  mrk_nback.event.desc : 112 = 0-back session, 128 = 2-back session, 144 = 3-back session
#   fNIRS mrk_nback.event.desc:   7 = 0-back session,   8 = 2-back session,   9 = 3-back session
# These TRIAL-onset codes (one per ~40 s task period) replace the old STIMULUS-level codes
# (16/48/64/80/96 for EEG; 1/2/3/4/5 placeholders) that fired every ~2 s within a trial.
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}      # NEW: EEG session/trial-onset codes -> class
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}          # NEW: fNIRS session/trial-onset codes -> class


# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    
    print(f"Loading EEG from: {cnt_path}")
    
    # Load signal
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    # making sure the sampling rate of 200
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    # EOG channels
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    # NOTE: this generic standard_1005 template montage is kept ONLY for MNE-internal purposes that
    # need SOME position estimate for every channel (e.g. interpolate_bads()'s spherical-spline
    # interpolation in preprocess_eeg()). It is NOT used for the spatial_mapping() grid anymore --
    # get_eeg_mnt() (further below) uses the REAL, dataset-provided positions from mnt_nback.mat
    # (loaded at the end of this function) for that purpose instead.
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    # Load markers 
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    
    # Event extraction
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")

    # NEW: load the EEG mnt_nback.mat file -- REAL electrode 3D positions from this dataset,
    # confirmed via user-provided inspection script output: same struct format as the fNIRS
    # mnt_nback.mat already used in load_and_preprocess_fnirs (fields ['x','y','pos_3d','clab'],
    # pos_3d shape (3, 30), clab shape (30,)). This CORRECTS an earlier (wrong) assumption that EEG
    # had no equivalent .mat position file and had to fall back on MNE's generic standard_1005
    # template positions (raw.set_montage() below, which is KEPT unchanged -- it is still needed
    # for interpolate_bads()'s spherical-spline interpolation, an unrelated MNE-internal use). For
    # the azimuthal-equidistant spatial_mapping() grid, get_eeg_mnt() (defined further below) now
    # uses THIS real, dataset-provided mnt_nback.mat instead, exactly mirroring how fNIRS's own
    # mnt_nback.mat is already used for its spatial grid.
    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')  # NEW
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_eeg_mnt_keys = [k for k in mat_eeg_mnt.keys() if not k.startswith('__')]  # NEW
    eeg_mnt_struct = mat_eeg_mnt[real_eeg_mnt_keys[0]]  # NEW: has .pos_3d (3,30) and .clab (30,)
    print(f"EEG mnt_nback.mat loaded: pos_3d shape {np.asarray(eeg_mnt_struct.pos_3d).shape}")  # NEW

    # return raw, events, sfreq
    return raw, events, sfreq, eeg_mnt_struct  # CHANGED: also return the real electrode-position struct


# LOAD AND PREPROCESS FNIRS DATA
def load_and_preprocess_fnirs(subject_id):
    # subject_num = subject_id[2:]
    # subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
    # fnirs_folder = os.path.join(subject_path, f"{subject_id}-NIRS")
    # FIX: removed the intermediate numeric subfolder (e.g. "001") which the user's actual dataset
    # layout does not have; fNIRS data sits directly under BASE_DATASET_PATH\VPxxx-NIRS.
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")  # NEW: flat folder structure
    
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')  # NEW: fNIRS trial markers needed for the quality-check plot and (optionally) downstream use
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    # cnt_data = loadmat(cnt_path)
    # cnt = cnt_data['cnt_nback'][0, 0]
    #
    # mnt_data = loadmat(mnt_path)
    # mnt = mnt_data['mnt_nback'][0, 0]
    # FIX (Problem 1): the lines above used the default scipy.io.loadmat() call (struct_as_record=True),
    # while the rest of the pipeline (load_eeg_data) consistently uses
    # struct_as_record=False, squeeze_me=True. Mixing the two access styles is fragile: under the
    # default mode, struct fields are accessed via numpy structured-array indexing (e.g. cnt['oxy'][0,0]['x']),
    # while under struct_as_record=False they are accessed as plain Python attributes (e.g. cnt.oxy.x).
    # Verified against the actual VP001 .mat structure (inspect_data.py output): cnt_nback has fields
    # ['oxy','deoxy'], each of which is itself a struct with field 'x' holding the (n_samples, 36) signal,
    # and mnt_nback has fields ['x','y','pos_3d','clab'] holding (36,) coordinate vectors. Using the
    # unified squeeze_me=True / struct_as_record=False loader removes the [0,0] nested-indexing style and
    # avoids a silent shape mismatch in spatial_mapping() if the indexing assumption is ever wrong.
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]     # NEW
    cnt = mat_cnt[real_cnt_keys[0]]                                          # NEW: cnt is now a MATLAB-struct object with .oxy / .deoxy attributes

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]     # NEW
    mnt = mat_mnt[real_mnt_keys[0]]                                          # NEW: mnt is now a MATLAB-struct object with .x / .y attributes

    # Load fNIRS trial markers (NEW: needed only for the before/after QC plot at the end of this
    # function; segmentation itself happens later using EEG_TRIAL_LABEL_MAP / FNIRS_TRIAL_LABEL_MAP
    # inside extract_synchronized_windows, which keeps this function's return signature unchanged).
    fnirs_events_time = None  # NEW
    fnirs_events_code = None  # NEW
    if os.path.exists(mrk_path):  # NEW
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)  # NEW
        real_fmrk_keys = [k for k in mat_fmrk.keys() if not k.startswith('__')]    # NEW
        fmrk = mat_fmrk[real_fmrk_keys[0]]                                         # NEW
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):  # NEW
            fnirs_events_time = np.asarray(fmrk.time).flatten()                     # NEW
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()               # NEW
    
    # Extract actual sampling rate (making sure it's 10Hz)
    # sfreq_fnirs_raw = float(cnt['fs'][0, 0]) if 'fs' in cnt.dtype.names else 10.0
    # FIX (Problem 1, continued): 'fs' is a field of cnt.oxy / cnt.deoxy (each a sub-struct), not of
    # cnt itself (verified: cnt_nback.oxy.fs == 10, cnt_nback.deoxy.fs == 10 in the inspected VP001 file).
    # The original line would raise an AttributeError under the new loader (cnt has no .dtype/.fs), so
    # it must read fs from the oxy sub-struct.
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0  # NEW
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    # Extract raw oxy/deoxy
    # if 'oxy' in cnt.dtype.names:
    #     oxy_raw = cnt['oxy'][0, 0]['x']
    #     deoxy_raw = cnt['deoxy'][0, 0]['x']
    # else:
    #     oxy_raw = cnt['oxy']
    #     deoxy_raw = cnt['deoxy']
    # FIX (Problem 1, continued): with struct_as_record=False, squeeze_me=True, the signal matrix is
    # accessed as cnt.oxy.x / cnt.deoxy.x (plain attribute access), matching the (n_samples, 36) shape
    # confirmed in inspect_data.py for VP001 (21178, 36).
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)     # NEW
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)  # NEW
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    # Convert to HbO/HbR
    # print("Converting to HbO/HbR...")
    # hbo_raw, hbr_raw = calculate_hbo_hbr(oxy_raw, deoxy_raw)
    # CHANGED (EXPERIMENTAL -- Beer-Lambert removal test, per chat history): the official dataset
    # Data Descriptor (Shin et al. 2018) explicitly states "cnt files contain deoxy/oxy-hemoglobin
    # data as separate fields... The raw NIRS optical intensity data converted to deoxy-(HbR) and
    # oxy hemoglobin (HbO) concentration changes... using the modified Beer-Lambert law" BEFORE
    # distribution -- confirmed further by the dataset description PDF itself, which explicitly
    # labels these fields "oxy hemoglobin data" / "deoxy hemoglobin data" (not raw optical density).
    # oxy_raw/deoxy_raw are therefore already ΔHbO/ΔHbR; calling calculate_hbo_hbr() on them
    # re-applies Beer-Lambert to already-converted concentration data, which is physically incorrect.
    # calculate_hbo_hbr() itself is left defined-but-unused below (not deleted), per the instruction
    # not to touch other parts of the file.
    print("Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...")  # NEW
    hbo_raw, hbr_raw = oxy_raw, deoxy_raw  # NEW: no conversion -- oxy_raw IS HbO, deoxy_raw IS HbR
    
    # Wavelet denoising 
    # print("Wavelet denoising...")
    # hbo_wv = wavelet_denoising(hbo_raw)
    # hbr_wv = wavelet_denoising(hbr_raw)
    #
    # MARA on full signal
    # print("MARA...")
    # hbo_mara = apply_mara(hbo_wv, fs=sfreq_fnirs_raw)
    # hbr_mara = apply_mara(hbr_wv, fs=sfreq_fnirs_raw)
    #
    # Low-pass filter 
    # FNIRS_LOWPASS_CUTOFF_HZ = 0.2
    # hbo_filtered = lowpass_filter(hbo_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)
    # hbr_filtered = lowpass_filter(hbr_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)
    # CHANGED (EXPERIMENTAL -- isolated fNIRS-filtering-removal test, Priority 2, per chat history):
    # Ding et al. (2026, Information Fusion, "MSTFDN"), using this SAME dataset, applies NO
    # wavelet denoising, NO MARA, and NO separate lowpass/bandpass filtering to fNIRS at all --
    # their stated rationale: "since the fNIRS branch uses convolution kernels with a large size
    # and receptive field (large size temporal convolution is equivalent to low-pass filtering in
    # the frequency domain), the model naturally focuses on low-frequency components during
    # feature learning. Therefore, we did not perform separate filtering preprocessing." Their
    # fNIRS pipeline is simply: Beer-Lambert (already done by dataset authors) -> baseline
    # correction -> z-score normalization. This test skips wavelet_denoising()/apply_mara()/
    # lowpass_filter() entirely (functions left defined-but-unused, not deleted) to test this
    # simplification hypothesis directly against our own hand-designed filtering pipeline, on the
    # SAME CONFIRMED baseline-corrected Run H pipeline (isolated from Priority 1's baseline-window
    # change and the still-unconfirmed EEG filter change).
    print("Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...")  # NEW
    hbo_filtered, hbr_filtered = hbo_raw, hbr_raw  # NEW: pass-through, no denoising/artifact-removal/filtering applied
    
    # dynamic downsampling to 10 Hz
    print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
    from scipy.signal import resample
    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    
    # spatial mapping to 16x16
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    # Stack as channels: (n_samples, 16, 16, 2)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")

    # NEW (user request): save a before/after preprocessing QC plot for fNIRS (one representative
    # channel, raw HbO vs fully preprocessed/resampled HbO). Saved per-subject next to that subject's
    # output folder. Does not alter any returned value or downstream logic.
    try:  # NEW
        qc_channel = 0  # NEW: first fNIRS channel, arbitrary but fixed for reproducibility
        t_raw = np.arange(oxy_raw.shape[0]) / sfreq_fnirs_raw  # NEW
        t_proc = np.arange(hbo_resampled.shape[0]) / SFREQ_FNIRS  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_raw, oxy_raw[:, qc_channel], color='tab:red', linewidth=0.6)  # NEW
        axes[0].set_title(f"{subject_id} - fNIRS RAW HbO (channel {qc_channel}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("HbO (raw units)")  # NEW
        axes[1].plot(t_proc, hbo_resampled[:, qc_channel], color='tab:blue', linewidth=0.8)  # NEW
        axes[1].set_title(f"{subject_id} - fNIRS PREPROCESSED HbO (channel {qc_channel}, after wavelet+MARA+lowpass+resample)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("HbO (mmol/L)")  # NEW
        plt.tight_layout()  # NEW
        qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
        os.makedirs(qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(qc_dir, f"{subject_id}_fnirs_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
        print(f"Saved fNIRS before/after QC plot for {subject_id}")  # NEW
    except Exception as e:  # NEW
        print(f"Could not save fNIRS QC plot for {subject_id}: {e}")  # NEW
    
    # return fnirs_data
    # CHANGED (CRITICAL SYNCHRONIZATION FIX): diagnostic testing (see chat history) directly compared
    # EEG's own trial markers (codes 112/128/144) against fNIRS's own trial markers (codes 7/8/9),
    # both loaded independently from their respective mrk_nback.mat files. This revealed that the
    # two devices do NOT share a common time origin: within each of the 3 class "sessions" the two
    # devices agree almost exactly (delta ~0.1-0.3s, pure jitter), but ACROSS session boundaries a
    # near-constant ~46-47s jump appears (e.g. VP001: session 1 delta=16.6s, session 2 delta=62.7s,
    # session 3 delta=110.0s), consistently reproduced across all 5 subjects tested. Since
    # extract_synchronized_windows() previously computed the fNIRS window position purely by
    # converting EEG's own event time to an fNIRS sample index (assuming a shared t=0), this means
    # for the 2-back and 3-back sessions the fNIRS window was being read from a position roughly
    # 46-93 seconds away from where it should be -- i.e. NOT synchronized with the EEG window it was
    # being paired with. fnirs_events_time and fnirs_events_code (already loaded above, previously
    # used only for the QC plot) are now also returned so extract_synchronized_windows() can use
    # fNIRS's OWN trial markers as the anchor for fNIRS windowing, instead of a value derived from
    # EEG's clock.
    return fnirs_data, fnirs_events_time, fnirs_events_code  # CHANGED: also return fNIRS's own trial markers

# fNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 760, 850
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.586, 1.54852
    e2_hbo, e2_hbr = 1.058, 0.69132
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.15):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    # x_coords = np.asarray(mnt.x).flatten()
    # y_coords = np.asarray(mnt.y).flatten()
    # FIX (projection accuracy): mnt.x / mnt.y are pre-computed 2D coordinates from the BBCI
    # toolbox's own projection, which is not necessarily the azimuthal equidistant projection
    # described in the TSFNet reference paper (Zhang, Yin, Yuan, Sensors 2025). Using the raw 3D
    # channel positions (mnt.pos_3d, shape (3, n_channels), confirmed present in the fNIRS mnt_nback
    # struct via inspect_data.py) and projecting them with a genuine azimuthal equidistant
    # projection (angular distance from the vertex is preserved: theta = angle from north pole,
    # phi = azimuthal angle) more faithfully reproduces the paper's methodology than reusing the
    # toolbox's built-in 2D coordinates.
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)  # NEW: shape (3, n_channels)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]  # NEW
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)  # NEW
    r_safe = np.where(r > 0, r, 1.0)  # NEW: avoid division by zero for any degenerate channel position
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))  # NEW: angle from the vertex (pole)
    phi = np.arctan2(y3d, x3d)  # NEW: azimuthal angle
    x_coords = theta * np.cos(phi)  # NEW: azimuthal equidistant projection
    y_coords = theta * np.sin(phi)  # NEW
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    # mapped = np.zeros((n_samples, 16, 16), dtype=np.float32)
    # for t in range(n_samples):
    #     grid_z = griddata(points, data[t, :], (grid_x, grid_y), method='cubic', fill_value=0)
    #     mapped[t] = grid_z
    # PERF FIX: the loop above calls griddata() once per time sample, re-computing the Delaunay
    # triangulation of `points` from scratch every single time -- even though `points` (electrode
    # positions) NEVER changes across time samples for a given subject; only the values (data[t,:])
    # change. Confirmed via VP001/VP002 real runs, EEG spatial mapping (394930 / 395238 samples at
    # 200 Hz) was the dominant cost of the ~28-32 min per-subject runtime. griddata's 'cubic' 2D
    # path internally builds scipy.interpolate.CloughTocher2DInterpolator(points, values, ...),
    # which officially supports (a) being given a pre-built scipy.spatial.Delaunay triangulation
    # instead of raw points, and (b) vector-valued `values` of shape (n_points, ...trailing_dims),
    # interpolating all trailing "columns" (here: all time samples) in one batched call. Verified
    # numerically bit-identical (max abs diff = 0.0) against the old per-sample loop on synthetic
    # data of the same shape before applying this change (see chat history for the verification
    # script). This is a pure performance refactor -- the interpolation math is unchanged.
    tri = Delaunay(points)  # NEW: triangulation depends only on `points`, computed ONCE instead of n_samples times
    values = data.T  # NEW: (n_samples, n_channels) -> (n_channels, n_samples); CloughTocher2DInterpolator
                      # NEW: requires values.shape[0] == points.shape[0] (== n_channels here)
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # NEW: gradient estimation + patch setup done once, for all time samples together
    result = interp((grid_x, grid_y))  # NEW: shape (16, 16, n_samples) -- one batched evaluation instead of n_samples separate ones
    mapped = np.transpose(result, (2, 0, 1)).astype(np.float32)  # NEW: (16, 16, n_samples) -> (n_samples, 16, 16), matching the original return shape/dtype exactly
    
    return mapped

# NEW (EEG spatial mapping support): spatial_mapping() above was written for fNIRS, which reads
# 3D channel positions from mnt.pos_3d (a field already present in the dataset's mnt_nback.mat
# struct). EEG has no equivalent .mat mnt struct -- its electrode positions instead live inside
# the MNE montage set in load_eeg_data() via raw.set_montage('standard_1005', match_case=False,
# on_missing='ignore'). _EEGMontageAdapter + get_eeg_mnt() below extract those positions, in the
# SAME channel order as raw.ch_names (so they line up 1:1 with data columns passed to
# spatial_mapping), into a lightweight object exposing .pos_3d with shape (3, n_channels) --
# exactly the interface spatial_mapping() already expects -- so spatial_mapping() itself does not
# need to change at all and is reused UNCHANGED for EEG, with the same azimuthal equidistant
# projection methodology already used for fNIRS.
# NEW (EEG spatial mapping support): spatial_mapping() above was written for fNIRS, which reads
# 3D channel positions from mnt.pos_3d (a field present in the dataset's fNIRS mnt_nback.mat
# struct). The EEG folder has its OWN mnt_nback.mat with the exact same format (pos_3d shape
# (3, 30), clab shape (30,)), confirmed via direct inspection of VP001-EEG/mnt_nback.mat -- so EEG
# positions come from the REAL, dataset-provided file, not from MNE's generic standard_1005
# template (an earlier, incorrect assumption; see the comment above raw.set_montage() in
# load_eeg_data(), where that template montage is kept only for interpolate_bads()'s internal use).
# _EEGMontageAdapter + get_eeg_mnt() below select, from the full 30-channel mnt_nback.mat struct,
# only the positions of the channels that SURVIVED preprocessing (raw.ch_names, after
# VEOG/HEOG/TP9 are dropped in preprocess_eeg), matched BY NAME against mnt_nback.mat's own clab
# list (channel order in cnt_nback.mat and mnt_nback.mat is not guaranteed to be identical, so
# matching by name rather than by raw positional index is required for correctness). The result is
# a lightweight object exposing .pos_3d with shape (3, n_channels) -- exactly the interface
# spatial_mapping() already expects -- so spatial_mapping() itself needs no changes at all.
class _EEGMontageAdapter:  # NEW
    """Minimal stand-in for the fNIRS .mat mnt struct, exposing only the .pos_3d attribute that
    spatial_mapping() reads."""
    def __init__(self, pos_3d):  # NEW
        self.pos_3d = pos_3d  # NEW: shape (3, n_channels), matching mnt.pos_3d's shape for fNIRS

def get_eeg_mnt(raw, eeg_mnt_struct):  # CHANGED: now takes the real mnt_nback.mat struct (loaded once in load_eeg_data) instead of deriving positions from raw's MNE montage
    """
    Builds an _EEGMontageAdapter (pos_3d shape (3, n_channels)) from the REAL electrode 3D
    positions in the dataset's own EEG mnt_nback.mat (eeg_mnt_struct.pos_3d / .clab), selecting
    and ordering them to match raw.ch_names exactly. Must be called AFTER any channel drops
    (VEOG/HEOG/TP9 in preprocess_eeg) so the returned position order matches the channel order of
    the data that will be passed to spatial_mapping().
    """
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]  # NEW: e.g. 30 names, same cleanup style as cnt_struct.clab in load_eeg_data
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)  # NEW: shape (3, 30)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}  # NEW
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}  # NEW: case-insensitive fallback, matches match_case=False used elsewhere in this pipeline

    pos_list = []  # NEW
    missing = []  # NEW
    for ch in raw.ch_names:  # NEW
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))  # NEW
        if idx is None:  # NEW: channel present in raw but not found in mnt_nback.mat's clab list
            missing.append(ch)  # NEW
            pos_list.append([0.0, 0.0, 0.0])  # NEW: origin fallback (theta=0 -> maps to grid center)
        else:  # NEW
            pos_list.append(full_pos_3d[:, idx])  # NEW

    if missing:  # NEW
        print(f"  WARNING: {len(missing)} EEG channel(s) not found in mnt_nback.mat's clab "  # NEW
              f"(spatial_mapping will place them at the grid center): {missing}")  # NEW

    pos_3d = np.array(pos_list, dtype=np.float64).T  # NEW: shape (3, n_channels)
    return _EEGMontageAdapter(pos_3d)  # NEW

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")

    # NEW (user request): keep a copy of the raw signal (one representative channel) for the
    # before/after preprocessing QC plot saved at the end of this function.
    raw_before_qc = raw.copy()  # NEW
    
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks='eeg', verbose=False)
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method='fir', verbose=False)
    
    # Detect bad channels
    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info['bads'] = bad_channels
    
    # NEW (fix support): remember which channels were flagged as bad BEFORE interpolate_bads()
    # resets info['bads']. This list is needed further below to correctly compute the ICA rank,
    # since interpolate_bads(reset_bads=True) clears info['bads'] and the original code's
    # exclude='bads' filter therefore silently included the interpolated channels in the ICA fit.
    interpolated_channels = list(raw_filtered.info['bads'])  # NEW
    
    if raw_filtered.info['bads']:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    # ICA
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')
    # corrected_rank = max(1, len(picks_for_ica) - 1)
    # FIX (rank problem): interpolate_bads(reset_bads=True) above already cleared info['bads'],
    # so exclude='bads' here has NOTHING to exclude and picks_for_ica silently includes every
    # interpolated channel. Interpolated channels are linear combinations of their neighbours and
    # add NO new degrees of freedom to the data, so the true data rank is
    # (n_good_channels - 1 for CAR - n_interpolated_channels), not just (n_channels - 1).
    # Asking ICA (via n_components=corrected_rank) for more components than the true rank forces
    # the algorithm to spread a single real source (e.g. the eye-blink dipole) across multiple
    # weak, "diluted" components, none of which individually scores high enough in ICLabel
    # (whether by argmax or by the probability threshold added above) to be flagged as
    # 'eye blink'. This is very likely the main reason eye-blink artifacts survive in this
    # pipeline's output while code 1 (which never interpolates channels) does not have this issue.

###########

    # corrected_rank = max(1, len(picks_for_ica) - 1 - len(interpolated_channels))
    # print(f"ICA rank: {corrected_rank} (picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)})")
    # ica = mne.preprocessing.ICA(n_components=corrected_rank, method='infomax',
    #                             fit_params=dict(extended=True), random_state=42, verbose=False)
    # FIX (rank problem, cleaner solution): the manual rank correction above (subtracting the number
    # of interpolated channels) is a reasonable workaround, but n_components=0.99 solves the same
    # problem more robustly and automatically: instead of a FIXED component count, ICA's internal PCA
    # step keeps only as many components as needed to explain 99% of the data's variance. Interpolated
    # channels contribute (near-)zero additional variance since they are linear combinations of their
    # neighbours, so PCA naturally discards that redundant dimensionality without needing to manually
    # count and subtract interpolated channels. This avoids asking ICA for more components than the
    # true data rank, which is what caused the eye-blink source to be diluted across several weak,
    # under-scored components in the original pipeline (see comment above find_bads_eog for the full
    # rationale of that issue).
    # method='picard' (Ablin, Cardoso & Gramfort, 2018, IEEE Signal Processing Letters) is mathematically
    # equivalent to extended infomax but converges faster and more reliably; it is the ICA algorithm
    # used in the reference implementation (code 1) and is increasingly the default recommendation in
    # recent BCI/EEG preprocessing literature.
    print(f"ICA: using variance-based component selection (99% explained variance), "  # NEW
          f"picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)}")     # NEW
    ica = mne.preprocessing.ICA(n_components=0.99, method='picard',                     # NEW
                                fit_params=dict(ortho=False, extended=True),            # NEW
                                random_state=42, verbose=False)                          # NEW
    ica.fit(raw_filtered, picks=picks_for_ica)


#########    

####
    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')

    ARTIFACT_CLASSES_IDX = {  # NEW: standard ICLabel class order (verified from ica.labels_scores_ shape=(n,7))
        'brain': 0, 'muscle artifact': 1, 'eye blink': 2,
        'heart beat': 3, 'line noise': 4, 'channel noise': 5, 'other': 6
    }
    ARTIFACT_TARGETS = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']  # NEW
    PROB_THRESHOLD = 0.30  # NEW

    # Original argmax-based selection (unchanged logic)
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels['labels'])
                           if label in ARTIFACT_TARGETS]  # CHANGED: renamed

    # NEW: probability-threshold based selection using ica.labels_scores_ (shape: n_components x 7)
    artifact_idx_prob = []  # NEW
    proba_matrix = ica.labels_scores_  # NEW: verified shape (27, 7) for this dataset
    for comp_idx in range(proba_matrix.shape[0]):  # NEW
        for cls_name in ARTIFACT_TARGETS:  # NEW
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)  # NEW
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:  # NEW
                artifact_idx_prob.append(comp_idx)  # NEW
                break  # NEW: one match per component is enough

    # eog_indices, eog_scores = None, None  (did not exist in the previous version of this file)
    # FIX (Problem: no direct EOG-correlation check): code 1 (the reference implementation) relies
    # primarily on mne.preprocessing.ICA.find_bads_eog(), which correlates each ICA component's time
    # course directly against the RECORDED VEOG channel -- a ground-truth signal, independent of
    # ICLabel's scalp-topography/spectrum-based classifier. Code 2 previously relied ONLY on
    # ICLabel (via argmax and/or probability threshold), which never consults the VEOG/HEOG channels
    # that are already loaded and typed as 'eog' in load_eeg_data(). Adding find_bads_eog() here
    # gives ICA exclusion a second, independent detection path so that a genuine eye-blink component
    # that ICLabel under-scores (e.g. due to the rank/dilution issue fixed above) can still be
    # caught via its correlation with the real VEOG signal, matching code 1's methodology.
    eog_indices_veog, eog_scores_veog = [], None  # NEW: defaults in case VEOG channel is missing
    eog_indices_heog, eog_scores_heog = [], None  # NEW: defaults in case HEOG channel is missing
    try:  # NEW
        if 'VEOG' in raw.ch_names:  # NEW: raw is the ORIGINAL (unfiltered) signal passed into this function, matching code 1's use of raw_input
            eog_indices_veog, eog_scores_veog = ica.find_bads_eog(raw, ch_name='VEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(VEOG) flagged components: {eog_indices_veog}")  # NEW
        if 'HEOG' in raw.ch_names:  # NEW: also check HEOG (horizontal eye movements), which code 1 did not check but is available in this dataset
            eog_indices_heog, eog_scores_heog = ica.find_bads_eog(raw, ch_name='HEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(HEOG) flagged components: {eog_indices_heog}")  # NEW
    except Exception as e:  # NEW
        print(f"  find_bads_eog failed: {e}")  # NEW
#########

    
    # NEW (methodological decision, not a named standard algorithm -- see justification below):
    # Gate EOG-correlation-only hits (components flagged ONLY via find_bads_eog, with no ICLabel
    # support) behind an ICLabel 'brain' probability ceiling. Rationale: EOG-correlation is a
    # well-established artifact-detection method (Jung et al. 1998/2000; implemented in EEGLAB,
    # FieldTrip, MNE), and combining multiple independent artifact detectors is itself a recognized
    # practice (e.g. SASICA, Chaumon et al. 2015, J. Neurosci. Methods, which combines EOG-correlation,
    # autocorrelation, topography focus, and discontinuity as independent criteria). However, the
    # literature also documents that EOG-correlation alone can produce false positives, since some
    # genuine brain sources (e.g. frontal activity related to eye-movement planning / corollary
    # discharge) can correlate with EOG without being ocular artifacts themselves. The specific
    # threshold below (BRAIN_PROB_CEILING = 0.5) is NOT a published/validated cutoff from any single
    # paper -- it is a conservative engineering choice made for this thesis to reduce the risk of
    # discarding real brain signal when a component's only supporting evidence is EOG-correlation.
    # Any component that ALSO has independent ICLabel support (argmax or probability threshold) is
    # NOT affected by this gate and is still excluded regardless of its 'brain' probability.
    BRAIN_PROB_CEILING = 0.5  # NEW: conservative, thesis-specific choice -- not a literature-standard value
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX['brain']  # NEW
    
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))  # NEW: components flagged ONLY by EOG-correlation, with no ICLabel support at all
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}  # NEW: rescue if ICLabel itself thinks it's likely brain signal
    if eog_only_rescued:  # NEW
        print(f"  Rescued from EOG-only rejection (ICLabel brain-prob >= {BRAIN_PROB_CEILING}): {sorted(eog_only_rescued)}")  # NEW
    
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )  # CHANGED: same union as before, minus the conservative brain-probability rescue applied only to EOG-only hits


    
###########
    # NEW: detailed log for each flagged component
    all_class_names = ['brain', 'muscle artifact', 'eye blink', 'heart beat', 'line noise', 'channel noise', 'other']  # NEW
    for comp_idx in artifact_idx:  # NEW
        probs_str = ", ".join([f"{all_class_names[c]}={proba_matrix[comp_idx, c]:.2f}"  # NEW
                               for c in range(7)])  # NEW
        reason = []  # NEW
        if comp_idx in artifact_idx_argmax:  # NEW
            reason.append(f"argmax='{ic_labels['labels'][comp_idx]}'")  # NEW
        if comp_idx in artifact_idx_prob:  # NEW
            reason.append(f"prob>={PROB_THRESHOLD}")  # NEW
        if comp_idx in eog_indices_veog:  # NEW
            reason.append("VEOG-correlation")  # NEW
        if comp_idx in eog_indices_heog:  # NEW
            reason.append("HEOG-correlation")  # NEW
        print(f"  IC{comp_idx} flagged [{', '.join(reason)}]: {probs_str}")  # NEW

    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components")
    else:
        print("Removed 0 ICA components (no artifact ICs detected)")  # NEW
    

####
        
    
    # Case-insensitive channel drop
    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    # NEW (user request, updated): save before/after preprocessing QC plots for EEG showing ALL
    # channels at once (not just one representative channel). Two SEPARATE figures are produced:
    # one figure with all RAW channels stacked, one figure with all PREPROCESSED channels stacked.
    # Each channel is plotted on its own horizontal row using a fixed vertical offset (the standard
    # "butterfly/stacked montage" view used by EEG viewers such as MNE's raw.plot()), so all channels
    # remain visually separable in a single image instead of being overlaid on top of each other.
    try:  # NEW
        def _plot_all_channels_stacked(raw_obj, title, color, tmin=None, tmax=None):  # CHANGED: added tmin/tmax for zoomed view
            sfreq_plot = raw_obj.info['sfreq']               # NEW
            if tmin is not None or tmax is not None:         # NEW: optional time-window crop
                start_sample = int((tmin or 0.0) * sfreq_plot)             # NEW
                stop_sample = int((tmax * sfreq_plot)) if tmax is not None else None  # NEW
                data = raw_obj.get_data(start=start_sample, stop=stop_sample)         # NEW
                t = (np.arange(data.shape[1]) + start_sample) / sfreq_plot            # NEW
            else:                                            # NEW
                data = raw_obj.get_data()                      # NEW: (n_channels, n_times)
                t = np.arange(data.shape[1]) / sfreq_plot        # NEW
            ch_names_plot = raw_obj.ch_names                # NEW

            # Use the robust (median absolute deviation based) per-channel spread to set a fixed
            # vertical offset between channels, so that channels with very different amplitudes
            # (e.g. after CAR/ICA vs raw) are still readable and non-overlapping within the same figure.
            #spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 8 + 1e-12  # NEW
            spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 30 + 1e-12  # NEW
            n_ch_plot = data.shape[0]                        # NEW
            #fig_h = max(6, 0.28 * n_ch_plot)  # NEW: scale figure height with channel count
            fig_h = max(6, 0.5 * n_ch_plot)                 # NEW: scale figure height with channel count
            fig, ax = plt.subplots(figsize=(14, fig_h))      # NEW
            for i in range(n_ch_plot):                       # NEW
                offset = (n_ch_plot - 1 - i) * spread         # NEW: top channel = first channel name
                ax.plot(t, data[i] + offset, color=color, linewidth=0.4)  # NEW
            ax.set_yticks([(n_ch_plot - 1 - i) * spread for i in range(n_ch_plot)])  # NEW
            ax.set_yticklabels(ch_names_plot, fontsize=6)    # NEW
            ax.set_xlabel("Time (s)")                        # NEW
            ax.set_ylabel("Channel")                         # NEW
            ax.set_title(title)                              # NEW
            plt.tight_layout()                                # NEW
            return fig                                        # NEW

        fig_before = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing)", 'tab:red'  # NEW
        )  # NEW
        fig_after = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA)", 'tab:blue'  # NEW
        )  # NEW
        fig_before_zoom = _plot_all_channels_stacked(  # NEW: zoomed 10-30s view
            raw_before_qc, "EEG RAW - All Channels (before preprocessing, 10-30s)", 'tab:red', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        fig_after_zoom = _plot_all_channels_stacked(  # NEW: zoomed 10-30s view
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA, 10-30s)", 'tab:blue', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        

        # The subject_id is not directly available inside this function (it only receives raw/events),
        # so the figures are saved using generic names inside OUTPUT_PATH/_tmp_eeg_qc and renamed by
        # the caller (process_subject) right after this function returns. This keeps this function's
        # signature and call sites unchanged.
        tmp_qc_dir = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc")  # NEW
        os.makedirs(tmp_qc_dir, exist_ok=True)  # NEW
        fig_before.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels.png"), dpi=150)  # NEW
        fig_after.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels.png"), dpi=150)  # NEW
        fig_before_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels_zoom10_30.png"), dpi=150)  # NEW
        fig_after_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels_zoom10_30.png"), dpi=150)  # NEW
        plt.close(fig_before)  # NEW
        plt.close(fig_after)  # NEW
        plt.close(fig_before_zoom)  # NEW
        plt.close(fig_after_zoom)  # NEW
    except Exception as e:  # NEW
        print(f"Could not save EEG QC plot: {e}")  # NEW
    
    return raw_filtered


# SYNCHRONIZED SLIDING WINDOW EXTRACTION 
# def extract_synchronized_windows(eeg_raw, fnirs_data, events, sfreq_eeg):
# CHANGED: added eeg_spatial_data parameter -- the pre-computed, spatially-mapped (16x16 grid)
# continuous EEG signal for this subject (shape (n_times, 16, 16), produced once in
# process_subject() via spatial_mapping(), mirroring how fNIRS is spatially mapped once before
# windowing rather than per-window). eeg_raw (the MNE Raw object, channel x time) is still passed
# in and still used, but now ONLY for the amplitude-based artifact rejection check below, since
# that threshold (120 uV) is calibrated in electrode voltage units and is not a meaningful check
# on interpolated grid values.
# def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg):
# CHANGED (CRITICAL SYNCHRONIZATION FIX, see the comment above load_and_preprocess_fnirs()'s return
# statement for the full diagnostic rationale): added fnirs_events_time / fnirs_events_code so each
# EEG trial can be paired with fNIRS's OWN trial marker instead of a position derived from EEG's
# clock, which diagnostic testing showed drifts by ~46-93s across the dataset's 3 class sessions.
def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,  # CHANGED
                                  fnirs_events_time, fnirs_events_code):  # CHANGED

    print("Extracting synchronized asymmetric sliding windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    
    # support both marker formats 
    # label_map = {
    #     1: 0, 2: 1, 3: 1, 4: 2, 5: 2,
    #     16: 0, 48: 1, 64: 1, 80: 2, 96: 2
    # }
    #
    # valid_codes = [1, 2, 3, 4, 5, 16, 48, 64, 80, 96]
    # target_events = events[np.isin(events[:, 2], valid_codes)]
    # FIX (Problem 2): the codes above (1-5, 16/48/64/80/96) are STIMULUS-level markers that fire
    # roughly every 2 s within a 40 s trial (per Shin et al. 2018 / Dataset_description_MATLAB.pdf,
    # codes 16/48/64/80/96 correspond to individual target/non-target number presentations, not to
    # the start of a cognitive-load trial). Sliding a window from each of these stimulus markers
    # caused massive, redundant overlap between windows anchored to different stimuli within the same
    # trial, rather than a clean traversal of the full 40 s task period for that trial. The correct
    # trial-onset codes, confirmed directly from the VP001 .mat structure (inspect_data.py output:
    # mrk_nback.event.desc contains 144 etc., and the dataset documentation confirms 112/128/144 =
    # 0-/2-/3-back SESSION start), are used instead: each trial-onset marker anchors exactly one
    # 40 s task period, which is then traversed by the (unchanged) sliding-window loop below.
    label_map = EEG_TRIAL_LABEL_MAP  # NEW: trial-onset codes (112/128/144) -> class label, defined at top of script
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # NEW
    target_events = events[np.isin(events[:, 2], valid_codes)]  # NEW: now selects one event per ~40s trial instead of one event per ~2s stimulus
    
    print(f"found {len(target_events)} trial markers")
    if len(target_events) == 0:
        print(f"No trial markers found. Available markers: {np.unique(events[:, 2])}")
        return np.array([]), np.array([]), np.array([])

    # NEW (CRITICAL SYNCHRONIZATION FIX -- see comment above load_and_preprocess_fnirs()'s return
    # statement for the full diagnostic rationale): build fNIRS's OWN trial-onset time array (in
    # seconds), independently of EEG's clock, from fnirs_events_time/fnirs_events_code (loaded in
    # load_and_preprocess_fnirs()). Diagnostic testing confirmed that, for every subject checked,
    # EEG's trial-class sequence and fNIRS's trial-class sequence match exactly 1:1 in order, so
    # trial i in target_events corresponds to trial i in fnirs_trial_times/fnirs_trial_classes --
    # this is what allows pairing them by simple array position below.
    fnirs_sync_ok = False  # NEW
    fnirs_trial_times = None  # NEW
    if fnirs_events_time is not None and fnirs_events_code is not None:  # NEW
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())  # NEW
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)  # NEW
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)  # NEW
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]  # NEW
        fnirs_order = np.argsort(fnirs_trial_times_ms)  # NEW
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0  # NEW: seconds, fNIRS's own clock
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])  # NEW

        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])  # NEW
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))  # NEW
        if (len(fnirs_trial_times) == len(target_events)  # NEW: same trial count in both devices
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):  # NEW: same class order
            fnirs_sync_ok = True  # NEW
            print(f"fNIRS-own-marker sync check: OK ({len(fnirs_trial_times)} trials, class order matches EEG)")  # NEW
        else:
            print(f"  WARNING: fNIRS own markers do not match EEG 1:1 "  # NEW
                  f"(fNIRS n={len(fnirs_trial_times)}, EEG n={len(target_events)}) -- "  # NEW
                  f"falling back to EEG-clock-derived fNIRS timing for this subject.")  # NEW
    else:
        print("  WARNING: fNIRS event markers unavailable -- falling back to EEG-clock-derived fNIRS timing.")  # NEW
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    
    for trial_idx, ev in enumerate(target_events):  # CHANGED: enumerate() added to index into fnirs_trial_times
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        
        # event_time = event_sample_eeg / sfreq_eeg
        # CHANGED (CRITICAL SYNCHRONIZATION FIX): event_time used to be derived purely from EEG's
        # own clock and then reused to index INTO fNIRS's array too, silently assuming a shared
        # t=0 between the two independently-recorded devices. eeg_event_time (below) is still used
        # for the EEG window and for the baseline window (both correctly indexed into EEG-derived
        # arrays / fnirs_data using EEG's own timeline as before -- baseline correction was already
        # working correctly within a session since the drift is BETWEEN sessions, not within one,
        # and baseline windows are always taken from shortly before the SAME trial). fnirs_event_time
        # (used only for the fNIRS window's actual position below) now comes from fNIRS's OWN trial
        # marker when the sync check above passed, eliminating the ~46-93s cross-session drift.
        eeg_event_time = event_sample_eeg / sfreq_eeg  # CHANGED: renamed from event_time for clarity
        if fnirs_sync_ok:  # NEW
            fnirs_event_time = fnirs_trial_times[trial_idx]  # NEW: fNIRS's own clock for this trial
        else:  # NEW
            fnirs_event_time = eeg_event_time  # NEW: fallback -- old (drift-prone) behavior, only used if sync check failed

        # NEW (isolated baseline-correction test): compute this trial's fNIRS baseline ONCE per
        # trial (not per segment/offset), from the continuous fnirs_data signal, using the
        # BASELINE_TMIN/BASELINE_TMAX window (-5 to -2 s before this trial's onset, on the SAME
        # sync-fixed fnirs_event_time anchor used for the segments themselves). Subtracted from
        # every fNIRS segment belonging to this trial below. If the pre-trial baseline window
        # falls before the start of the recording (only possible for a trial very close to time
        # 0), no correction is applied for that trial (fnirs_baseline stays 0) rather than
        # raising an error or using an out-of-bounds/negative index.
        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))  # NEW
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))    # NEW
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:  # NEW
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)  # NEW: shape (1,16,16,2)
        else:  # NEW
            fnirs_baseline = 0.0  # NEW: not enough pre-trial data (e.g. the very first trial) -- skip correction for this trial

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            
            # EEG window
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            # fNIRS window 
            # w_start_fnirs = int(np.round((event_time + offset_sec) * SFREQ_FNIRS))
            # w_start_fnirs = int(np.round((fnirs_event_time + offset_sec) * SFREQ_FNIRS))  # CHANGED: uses fNIRS's own clock (fnirs_event_time), not EEG-derived event_time
            # w_end_fnirs = w_start_fnirs + window_samples_fnirs
            #
            # if w_end_eeg > eeg_raw.n_times:
            #     continue
            #
            # if w_end_fnirs > fnirs_data.shape[0]:
            #     continue
            # CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing): instead of ONE continuous
            # 10s fNIRS window anchored at fnirs_event_time+offset_sec, extract FNIRS_NUM_SEGMENTS=11
            # overlapping 3s segments, each shifted by FNIRS_STEP=1s from the previous one (matching
            # the TSFNet paper's approach, and the same segment structure used in the other expert's
            # extract_tsfnet_windows()). The sync-fix anchor (fnirs_event_time, from fNIRS's OWN
            # trial marker -- completely UNCHANGED from above) is still used as the base time for
            # segment 0; this is the one improvement over the other expert's version, which anchored
            # segments to EEG's clock directly and was therefore subject to the ~46-93s cross-session
            # drift this project already diagnosed and fixed.
            if w_end_eeg > eeg_raw.n_times:  # CHANGED: moved this check up, unchanged logic
                continue

            fnirs_segments = []  # NEW
            valid_segments = True  # NEW
            for seg_idx in range(FNIRS_NUM_SEGMENTS):  # NEW
                seg_offset = offset_sec + seg_idx * FNIRS_STEP  # NEW: each segment shifted by 1s from the previous
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))  # NEW: still uses the sync-fixed anchor
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs  # NEW: window_samples_fnirs now = 3s worth of samples (FNIRS_WINDOW_LEN=3.0)

                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:  # NEW
                    valid_segments = False  # NEW
                    break  # NEW

                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)  # CHANGED: subtract this trial's baseline, isolated from the filter change

            if not valid_segments:  # NEW
                continue  # NEW
            
            # Extract EEG
            # eeg_slice = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            # if np.max(np.abs(eeg_slice)) > 120e-6:
            #     continue
            # CHANGED (EEG spatial mapping): the SAVED EEG window must now be the spatially-mapped
            # (16, 16, 600) grid representation, not the raw (28, 600) channel-space slice. The
            # artifact-rejection amplitude check, however, stays on the raw channel-space slice
            # (eeg_slice_raw) since the 120 uV threshold is only meaningful in electrode voltage
            # units, not on cubic-interpolated grid values.
            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)  # CHANGED: renamed, QC-only now
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:  # CHANGED: uses eeg_slice_raw
                continue

            # NEW: slice the pre-computed spatially-mapped EEG (n_times, 16, 16) at the SAME sample
            # range as eeg_slice_raw (both index the same continuous recording at sfreq_eeg=200Hz),
            # then transpose (600, 16, 16) -> (16, 16, 600) i.e. grid-H x grid-W x time, per spec.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # NEW: (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # NEW: (16, 16, 600) -- this is what gets saved

            # Extract fNIRS
            # fnirs_slice = fnirs_data[w_start_fnirs:w_end_fnirs]
            # fnirs_slice = fnirs_slice.transpose(3, 0, 1, 2) 
            # CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing): stack the 11 already-extracted
            # segments (each (30,16,16,2)) into (11,30,16,16,2), then transpose to put the segment
            # axis first and channel (HbO/HbR) axis last, matching the shape convention used by the
            # other expert's implementation of this same approach: (11, 16, 16, 30, 2).
            fnirs_slice = np.stack(fnirs_segments, axis=0)      # NEW: (11, 30, 16, 16, 2)
            fnirs_slice = fnirs_slice.transpose(0, 2, 3, 1, 4)  # NEW: (11, 16, 16, 30, 2)
            
            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
    
    eeg_windows = np.array(eeg_windows)
    fnirs_windows = np.array(fnirs_windows)
    labels = np.array(labels)
    
    print(f"Extracted {len(labels)} synchronized windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
    
    return eeg_windows, fnirs_windows, labels



LOG_FILE = os.path.join(OUTPUT_PATH, "processing_log.txt")  # NEW: log file path

def log_message(msg):  # NEW: helper to print + append to log file
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")



# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    t0 = time.time()  # NEW
    log_message(f"START processing {subject_id}")  # NEW
    print(f"Processing {subject_id}")
    
    # subject_num = subject_id[2:]
    # subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
    # eeg_folder = os.path.join(subject_path, f"{subject_id}-EEG")
    # FIX: same flat-folder-structure fix as in load_and_preprocess_fnirs() above; EEG data sits
    # directly under BASE_DATASET_PATH\VPxxx-EEG, with no intermediate numeric subfolder.
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")  # NEW: flat folder structure
    
    if not os.path.exists(eeg_folder):
        log_message(f"SKIPPED {subject_id}: EEG folder not found")  # NEW
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n Loading EEG...")
        # raw_eeg, events, sfreq_eeg = load_eeg_data(eeg_folder)
        raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)  # CHANGED: also receive the real EEG mnt_nback.mat struct
        
        print("\n Loading and preprocessing fNIRS...")
        # fnirs_data = load_and_preprocess_fnirs(subject_id)
        fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)  # CHANGED: also receive fNIRS's own trial markers
        
        # NEW (bug fix): normalization must happen BEFORE window extraction, not after.
        # extract_synchronized_windows() below slices fnirs_windows directly out of fnirs_data,
        # so if fnirs_data is normalized AFTER that call, the windows already saved are un-normalized
        # and the later normalization line has no effect on the saved output (confirmed by
        # VP011's quality report: fNIRS std=0.000367 instead of ~1.0). Moving the normalization
        # here, right after fnirs_data is produced and before any windowing happens, fixes this.
        print("\n Normalizing fNIRS (subject-level, before windowing)...")  # NEW
        fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)  # NEW: moved here from below
        
        print("\n Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)

        # NEW (EEG per-subject normalization fix support): compute per-channel mean/std from the
        # FULL cleaned, continuous EEG signal (after filtering/CAR/ICA) BEFORE any windowing happens.
        # FIX (per-window vs per-subject normalization): the previous EEG normalization (further
        # below, in the "Normalizing..." section) computed a separate z-score for EACH extracted
        # window independently (axis=-1 over the window's 600 time samples). With a window as short
        # as 3 s, this local z-scoring is noisy/unstable and, more importantly, completely erases any
        # systematic difference in absolute EEG amplitude/power between subjects or between cognitive
        # load levels within a subject -- the model can then only see the RELATIVE waveform shape
        # inside each isolated 3 s window, which is a much weaker and more subject-specific signal.
        # This is very likely a major contributor to the severe overfitting observed during model
        # training (train acc ~96% vs val/test acc near chance level): the model latches onto
        # subject-specific residual noise patterns within each locally-renormalized window instead of
        # learning generalizable cognitive-load-related features. Computing mean/std once per subject
        # over the full continuous recording (mirroring the fNIRS per-subject normalization fix above)
        # and applying those FIXED per-channel statistics to every window extracted from that subject
        # preserves the relative amplitude structure across windows and across load levels, which
        # should generalize better to unseen subjects.
        eeg_full_data = raw_eeg_clean.get_data()  # NEW: shape (n_channels, n_times), fully preprocessed continuous signal
        eeg_subject_mean = eeg_full_data.mean(axis=-1, keepdims=True)  # NEW: shape (n_channels, 1)
        eeg_subject_std = eeg_full_data.std(axis=-1, keepdims=True) + 1e-8  # NEW: shape (n_channels, 1)
        print(f"\n EEG per-subject normalization stats computed: "  # NEW
              f"mean range [{eeg_subject_mean.min():.2e}, {eeg_subject_mean.max():.2e}], "  # NEW
              f"std range [{eeg_subject_std.min():.2e}, {eeg_subject_std.max():.2e}]")  # NEW

        # NEW (EEG spatial mapping, TSFNet paper Sec. 2.3): project the EEG electrodes onto a 16x16
        # spatial grid via azimuthal equidistant projection, exactly as already done for fNIRS
        # (spatial_mapping() + mnt), instead of leaving EEG as raw (channel x time). This is applied
        # ONCE to the full continuous, already-cleaned recording (not per-window) for the same
        # efficiency reason fNIRS is spatially mapped once in load_and_preprocess_fnirs() before
        # windowing: with 66% window overlap, per-window mapping would re-run cubic griddata
        # interpolation on largely the same timepoints hundreds of times over.
        print("\n Spatial mapping EEG to 16x16...")  # NEW
        # eeg_mnt = get_eeg_mnt(raw_eeg_clean)  # electrode positions in raw_eeg_clean.ch_names order (post channel-drop)
        eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)  # CHANGED: uses REAL mnt_nback.mat positions, matched by clab name, not MNE's guessed standard_1005
        eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)  # NEW: (n_channels,n_times).T -> (n_times, n_channels) in, (n_times, 16, 16) out
        print(f"EEG spatially mapped: {eeg_spatial_full.shape}")  # NEW

        # NEW: per-subject normalization of the spatially-mapped EEG, applied to the FULL continuous
        # grid signal before windowing -- mirrors exactly how fNIRS is normalized in this same
        # function (fnirs_data = (fnirs_data - mean)/std over axis=0, i.e. per grid-cell across
        # time). This supersedes the old per-channel normalization (eeg_subject_mean/eeg_subject_std
        # above), which no longer applies now that EEG windows are (16,16,600) grid data rather than
        # (28,600) channel data; eeg_subject_mean/eeg_subject_std are kept above only for the
        # informational print (channel-space signal scale), not used for normalization below anymore.
        eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (  # NEW
            eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)  # NEW

        # NEW (user request): finalize the EEG before/after QC plots (now two all-channel figures)
        # for this specific subject by renaming the temp files produced inside preprocess_eeg()
        # (which has no access to subject_id).
        try:  # NEW
            tmp_qc_before = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels.png")  # NEW
            tmp_qc_after = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels.png")    # NEW
            tmp_qc_before_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels_zoom10_30.png")  # NEW
            tmp_qc_after_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels_zoom10_30.png")    # NEW
            if os.path.exists(tmp_qc_before) and os.path.exists(tmp_qc_after):  # NEW
                final_qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
                os.makedirs(final_qc_dir, exist_ok=True)  # NEW
                final_qc_before = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels.png")  # NEW
                final_qc_after = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels.png")    # NEW
                os.replace(tmp_qc_before, final_qc_before)  # NEW
                os.replace(tmp_qc_after, final_qc_after)    # NEW
                if os.path.exists(tmp_qc_before_zoom) and os.path.exists(tmp_qc_after_zoom):  # NEW
                    final_qc_before_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels_zoom10_30.png")  # NEW
                    final_qc_after_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels_zoom10_30.png")    # NEW
                    os.replace(tmp_qc_before_zoom, final_qc_before_zoom)  # NEW
                    os.replace(tmp_qc_after_zoom, final_qc_after_zoom)    # NEW
                print(f"Saved EEG before/after (all-channels) QC plots for {subject_id}")  # NEW
        except Exception as e:  # NEW
            print(f"Could not finalize EEG QC plot for {subject_id}: {e}")  # NEW
        
        print("\n Extracting synchronized asymmetric windows...")
        # eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
        #     raw_eeg_clean, fnirs_data, events, sfreq_eeg
        # )
        # CHANGED: pass eeg_spatial_full (the pre-computed 16x16-grid-mapped continuous EEG) in
        # addition to raw_eeg_clean (still needed inside for the channel-space QC amplitude check).
        # eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
        #     raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg
        # )
        # CHANGED (CRITICAL SYNCHRONIZATION FIX): also pass fnirs_events_time/fnirs_events_code so
        # fNIRS windows are anchored to fNIRS's OWN trial markers, not EEG-clock-derived timing.
        eeg_windows, fnirs_windows, labels = extract_synchronized_windows(  # CHANGED
            raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg,  # CHANGED
            fnirs_events_time, fnirs_events_code  # NEW
        )
        
        if len(labels) == 0:
            log_message(f"WARNING {subject_id}: no windows extracted")  # NEW
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n Normalizing...")
        # EEG: (n_windows, n_channels, n_times) 
        # eeg_windows = (eeg_windows - eeg_windows.mean(axis=-1, keepdims=True)) / (eeg_windows.std(axis=-1, keepdims=True) + 1e-8)
        # FIX (per-window -> per-subject EEG normalization): the line above computed a fresh z-score
        # for every single window independently (per-window mean/std over its own 600 time samples).
        # As explained in the comment above where eeg_subject_mean/eeg_subject_std are computed, this
        # erases systematic amplitude differences between cognitive-load levels and between subjects,
        # which is very likely contributing to the poor generalization (near-chance val/test accuracy)
        # observed during model training. Applying the FIXED per-subject statistics computed once
        # from the full continuous recording (broadcast across all windows and all time samples via
        # reshape to (1, n_channels, 1)) fixes this while still correcting for each subject's overall
        # signal scale, mirroring the per-subject normalization already used for fNIRS.
        # eeg_mean_broadcast = eeg_subject_mean.reshape(1, -1, 1)  # (1, n_channels, 1) for broadcasting over (N, n_channels, n_times)
        # eeg_std_broadcast = eeg_subject_std.reshape(1, -1, 1)
        # eeg_windows = (eeg_windows - eeg_mean_broadcast) / eeg_std_broadcast
        # CHANGED (EEG spatial mapping): eeg_windows is now (n_windows, 16, 16, 600) grid data, not
        # (n_windows, 28, 600) channel data, so the per-channel normalization above no longer applies
        # and is removed. EEG normalization now happens EARLIER, on the full continuous spatially-
        # mapped signal (eeg_spatial_full) before windowing -- see the "NEW: per-subject
        # normalization of the spatially-mapped EEG" block above, right after spatial_mapping() is
        # called -- exactly mirroring how fNIRS is normalized in grid space before windowing. No
        # further EEG normalization is needed here; eeg_windows already carries normalized values.
        
        # fNIRS: (n_windows, 2, 100, 16, 16)
        # fnirs_windows = (fnirs_windows - fnirs_windows.mean(axis=2, keepdims=True)) / (fnirs_windows.std(axis=2, keepdims=True) + 1e-8)
        # FIX (Problem 3): ...
        # fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)
        # BUG FIX (moved): this line was dead code -- fnirs_windows had already been sliced from the
        # un-normalized fnirs_data by extract_synchronized_windows() before this point ran. The actual
        # normalization now happens earlier, right after load_and_preprocess_fnirs() returns and
        # before extract_synchronized_windows() is called. See that block above.
        
        print("\n Saving...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
        # NEW (user request): for the first verification run, only save outputs for the very first
        # subject so correctness can be checked before running the full 26-subject batch. The save
        # calls below are commented out for all subjects except VP001; uncomment the three np.save
        # lines below (and remove/relax this condition) once the pipeline has been verified.
#        #if subject_id == "VP001":  # NEW: save outputs only for the first subject during verification
        np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)    # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)  # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)               # NEW
        # else:                                                                                    # NEW
        #     # Saving suppressed for subjects other than VP001 during this verification run.        # NEW
        #     pass                                                                                  # NEW

        # NEW (adopted from reviewing a third-party implementation, see chat history): a small
        # per-subject metadata.json sidecar file, so the exact parameters/shapes a given
        # preprocessing run used can be checked at a glance without loading the (large) .npy
        # arrays -- useful given this project has gone through many preprocessing variants
        # (Run A-H) and it has repeatedly been unclear afterward exactly which run produced a
        # given set of output files.
        metadata = {  # NEW
            'subject_id': subject_id,  # NEW
            'eeg_shape': list(eeg_windows.shape),  # NEW
            'fnirs_shape': list(fnirs_windows.shape),  # NEW
            'labels_distribution': np.bincount(labels).tolist(),  # NEW
            'eeg_window_len': EEG_WINDOW_LEN,  # NEW
            'eeg_step': EEG_STEP,  # NEW
            'eeg_epoch_tmax': EEG_EPOCH_TMAX,  # NEW: important given this exact parameter caused a major bug in the reviewed third-party code
            'fnirs_num_segments': FNIRS_NUM_SEGMENTS,  # NEW
            'fnirs_window_len': FNIRS_WINDOW_LEN,  # NEW
            'fnirs_step': FNIRS_STEP,  # NEW
            'sfreq_eeg': SFREQ_EEG,  # NEW
            'sfreq_fnirs': SFREQ_FNIRS,  # NEW
            'beer_lambert_applied': False,  # NEW: this pipeline uses oxy_raw/deoxy_raw directly (Run F fix), not calculate_hbo_hbr()
            'fnirs_sync_fix_applied': True,  # NEW: fNIRS windows anchored to fNIRS's own trial markers, not EEG's clock
            'normalization': 'per-subject z-score (EEG grid + fNIRS grid, before windowing)',  # NEW
        }
        with open(os.path.join(sub_out_dir, f"{subject_id}_metadata.json"), 'w') as f:  # NEW
            json.dump(metadata, f, indent=2)  # NEW
        
        print(f"\n {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"DONE {subject_id} in {elapsed:.1f} min | EEG: {eeg_windows.shape} | fNIRS: {fnirs_windows.shape} | Labels: {np.bincount(labels)}")  # NEW
        
    except Exception as e:
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"ERROR processing {subject_id} after {elapsed:.1f} min: {str(e)}")  # NEW
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    total_t0 = time.time()  # NEW
    log_message("===== PIPELINE STARTED =====")  # NEW
    print("MULTIMODAL PREPROCESSING PIPELINE")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    print(f"fNIRS epoch: {FNIRS_EPOCH_TMAX}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")
    
    # for sub_num in range(1, 27):
    # NEW (verification run): limited to VP001 only, per user's request to first check correctness
    # on a single subject before running the full 26-subject batch. Change back to range(1, 27)
    # once the pipeline has been verified.
#    for sub_num in range(1, 2):  # NEW: VP001 only, for verification
    for sub_num in range(14, 27):  # NEW
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)

    
    print("ALL THE SELECTED SUBJECTS PROCESSED")

[2026-07-09 20:11:28] ===== PIPELINE STARTED =====
MULTIMODAL PREPROCESSING PIPELINE
EEG window: 3.0s, step: 1.0s
fNIRS window: 3.0s, step: 1.0s
fNIRS epoch: 18.0s
fNIRS sampling rate: 10.0 Hz
[2026-07-09 20:11:28] START processing VP014
Processing VP014

 Loading EEG...
Loading EEG from: E:\kntu\Thesis\codes\DATA\VP014-EEG\cnt_nback.mat
EEG loaded: 386876 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]
EEG mnt_nback.mat loaded: pos_3d shape (3, 30)

 Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP014-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (20706, 36), deoxy (20706, 36)
Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...
Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...
Resampling from 10.0 Hz to 10.0 Hz...
Spatial mapping

In [3]:
#version2 - session_id
# MULTIMODAL PREPROCESSING: EEG + fNIRS 
# Set the paths based on your system before running the code
# you can adjust which subjects to process at the bottom of this cell

import os
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
from scipy.interpolate import CloughTocher2DInterpolator  # NEW: used by the vectorized spatial_mapping()
from scipy.spatial import Delaunay  # NEW: used by the vectorized spatial_mapping()
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt  # NEW: needed for before/after preprocessing plots
import time
import datetime
import json  # NEW: for per-subject metadata.json export (adopted from reviewing a third-party implementation, see chat history)
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
# BASE_DATASET_PATH = r"D:\Research\EEG-fNIRS\project\project\Dataset"
# OUTPUT_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"
# FIX (path update per user's actual folder layout): the user's dataset is laid out as
# DATA\VP001-EEG, DATA\VP001-NIRS, ... (i.e. WITHOUT an intermediate numeric subfolder like
# DATA\001\VP001-EEG that the original code assumed). Paths below updated accordingly; see the
# matching fix in load_and_preprocess_fnirs() and process_subject() further down where
# subject_path/subject_num are removed for the same reason.
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"  # NEW: matches user's actual dataset folder
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"  # NEW: matches user's actual output folder

# window parameters
EEG_WINDOW_LEN = 3.0      
EEG_STEP = 1.0            # 66% overlap
EEG_EPOCH_TMIN = 0.0
# EEG_EPOCH_TMAX = 8.0
# FIX (Problem 2): the n-back task period is 40 s long (2 s instruction + 40 s task + 20 s rest,
# Shin et al. 2018). The previous value of 8 s only let the sliding window traverse the first
# 8 s of each trial, discarding ~80% of the task period and, more importantly, was being applied
# on top of STIMULUS-level markers (every ~2 s) rather than TRIAL-level markers, which produced
# massive window overlap across different stimuli instead of covering the full cognitive-load trial.
EEG_EPOCH_TMAX = 40.0      # NEW: cover the full 40 s task period of a single n-back trial

# FNIRS_WINDOW_LEN = 10.0
# CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing, per chat history): matches the TSFNet
# paper's actual approach more closely than the single continuous 10s window used previously --
# instead of one 10s fNIRS window per EEG window, 11 overlapping 3s segments (stepped by
# FNIRS_STEP=1s each) are extracted and stacked, giving the model an explicit per-segment temporal
# axis (matching Table 1's fNIRS branch input) rather than one pre-averaged continuous block.
FNIRS_WINDOW_LEN = 3.0    # CHANGED: 10.0 -> 3.0 (length of EACH of the 11 segments, not the whole span)
FNIRS_NUM_SEGMENTS = 11   # NEW: number of overlapping 3s segments per EEG window, per the paper's approach
FNIRS_STEP = 1.0          # 90% overlap

# NEW (isolated baseline-correction test, per chat history): the paper's Sec. 2.2 also specifies
# "baseline correction was performed by subtracting the average signal within the -5 to -2 s
# interval before each trial" -- a step that was previously only ever tested BUNDLED together
# with the paper's bandpass filter (Run C/E, both rejected), so its effect was never isolated.
# Testing it alone here, with the current (already-validated) 0.2 Hz lowpass filter kept UNCHANGED.
BASELINE_TMIN = -5.0  # NEW: seconds before trial onset (start of baseline window)
BASELINE_TMAX = -2.0  # NEW: seconds before trial onset (end of baseline window)
FNIRS_EPOCH_TMIN = 0.0
FNIRS_EPOCH_TMAX = 18.0   # bacause of the hemodynamic delay
# NOTE: FNIRS_EPOCH_TMAX is no longer used directly to bound the slide range (the EEG trial length
# now governs how many windows are produced, see extract_synchronized_windows below), but it is left
# untouched here exactly as before so no other part of the code/structure changes.

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0      

# NEW (Problem 1 fix support): explicit trial-level marker maps based on the official dataset
# documentation (Shin et al. 2018, Dataset_description_MATLAB.pdf):
#   EEG  mrk_nback.event.desc : 112 = 0-back session, 128 = 2-back session, 144 = 3-back session
#   fNIRS mrk_nback.event.desc:   7 = 0-back session,   8 = 2-back session,   9 = 3-back session
# These TRIAL-onset codes (one per ~40 s task period) replace the old STIMULUS-level codes
# (16/48/64/80/96 for EEG; 1/2/3/4/5 placeholders) that fired every ~2 s within a trial.
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}      # NEW: EEG session/trial-onset codes -> class
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}          # NEW: fNIRS session/trial-onset codes -> class


# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    
    print(f"Loading EEG from: {cnt_path}")
    
    # Load signal
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    # making sure the sampling rate of 200
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    # EOG channels
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    # NOTE: this generic standard_1005 template montage is kept ONLY for MNE-internal purposes that
    # need SOME position estimate for every channel (e.g. interpolate_bads()'s spherical-spline
    # interpolation in preprocess_eeg()). It is NOT used for the spatial_mapping() grid anymore --
    # get_eeg_mnt() (further below) uses the REAL, dataset-provided positions from mnt_nback.mat
    # (loaded at the end of this function) for that purpose instead.
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    # Load markers 
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    
    # Event extraction
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")

    # NEW: load the EEG mnt_nback.mat file -- REAL electrode 3D positions from this dataset,
    # confirmed via user-provided inspection script output: same struct format as the fNIRS
    # mnt_nback.mat already used in load_and_preprocess_fnirs (fields ['x','y','pos_3d','clab'],
    # pos_3d shape (3, 30), clab shape (30,)). This CORRECTS an earlier (wrong) assumption that EEG
    # had no equivalent .mat position file and had to fall back on MNE's generic standard_1005
    # template positions (raw.set_montage() below, which is KEPT unchanged -- it is still needed
    # for interpolate_bads()'s spherical-spline interpolation, an unrelated MNE-internal use). For
    # the azimuthal-equidistant spatial_mapping() grid, get_eeg_mnt() (defined further below) now
    # uses THIS real, dataset-provided mnt_nback.mat instead, exactly mirroring how fNIRS's own
    # mnt_nback.mat is already used for its spatial grid.
    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')  # NEW
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_eeg_mnt_keys = [k for k in mat_eeg_mnt.keys() if not k.startswith('__')]  # NEW
    eeg_mnt_struct = mat_eeg_mnt[real_eeg_mnt_keys[0]]  # NEW: has .pos_3d (3,30) and .clab (30,)
    print(f"EEG mnt_nback.mat loaded: pos_3d shape {np.asarray(eeg_mnt_struct.pos_3d).shape}")  # NEW

    # return raw, events, sfreq
    return raw, events, sfreq, eeg_mnt_struct  # CHANGED: also return the real electrode-position struct


# LOAD AND PREPROCESS FNIRS DATA
def load_and_preprocess_fnirs(subject_id):
    # subject_num = subject_id[2:]
    # subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
    # fnirs_folder = os.path.join(subject_path, f"{subject_id}-NIRS")
    # FIX: removed the intermediate numeric subfolder (e.g. "001") which the user's actual dataset
    # layout does not have; fNIRS data sits directly under BASE_DATASET_PATH\VPxxx-NIRS.
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")  # NEW: flat folder structure
    
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')  # NEW: fNIRS trial markers needed for the quality-check plot and (optionally) downstream use
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    # cnt_data = loadmat(cnt_path)
    # cnt = cnt_data['cnt_nback'][0, 0]
    #
    # mnt_data = loadmat(mnt_path)
    # mnt = mnt_data['mnt_nback'][0, 0]
    # FIX (Problem 1): the lines above used the default scipy.io.loadmat() call (struct_as_record=True),
    # while the rest of the pipeline (load_eeg_data) consistently uses
    # struct_as_record=False, squeeze_me=True. Mixing the two access styles is fragile: under the
    # default mode, struct fields are accessed via numpy structured-array indexing (e.g. cnt['oxy'][0,0]['x']),
    # while under struct_as_record=False they are accessed as plain Python attributes (e.g. cnt.oxy.x).
    # Verified against the actual VP001 .mat structure (inspect_data.py output): cnt_nback has fields
    # ['oxy','deoxy'], each of which is itself a struct with field 'x' holding the (n_samples, 36) signal,
    # and mnt_nback has fields ['x','y','pos_3d','clab'] holding (36,) coordinate vectors. Using the
    # unified squeeze_me=True / struct_as_record=False loader removes the [0,0] nested-indexing style and
    # avoids a silent shape mismatch in spatial_mapping() if the indexing assumption is ever wrong.
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]     # NEW
    cnt = mat_cnt[real_cnt_keys[0]]                                          # NEW: cnt is now a MATLAB-struct object with .oxy / .deoxy attributes

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]     # NEW
    mnt = mat_mnt[real_mnt_keys[0]]                                          # NEW: mnt is now a MATLAB-struct object with .x / .y attributes

    # Load fNIRS trial markers (NEW: needed only for the before/after QC plot at the end of this
    # function; segmentation itself happens later using EEG_TRIAL_LABEL_MAP / FNIRS_TRIAL_LABEL_MAP
    # inside extract_synchronized_windows, which keeps this function's return signature unchanged).
    fnirs_events_time = None  # NEW
    fnirs_events_code = None  # NEW
    if os.path.exists(mrk_path):  # NEW
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)  # NEW
        real_fmrk_keys = [k for k in mat_fmrk.keys() if not k.startswith('__')]    # NEW
        fmrk = mat_fmrk[real_fmrk_keys[0]]                                         # NEW
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):  # NEW
            fnirs_events_time = np.asarray(fmrk.time).flatten()                     # NEW
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()               # NEW
    
    # Extract actual sampling rate (making sure it's 10Hz)
    # sfreq_fnirs_raw = float(cnt['fs'][0, 0]) if 'fs' in cnt.dtype.names else 10.0
    # FIX (Problem 1, continued): 'fs' is a field of cnt.oxy / cnt.deoxy (each a sub-struct), not of
    # cnt itself (verified: cnt_nback.oxy.fs == 10, cnt_nback.deoxy.fs == 10 in the inspected VP001 file).
    # The original line would raise an AttributeError under the new loader (cnt has no .dtype/.fs), so
    # it must read fs from the oxy sub-struct.
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0  # NEW
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    # Extract raw oxy/deoxy
    # if 'oxy' in cnt.dtype.names:
    #     oxy_raw = cnt['oxy'][0, 0]['x']
    #     deoxy_raw = cnt['deoxy'][0, 0]['x']
    # else:
    #     oxy_raw = cnt['oxy']
    #     deoxy_raw = cnt['deoxy']
    # FIX (Problem 1, continued): with struct_as_record=False, squeeze_me=True, the signal matrix is
    # accessed as cnt.oxy.x / cnt.deoxy.x (plain attribute access), matching the (n_samples, 36) shape
    # confirmed in inspect_data.py for VP001 (21178, 36).
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)     # NEW
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)  # NEW
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    # Convert to HbO/HbR
    # print("Converting to HbO/HbR...")
    # hbo_raw, hbr_raw = calculate_hbo_hbr(oxy_raw, deoxy_raw)
    # CHANGED (EXPERIMENTAL -- Beer-Lambert removal test, per chat history): the official dataset
    # Data Descriptor (Shin et al. 2018) explicitly states "cnt files contain deoxy/oxy-hemoglobin
    # data as separate fields... The raw NIRS optical intensity data converted to deoxy-(HbR) and
    # oxy hemoglobin (HbO) concentration changes... using the modified Beer-Lambert law" BEFORE
    # distribution -- confirmed further by the dataset description PDF itself, which explicitly
    # labels these fields "oxy hemoglobin data" / "deoxy hemoglobin data" (not raw optical density).
    # oxy_raw/deoxy_raw are therefore already ΔHbO/ΔHbR; calling calculate_hbo_hbr() on them
    # re-applies Beer-Lambert to already-converted concentration data, which is physically incorrect.
    # calculate_hbo_hbr() itself is left defined-but-unused below (not deleted), per the instruction
    # not to touch other parts of the file.
    print("Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...")  # NEW
    hbo_raw, hbr_raw = oxy_raw, deoxy_raw  # NEW: no conversion -- oxy_raw IS HbO, deoxy_raw IS HbR
    
    # Wavelet denoising 
    # print("Wavelet denoising...")
    # hbo_wv = wavelet_denoising(hbo_raw)
    # hbr_wv = wavelet_denoising(hbr_raw)
    #
    # MARA on full signal
    # print("MARA...")
    # hbo_mara = apply_mara(hbo_wv, fs=sfreq_fnirs_raw)
    # hbr_mara = apply_mara(hbr_wv, fs=sfreq_fnirs_raw)
    #
    # Low-pass filter 
    # FNIRS_LOWPASS_CUTOFF_HZ = 0.2
    # hbo_filtered = lowpass_filter(hbo_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)
    # hbr_filtered = lowpass_filter(hbr_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)
    # CHANGED (EXPERIMENTAL -- isolated fNIRS-filtering-removal test, Priority 2, per chat history):
    # Ding et al. (2026, Information Fusion, "MSTFDN"), using this SAME dataset, applies NO
    # wavelet denoising, NO MARA, and NO separate lowpass/bandpass filtering to fNIRS at all --
    # their stated rationale: "since the fNIRS branch uses convolution kernels with a large size
    # and receptive field (large size temporal convolution is equivalent to low-pass filtering in
    # the frequency domain), the model naturally focuses on low-frequency components during
    # feature learning. Therefore, we did not perform separate filtering preprocessing." Their
    # fNIRS pipeline is simply: Beer-Lambert (already done by dataset authors) -> baseline
    # correction -> z-score normalization. This test skips wavelet_denoising()/apply_mara()/
    # lowpass_filter() entirely (functions left defined-but-unused, not deleted) to test this
    # simplification hypothesis directly against our own hand-designed filtering pipeline, on the
    # SAME CONFIRMED baseline-corrected Run H pipeline (isolated from Priority 1's baseline-window
    # change and the still-unconfirmed EEG filter change).
    print("Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...")  # NEW
    hbo_filtered, hbr_filtered = hbo_raw, hbr_raw  # NEW: pass-through, no denoising/artifact-removal/filtering applied
    
    # dynamic downsampling to 10 Hz
    print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
    from scipy.signal import resample
    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    
    # spatial mapping to 16x16
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    # Stack as channels: (n_samples, 16, 16, 2)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")

    # NEW (user request): save a before/after preprocessing QC plot for fNIRS (one representative
    # channel, raw HbO vs fully preprocessed/resampled HbO). Saved per-subject next to that subject's
    # output folder. Does not alter any returned value or downstream logic.
    try:  # NEW
        qc_channel = 0  # NEW: first fNIRS channel, arbitrary but fixed for reproducibility
        t_raw = np.arange(oxy_raw.shape[0]) / sfreq_fnirs_raw  # NEW
        t_proc = np.arange(hbo_resampled.shape[0]) / SFREQ_FNIRS  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_raw, oxy_raw[:, qc_channel], color='tab:red', linewidth=0.6)  # NEW
        axes[0].set_title(f"{subject_id} - fNIRS RAW HbO (channel {qc_channel}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("HbO (raw units)")  # NEW
        axes[1].plot(t_proc, hbo_resampled[:, qc_channel], color='tab:blue', linewidth=0.8)  # NEW
        axes[1].set_title(f"{subject_id} - fNIRS PREPROCESSED HbO (channel {qc_channel}, after wavelet+MARA+lowpass+resample)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("HbO (mmol/L)")  # NEW
        plt.tight_layout()  # NEW
        qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
        os.makedirs(qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(qc_dir, f"{subject_id}_fnirs_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
        print(f"Saved fNIRS before/after QC plot for {subject_id}")  # NEW
    except Exception as e:  # NEW
        print(f"Could not save fNIRS QC plot for {subject_id}: {e}")  # NEW
    
    # return fnirs_data
    # CHANGED (CRITICAL SYNCHRONIZATION FIX): diagnostic testing (see chat history) directly compared
    # EEG's own trial markers (codes 112/128/144) against fNIRS's own trial markers (codes 7/8/9),
    # both loaded independently from their respective mrk_nback.mat files. This revealed that the
    # two devices do NOT share a common time origin: within each of the 3 class "sessions" the two
    # devices agree almost exactly (delta ~0.1-0.3s, pure jitter), but ACROSS session boundaries a
    # near-constant ~46-47s jump appears (e.g. VP001: session 1 delta=16.6s, session 2 delta=62.7s,
    # session 3 delta=110.0s), consistently reproduced across all 5 subjects tested. Since
    # extract_synchronized_windows() previously computed the fNIRS window position purely by
    # converting EEG's own event time to an fNIRS sample index (assuming a shared t=0), this means
    # for the 2-back and 3-back sessions the fNIRS window was being read from a position roughly
    # 46-93 seconds away from where it should be -- i.e. NOT synchronized with the EEG window it was
    # being paired with. fnirs_events_time and fnirs_events_code (already loaded above, previously
    # used only for the QC plot) are now also returned so extract_synchronized_windows() can use
    # fNIRS's OWN trial markers as the anchor for fNIRS windowing, instead of a value derived from
    # EEG's clock.
    return fnirs_data, fnirs_events_time, fnirs_events_code  # CHANGED: also return fNIRS's own trial markers

# fNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 760, 850
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.586, 1.54852
    e2_hbo, e2_hbr = 1.058, 0.69132
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.15):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    # x_coords = np.asarray(mnt.x).flatten()
    # y_coords = np.asarray(mnt.y).flatten()
    # FIX (projection accuracy): mnt.x / mnt.y are pre-computed 2D coordinates from the BBCI
    # toolbox's own projection, which is not necessarily the azimuthal equidistant projection
    # described in the TSFNet reference paper (Zhang, Yin, Yuan, Sensors 2025). Using the raw 3D
    # channel positions (mnt.pos_3d, shape (3, n_channels), confirmed present in the fNIRS mnt_nback
    # struct via inspect_data.py) and projecting them with a genuine azimuthal equidistant
    # projection (angular distance from the vertex is preserved: theta = angle from north pole,
    # phi = azimuthal angle) more faithfully reproduces the paper's methodology than reusing the
    # toolbox's built-in 2D coordinates.
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)  # NEW: shape (3, n_channels)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]  # NEW
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)  # NEW
    r_safe = np.where(r > 0, r, 1.0)  # NEW: avoid division by zero for any degenerate channel position
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))  # NEW: angle from the vertex (pole)
    phi = np.arctan2(y3d, x3d)  # NEW: azimuthal angle
    x_coords = theta * np.cos(phi)  # NEW: azimuthal equidistant projection
    y_coords = theta * np.sin(phi)  # NEW
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    # mapped = np.zeros((n_samples, 16, 16), dtype=np.float32)
    # for t in range(n_samples):
    #     grid_z = griddata(points, data[t, :], (grid_x, grid_y), method='cubic', fill_value=0)
    #     mapped[t] = grid_z
    # PERF FIX: the loop above calls griddata() once per time sample, re-computing the Delaunay
    # triangulation of `points` from scratch every single time -- even though `points` (electrode
    # positions) NEVER changes across time samples for a given subject; only the values (data[t,:])
    # change. Confirmed via VP001/VP002 real runs, EEG spatial mapping (394930 / 395238 samples at
    # 200 Hz) was the dominant cost of the ~28-32 min per-subject runtime. griddata's 'cubic' 2D
    # path internally builds scipy.interpolate.CloughTocher2DInterpolator(points, values, ...),
    # which officially supports (a) being given a pre-built scipy.spatial.Delaunay triangulation
    # instead of raw points, and (b) vector-valued `values` of shape (n_points, ...trailing_dims),
    # interpolating all trailing "columns" (here: all time samples) in one batched call. Verified
    # numerically bit-identical (max abs diff = 0.0) against the old per-sample loop on synthetic
    # data of the same shape before applying this change (see chat history for the verification
    # script). This is a pure performance refactor -- the interpolation math is unchanged.
    tri = Delaunay(points)  # NEW: triangulation depends only on `points`, computed ONCE instead of n_samples times
    values = data.T  # NEW: (n_samples, n_channels) -> (n_channels, n_samples); CloughTocher2DInterpolator
                      # NEW: requires values.shape[0] == points.shape[0] (== n_channels here)
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # NEW: gradient estimation + patch setup done once, for all time samples together
    result = interp((grid_x, grid_y))  # NEW: shape (16, 16, n_samples) -- one batched evaluation instead of n_samples separate ones
    mapped = np.transpose(result, (2, 0, 1)).astype(np.float32)  # NEW: (16, 16, n_samples) -> (n_samples, 16, 16), matching the original return shape/dtype exactly
    
    return mapped

# NEW (EEG spatial mapping support): spatial_mapping() above was written for fNIRS, which reads
# 3D channel positions from mnt.pos_3d (a field already present in the dataset's mnt_nback.mat
# struct). EEG has no equivalent .mat mnt struct -- its electrode positions instead live inside
# the MNE montage set in load_eeg_data() via raw.set_montage('standard_1005', match_case=False,
# on_missing='ignore'). _EEGMontageAdapter + get_eeg_mnt() below extract those positions, in the
# SAME channel order as raw.ch_names (so they line up 1:1 with data columns passed to
# spatial_mapping), into a lightweight object exposing .pos_3d with shape (3, n_channels) --
# exactly the interface spatial_mapping() already expects -- so spatial_mapping() itself does not
# need to change at all and is reused UNCHANGED for EEG, with the same azimuthal equidistant
# projection methodology already used for fNIRS.
# NEW (EEG spatial mapping support): spatial_mapping() above was written for fNIRS, which reads
# 3D channel positions from mnt.pos_3d (a field present in the dataset's fNIRS mnt_nback.mat
# struct). The EEG folder has its OWN mnt_nback.mat with the exact same format (pos_3d shape
# (3, 30), clab shape (30,)), confirmed via direct inspection of VP001-EEG/mnt_nback.mat -- so EEG
# positions come from the REAL, dataset-provided file, not from MNE's generic standard_1005
# template (an earlier, incorrect assumption; see the comment above raw.set_montage() in
# load_eeg_data(), where that template montage is kept only for interpolate_bads()'s internal use).
# _EEGMontageAdapter + get_eeg_mnt() below select, from the full 30-channel mnt_nback.mat struct,
# only the positions of the channels that SURVIVED preprocessing (raw.ch_names, after
# VEOG/HEOG/TP9 are dropped in preprocess_eeg), matched BY NAME against mnt_nback.mat's own clab
# list (channel order in cnt_nback.mat and mnt_nback.mat is not guaranteed to be identical, so
# matching by name rather than by raw positional index is required for correctness). The result is
# a lightweight object exposing .pos_3d with shape (3, n_channels) -- exactly the interface
# spatial_mapping() already expects -- so spatial_mapping() itself needs no changes at all.
class _EEGMontageAdapter:  # NEW
    """Minimal stand-in for the fNIRS .mat mnt struct, exposing only the .pos_3d attribute that
    spatial_mapping() reads."""
    def __init__(self, pos_3d):  # NEW
        self.pos_3d = pos_3d  # NEW: shape (3, n_channels), matching mnt.pos_3d's shape for fNIRS

def get_eeg_mnt(raw, eeg_mnt_struct):  # CHANGED: now takes the real mnt_nback.mat struct (loaded once in load_eeg_data) instead of deriving positions from raw's MNE montage
    """
    Builds an _EEGMontageAdapter (pos_3d shape (3, n_channels)) from the REAL electrode 3D
    positions in the dataset's own EEG mnt_nback.mat (eeg_mnt_struct.pos_3d / .clab), selecting
    and ordering them to match raw.ch_names exactly. Must be called AFTER any channel drops
    (VEOG/HEOG/TP9 in preprocess_eeg) so the returned position order matches the channel order of
    the data that will be passed to spatial_mapping().
    """
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]  # NEW: e.g. 30 names, same cleanup style as cnt_struct.clab in load_eeg_data
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)  # NEW: shape (3, 30)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}  # NEW
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}  # NEW: case-insensitive fallback, matches match_case=False used elsewhere in this pipeline

    pos_list = []  # NEW
    missing = []  # NEW
    for ch in raw.ch_names:  # NEW
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))  # NEW
        if idx is None:  # NEW: channel present in raw but not found in mnt_nback.mat's clab list
            missing.append(ch)  # NEW
            pos_list.append([0.0, 0.0, 0.0])  # NEW: origin fallback (theta=0 -> maps to grid center)
        else:  # NEW
            pos_list.append(full_pos_3d[:, idx])  # NEW

    if missing:  # NEW
        print(f"  WARNING: {len(missing)} EEG channel(s) not found in mnt_nback.mat's clab "  # NEW
              f"(spatial_mapping will place them at the grid center): {missing}")  # NEW

    pos_3d = np.array(pos_list, dtype=np.float64).T  # NEW: shape (3, n_channels)
    return _EEGMontageAdapter(pos_3d)  # NEW

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")

    # NEW (user request): keep a copy of the raw signal (one representative channel) for the
    # before/after preprocessing QC plot saved at the end of this function.
    raw_before_qc = raw.copy()  # NEW
    
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks='eeg', verbose=False)
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method='fir', verbose=False)
    
    # Detect bad channels
    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info['bads'] = bad_channels
    
    # NEW (fix support): remember which channels were flagged as bad BEFORE interpolate_bads()
    # resets info['bads']. This list is needed further below to correctly compute the ICA rank,
    # since interpolate_bads(reset_bads=True) clears info['bads'] and the original code's
    # exclude='bads' filter therefore silently included the interpolated channels in the ICA fit.
    interpolated_channels = list(raw_filtered.info['bads'])  # NEW
    
    if raw_filtered.info['bads']:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    # ICA
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')
    # corrected_rank = max(1, len(picks_for_ica) - 1)
    # FIX (rank problem): interpolate_bads(reset_bads=True) above already cleared info['bads'],
    # so exclude='bads' here has NOTHING to exclude and picks_for_ica silently includes every
    # interpolated channel. Interpolated channels are linear combinations of their neighbours and
    # add NO new degrees of freedom to the data, so the true data rank is
    # (n_good_channels - 1 for CAR - n_interpolated_channels), not just (n_channels - 1).
    # Asking ICA (via n_components=corrected_rank) for more components than the true rank forces
    # the algorithm to spread a single real source (e.g. the eye-blink dipole) across multiple
    # weak, "diluted" components, none of which individually scores high enough in ICLabel
    # (whether by argmax or by the probability threshold added above) to be flagged as
    # 'eye blink'. This is very likely the main reason eye-blink artifacts survive in this
    # pipeline's output while code 1 (which never interpolates channels) does not have this issue.

###########

    # corrected_rank = max(1, len(picks_for_ica) - 1 - len(interpolated_channels))
    # print(f"ICA rank: {corrected_rank} (picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)})")
    # ica = mne.preprocessing.ICA(n_components=corrected_rank, method='infomax',
    #                             fit_params=dict(extended=True), random_state=42, verbose=False)
    # FIX (rank problem, cleaner solution): the manual rank correction above (subtracting the number
    # of interpolated channels) is a reasonable workaround, but n_components=0.99 solves the same
    # problem more robustly and automatically: instead of a FIXED component count, ICA's internal PCA
    # step keeps only as many components as needed to explain 99% of the data's variance. Interpolated
    # channels contribute (near-)zero additional variance since they are linear combinations of their
    # neighbours, so PCA naturally discards that redundant dimensionality without needing to manually
    # count and subtract interpolated channels. This avoids asking ICA for more components than the
    # true data rank, which is what caused the eye-blink source to be diluted across several weak,
    # under-scored components in the original pipeline (see comment above find_bads_eog for the full
    # rationale of that issue).
    # method='picard' (Ablin, Cardoso & Gramfort, 2018, IEEE Signal Processing Letters) is mathematically
    # equivalent to extended infomax but converges faster and more reliably; it is the ICA algorithm
    # used in the reference implementation (code 1) and is increasingly the default recommendation in
    # recent BCI/EEG preprocessing literature.
    print(f"ICA: using variance-based component selection (99% explained variance), "  # NEW
          f"picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)}")     # NEW
    ica = mne.preprocessing.ICA(n_components=0.99, method='picard',                     # NEW
                                fit_params=dict(ortho=False, extended=True),            # NEW
                                random_state=42, verbose=False)                          # NEW
    ica.fit(raw_filtered, picks=picks_for_ica)


#########    

####
    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')

    ARTIFACT_CLASSES_IDX = {  # NEW: standard ICLabel class order (verified from ica.labels_scores_ shape=(n,7))
        'brain': 0, 'muscle artifact': 1, 'eye blink': 2,
        'heart beat': 3, 'line noise': 4, 'channel noise': 5, 'other': 6
    }
    ARTIFACT_TARGETS = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']  # NEW
    PROB_THRESHOLD = 0.30  # NEW

    # Original argmax-based selection (unchanged logic)
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels['labels'])
                           if label in ARTIFACT_TARGETS]  # CHANGED: renamed

    # NEW: probability-threshold based selection using ica.labels_scores_ (shape: n_components x 7)
    artifact_idx_prob = []  # NEW
    proba_matrix = ica.labels_scores_  # NEW: verified shape (27, 7) for this dataset
    for comp_idx in range(proba_matrix.shape[0]):  # NEW
        for cls_name in ARTIFACT_TARGETS:  # NEW
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)  # NEW
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:  # NEW
                artifact_idx_prob.append(comp_idx)  # NEW
                break  # NEW: one match per component is enough

    # eog_indices, eog_scores = None, None  (did not exist in the previous version of this file)
    # FIX (Problem: no direct EOG-correlation check): code 1 (the reference implementation) relies
    # primarily on mne.preprocessing.ICA.find_bads_eog(), which correlates each ICA component's time
    # course directly against the RECORDED VEOG channel -- a ground-truth signal, independent of
    # ICLabel's scalp-topography/spectrum-based classifier. Code 2 previously relied ONLY on
    # ICLabel (via argmax and/or probability threshold), which never consults the VEOG/HEOG channels
    # that are already loaded and typed as 'eog' in load_eeg_data(). Adding find_bads_eog() here
    # gives ICA exclusion a second, independent detection path so that a genuine eye-blink component
    # that ICLabel under-scores (e.g. due to the rank/dilution issue fixed above) can still be
    # caught via its correlation with the real VEOG signal, matching code 1's methodology.
    eog_indices_veog, eog_scores_veog = [], None  # NEW: defaults in case VEOG channel is missing
    eog_indices_heog, eog_scores_heog = [], None  # NEW: defaults in case HEOG channel is missing
    try:  # NEW
        if 'VEOG' in raw.ch_names:  # NEW: raw is the ORIGINAL (unfiltered) signal passed into this function, matching code 1's use of raw_input
            eog_indices_veog, eog_scores_veog = ica.find_bads_eog(raw, ch_name='VEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(VEOG) flagged components: {eog_indices_veog}")  # NEW
        if 'HEOG' in raw.ch_names:  # NEW: also check HEOG (horizontal eye movements), which code 1 did not check but is available in this dataset
            eog_indices_heog, eog_scores_heog = ica.find_bads_eog(raw, ch_name='HEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(HEOG) flagged components: {eog_indices_heog}")  # NEW
    except Exception as e:  # NEW
        print(f"  find_bads_eog failed: {e}")  # NEW
#########

    
    # NEW (methodological decision, not a named standard algorithm -- see justification below):
    # Gate EOG-correlation-only hits (components flagged ONLY via find_bads_eog, with no ICLabel
    # support) behind an ICLabel 'brain' probability ceiling. Rationale: EOG-correlation is a
    # well-established artifact-detection method (Jung et al. 1998/2000; implemented in EEGLAB,
    # FieldTrip, MNE), and combining multiple independent artifact detectors is itself a recognized
    # practice (e.g. SASICA, Chaumon et al. 2015, J. Neurosci. Methods, which combines EOG-correlation,
    # autocorrelation, topography focus, and discontinuity as independent criteria). However, the
    # literature also documents that EOG-correlation alone can produce false positives, since some
    # genuine brain sources (e.g. frontal activity related to eye-movement planning / corollary
    # discharge) can correlate with EOG without being ocular artifacts themselves. The specific
    # threshold below (BRAIN_PROB_CEILING = 0.5) is NOT a published/validated cutoff from any single
    # paper -- it is a conservative engineering choice made for this thesis to reduce the risk of
    # discarding real brain signal when a component's only supporting evidence is EOG-correlation.
    # Any component that ALSO has independent ICLabel support (argmax or probability threshold) is
    # NOT affected by this gate and is still excluded regardless of its 'brain' probability.
    BRAIN_PROB_CEILING = 0.5  # NEW: conservative, thesis-specific choice -- not a literature-standard value
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX['brain']  # NEW
    
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))  # NEW: components flagged ONLY by EOG-correlation, with no ICLabel support at all
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}  # NEW: rescue if ICLabel itself thinks it's likely brain signal
    if eog_only_rescued:  # NEW
        print(f"  Rescued from EOG-only rejection (ICLabel brain-prob >= {BRAIN_PROB_CEILING}): {sorted(eog_only_rescued)}")  # NEW
    
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )  # CHANGED: same union as before, minus the conservative brain-probability rescue applied only to EOG-only hits


    
###########
    # NEW: detailed log for each flagged component
    all_class_names = ['brain', 'muscle artifact', 'eye blink', 'heart beat', 'line noise', 'channel noise', 'other']  # NEW
    for comp_idx in artifact_idx:  # NEW
        probs_str = ", ".join([f"{all_class_names[c]}={proba_matrix[comp_idx, c]:.2f}"  # NEW
                               for c in range(7)])  # NEW
        reason = []  # NEW
        if comp_idx in artifact_idx_argmax:  # NEW
            reason.append(f"argmax='{ic_labels['labels'][comp_idx]}'")  # NEW
        if comp_idx in artifact_idx_prob:  # NEW
            reason.append(f"prob>={PROB_THRESHOLD}")  # NEW
        if comp_idx in eog_indices_veog:  # NEW
            reason.append("VEOG-correlation")  # NEW
        if comp_idx in eog_indices_heog:  # NEW
            reason.append("HEOG-correlation")  # NEW
        print(f"  IC{comp_idx} flagged [{', '.join(reason)}]: {probs_str}")  # NEW

    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components")
    else:
        print("Removed 0 ICA components (no artifact ICs detected)")  # NEW
    

####
        
    
    # Case-insensitive channel drop
    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    # NEW (user request, updated): save before/after preprocessing QC plots for EEG showing ALL
    # channels at once (not just one representative channel). Two SEPARATE figures are produced:
    # one figure with all RAW channels stacked, one figure with all PREPROCESSED channels stacked.
    # Each channel is plotted on its own horizontal row using a fixed vertical offset (the standard
    # "butterfly/stacked montage" view used by EEG viewers such as MNE's raw.plot()), so all channels
    # remain visually separable in a single image instead of being overlaid on top of each other.
    try:  # NEW
        def _plot_all_channels_stacked(raw_obj, title, color, tmin=None, tmax=None):  # CHANGED: added tmin/tmax for zoomed view
            sfreq_plot = raw_obj.info['sfreq']               # NEW
            if tmin is not None or tmax is not None:         # NEW: optional time-window crop
                start_sample = int((tmin or 0.0) * sfreq_plot)             # NEW
                stop_sample = int((tmax * sfreq_plot)) if tmax is not None else None  # NEW
                data = raw_obj.get_data(start=start_sample, stop=stop_sample)         # NEW
                t = (np.arange(data.shape[1]) + start_sample) / sfreq_plot            # NEW
            else:                                            # NEW
                data = raw_obj.get_data()                      # NEW: (n_channels, n_times)
                t = np.arange(data.shape[1]) / sfreq_plot        # NEW
            ch_names_plot = raw_obj.ch_names                # NEW

            # Use the robust (median absolute deviation based) per-channel spread to set a fixed
            # vertical offset between channels, so that channels with very different amplitudes
            # (e.g. after CAR/ICA vs raw) are still readable and non-overlapping within the same figure.
            #spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 8 + 1e-12  # NEW
            spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 30 + 1e-12  # NEW
            n_ch_plot = data.shape[0]                        # NEW
            #fig_h = max(6, 0.28 * n_ch_plot)  # NEW: scale figure height with channel count
            fig_h = max(6, 0.5 * n_ch_plot)                 # NEW: scale figure height with channel count
            fig, ax = plt.subplots(figsize=(14, fig_h))      # NEW
            for i in range(n_ch_plot):                       # NEW
                offset = (n_ch_plot - 1 - i) * spread         # NEW: top channel = first channel name
                ax.plot(t, data[i] + offset, color=color, linewidth=0.4)  # NEW
            ax.set_yticks([(n_ch_plot - 1 - i) * spread for i in range(n_ch_plot)])  # NEW
            ax.set_yticklabels(ch_names_plot, fontsize=6)    # NEW
            ax.set_xlabel("Time (s)")                        # NEW
            ax.set_ylabel("Channel")                         # NEW
            ax.set_title(title)                              # NEW
            plt.tight_layout()                                # NEW
            return fig                                        # NEW

        fig_before = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing)", 'tab:red'  # NEW
        )  # NEW
        fig_after = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA)", 'tab:blue'  # NEW
        )  # NEW
        fig_before_zoom = _plot_all_channels_stacked(  # NEW: zoomed 10-30s view
            raw_before_qc, "EEG RAW - All Channels (before preprocessing, 10-30s)", 'tab:red', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        fig_after_zoom = _plot_all_channels_stacked(  # NEW: zoomed 10-30s view
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA, 10-30s)", 'tab:blue', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        

        # The subject_id is not directly available inside this function (it only receives raw/events),
        # so the figures are saved using generic names inside OUTPUT_PATH/_tmp_eeg_qc and renamed by
        # the caller (process_subject) right after this function returns. This keeps this function's
        # signature and call sites unchanged.
        tmp_qc_dir = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc")  # NEW
        os.makedirs(tmp_qc_dir, exist_ok=True)  # NEW
        fig_before.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels.png"), dpi=150)  # NEW
        fig_after.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels.png"), dpi=150)  # NEW
        fig_before_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels_zoom10_30.png"), dpi=150)  # NEW
        fig_after_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels_zoom10_30.png"), dpi=150)  # NEW
        plt.close(fig_before)  # NEW
        plt.close(fig_after)  # NEW
        plt.close(fig_before_zoom)  # NEW
        plt.close(fig_after_zoom)  # NEW
    except Exception as e:  # NEW
        print(f"Could not save EEG QC plot: {e}")  # NEW
    
    return raw_filtered


# SYNCHRONIZED SLIDING WINDOW EXTRACTION 
# def extract_synchronized_windows(eeg_raw, fnirs_data, events, sfreq_eeg):
# CHANGED: added eeg_spatial_data parameter -- the pre-computed, spatially-mapped (16x16 grid)
# continuous EEG signal for this subject (shape (n_times, 16, 16), produced once in
# process_subject() via spatial_mapping(), mirroring how fNIRS is spatially mapped once before
# windowing rather than per-window). eeg_raw (the MNE Raw object, channel x time) is still passed
# in and still used, but now ONLY for the amplitude-based artifact rejection check below, since
# that threshold (120 uV) is calibrated in electrode voltage units and is not a meaningful check
# on interpolated grid values.
# def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg):
# CHANGED (CRITICAL SYNCHRONIZATION FIX, see the comment above load_and_preprocess_fnirs()'s return
# statement for the full diagnostic rationale): added fnirs_events_time / fnirs_events_code so each
# EEG trial can be paired with fNIRS's OWN trial marker instead of a position derived from EEG's
# clock, which diagnostic testing showed drifts by ~46-93s across the dataset's 3 class sessions.
def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,  # CHANGED
                                  fnirs_events_time, fnirs_events_code):  # CHANGED

    print("Extracting synchronized asymmetric sliding windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    
    # support both marker formats 
    # label_map = {
    #     1: 0, 2: 1, 3: 1, 4: 2, 5: 2,
    #     16: 0, 48: 1, 64: 1, 80: 2, 96: 2
    # }
    #
    # valid_codes = [1, 2, 3, 4, 5, 16, 48, 64, 80, 96]
    # target_events = events[np.isin(events[:, 2], valid_codes)]
    # FIX (Problem 2): the codes above (1-5, 16/48/64/80/96) are STIMULUS-level markers that fire
    # roughly every 2 s within a 40 s trial (per Shin et al. 2018 / Dataset_description_MATLAB.pdf,
    # codes 16/48/64/80/96 correspond to individual target/non-target number presentations, not to
    # the start of a cognitive-load trial). Sliding a window from each of these stimulus markers
    # caused massive, redundant overlap between windows anchored to different stimuli within the same
    # trial, rather than a clean traversal of the full 40 s task period for that trial. The correct
    # trial-onset codes, confirmed directly from the VP001 .mat structure (inspect_data.py output:
    # mrk_nback.event.desc contains 144 etc., and the dataset documentation confirms 112/128/144 =
    # 0-/2-/3-back SESSION start), are used instead: each trial-onset marker anchors exactly one
    # 40 s task period, which is then traversed by the (unchanged) sliding-window loop below.
    label_map = EEG_TRIAL_LABEL_MAP  # NEW: trial-onset codes (112/128/144) -> class label, defined at top of script
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # NEW
    target_events = events[np.isin(events[:, 2], valid_codes)]  # NEW: now selects one event per ~40s trial instead of one event per ~2s stimulus
    
    print(f"found {len(target_events)} trial markers")
    if len(target_events) == 0:
        print(f"No trial markers found. Available markers: {np.unique(events[:, 2])}")
        return np.array([]), np.array([]), np.array([])

    # NEW (CRITICAL SYNCHRONIZATION FIX -- see comment above load_and_preprocess_fnirs()'s return
    # statement for the full diagnostic rationale): build fNIRS's OWN trial-onset time array (in
    # seconds), independently of EEG's clock, from fnirs_events_time/fnirs_events_code (loaded in
    # load_and_preprocess_fnirs()). Diagnostic testing confirmed that, for every subject checked,
    # EEG's trial-class sequence and fNIRS's trial-class sequence match exactly 1:1 in order, so
    # trial i in target_events corresponds to trial i in fnirs_trial_times/fnirs_trial_classes --
    # this is what allows pairing them by simple array position below.
    fnirs_sync_ok = False  # NEW
    fnirs_trial_times = None  # NEW
    if fnirs_events_time is not None and fnirs_events_code is not None:  # NEW
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())  # NEW
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)  # NEW
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)  # NEW
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]  # NEW
        fnirs_order = np.argsort(fnirs_trial_times_ms)  # NEW
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0  # NEW: seconds, fNIRS's own clock
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])  # NEW

        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])  # NEW
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))  # NEW
        if (len(fnirs_trial_times) == len(target_events)  # NEW: same trial count in both devices
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):  # NEW: same class order
            fnirs_sync_ok = True  # NEW
            print(f"fNIRS-own-marker sync check: OK ({len(fnirs_trial_times)} trials, class order matches EEG)")  # NEW
        else:
            print(f"  WARNING: fNIRS own markers do not match EEG 1:1 "  # NEW
                  f"(fNIRS n={len(fnirs_trial_times)}, EEG n={len(target_events)}) -- "  # NEW
                  f"falling back to EEG-clock-derived fNIRS timing for this subject.")  # NEW
    else:
        print("  WARNING: fNIRS event markers unavailable -- falling back to EEG-clock-derived fNIRS timing.")  # NEW
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    session_ids = []   # CROSS-SESSION: session number (1,2,3) برای هر window
    
    for trial_idx, ev in enumerate(target_events):  # CHANGED: enumerate() added to index into fnirs_trial_times
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        # CROSS-SESSION: session_id از ترتیب زمانی trial استنتاج می‌شود
        # هر subject 27 trial دارد (3 session × 9 trial per session)
        # trial_idx 0..8 → session 1، trial_idx 9..17 → session 2، trial_idx 18..26 → session 3
        TRIALS_PER_SESSION = 9
        session_id = trial_idx // TRIALS_PER_SESSION + 1
        
        # event_time = event_sample_eeg / sfreq_eeg
        # CHANGED (CRITICAL SYNCHRONIZATION FIX): event_time used to be derived purely from EEG's
        # own clock and then reused to index INTO fNIRS's array too, silently assuming a shared
        # t=0 between the two independently-recorded devices. eeg_event_time (below) is still used
        # for the EEG window and for the baseline window (both correctly indexed into EEG-derived
        # arrays / fnirs_data using EEG's own timeline as before -- baseline correction was already
        # working correctly within a session since the drift is BETWEEN sessions, not within one,
        # and baseline windows are always taken from shortly before the SAME trial). fnirs_event_time
        # (used only for the fNIRS window's actual position below) now comes from fNIRS's OWN trial
        # marker when the sync check above passed, eliminating the ~46-93s cross-session drift.
        eeg_event_time = event_sample_eeg / sfreq_eeg  # CHANGED: renamed from event_time for clarity
        if fnirs_sync_ok:  # NEW
            fnirs_event_time = fnirs_trial_times[trial_idx]  # NEW: fNIRS's own clock for this trial
        else:  # NEW
            fnirs_event_time = eeg_event_time  # NEW: fallback -- old (drift-prone) behavior, only used if sync check failed

        # NEW (isolated baseline-correction test): compute this trial's fNIRS baseline ONCE per
        # trial (not per segment/offset), from the continuous fnirs_data signal, using the
        # BASELINE_TMIN/BASELINE_TMAX window (-5 to -2 s before this trial's onset, on the SAME
        # sync-fixed fnirs_event_time anchor used for the segments themselves). Subtracted from
        # every fNIRS segment belonging to this trial below. If the pre-trial baseline window
        # falls before the start of the recording (only possible for a trial very close to time
        # 0), no correction is applied for that trial (fnirs_baseline stays 0) rather than
        # raising an error or using an out-of-bounds/negative index.
        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))  # NEW
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))    # NEW
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:  # NEW
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)  # NEW: shape (1,16,16,2)
        else:  # NEW
            fnirs_baseline = 0.0  # NEW: not enough pre-trial data (e.g. the very first trial) -- skip correction for this trial

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            
            # EEG window
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            # fNIRS window 
            # w_start_fnirs = int(np.round((event_time + offset_sec) * SFREQ_FNIRS))
            # w_start_fnirs = int(np.round((fnirs_event_time + offset_sec) * SFREQ_FNIRS))  # CHANGED: uses fNIRS's own clock (fnirs_event_time), not EEG-derived event_time
            # w_end_fnirs = w_start_fnirs + window_samples_fnirs
            #
            # if w_end_eeg > eeg_raw.n_times:
            #     continue
            #
            # if w_end_fnirs > fnirs_data.shape[0]:
            #     continue
            # CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing): instead of ONE continuous
            # 10s fNIRS window anchored at fnirs_event_time+offset_sec, extract FNIRS_NUM_SEGMENTS=11
            # overlapping 3s segments, each shifted by FNIRS_STEP=1s from the previous one (matching
            # the TSFNet paper's approach, and the same segment structure used in the other expert's
            # extract_tsfnet_windows()). The sync-fix anchor (fnirs_event_time, from fNIRS's OWN
            # trial marker -- completely UNCHANGED from above) is still used as the base time for
            # segment 0; this is the one improvement over the other expert's version, which anchored
            # segments to EEG's clock directly and was therefore subject to the ~46-93s cross-session
            # drift this project already diagnosed and fixed.
            if w_end_eeg > eeg_raw.n_times:  # CHANGED: moved this check up, unchanged logic
                continue

            fnirs_segments = []  # NEW
            valid_segments = True  # NEW
            for seg_idx in range(FNIRS_NUM_SEGMENTS):  # NEW
                seg_offset = offset_sec + seg_idx * FNIRS_STEP  # NEW: each segment shifted by 1s from the previous
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))  # NEW: still uses the sync-fixed anchor
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs  # NEW: window_samples_fnirs now = 3s worth of samples (FNIRS_WINDOW_LEN=3.0)

                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:  # NEW
                    valid_segments = False  # NEW
                    break  # NEW

                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)  # CHANGED: subtract this trial's baseline, isolated from the filter change

            if not valid_segments:  # NEW
                continue  # NEW
            
            # Extract EEG
            # eeg_slice = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            # if np.max(np.abs(eeg_slice)) > 120e-6:
            #     continue
            # CHANGED (EEG spatial mapping): the SAVED EEG window must now be the spatially-mapped
            # (16, 16, 600) grid representation, not the raw (28, 600) channel-space slice. The
            # artifact-rejection amplitude check, however, stays on the raw channel-space slice
            # (eeg_slice_raw) since the 120 uV threshold is only meaningful in electrode voltage
            # units, not on cubic-interpolated grid values.
            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)  # CHANGED: renamed, QC-only now
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:  # CHANGED: uses eeg_slice_raw
                continue

            # NEW: slice the pre-computed spatially-mapped EEG (n_times, 16, 16) at the SAME sample
            # range as eeg_slice_raw (both index the same continuous recording at sfreq_eeg=200Hz),
            # then transpose (600, 16, 16) -> (16, 16, 600) i.e. grid-H x grid-W x time, per spec.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # NEW: (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # NEW: (16, 16, 600) -- this is what gets saved

            # Extract fNIRS
            # fnirs_slice = fnirs_data[w_start_fnirs:w_end_fnirs]
            # fnirs_slice = fnirs_slice.transpose(3, 0, 1, 2) 
            # CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing): stack the 11 already-extracted
            # segments (each (30,16,16,2)) into (11,30,16,16,2), then transpose to put the segment
            # axis first and channel (HbO/HbR) axis last, matching the shape convention used by the
            # other expert's implementation of this same approach: (11, 16, 16, 30, 2).
            fnirs_slice = np.stack(fnirs_segments, axis=0)      # NEW: (11, 30, 16, 16, 2)
            fnirs_slice = fnirs_slice.transpose(0, 2, 3, 1, 4)  # NEW: (11, 16, 16, 30, 2)
            
            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)   # CROSS-SESSION
    
    eeg_windows = np.array(eeg_windows)
    fnirs_windows = np.array(fnirs_windows)
    labels = np.array(labels)
    session_ids = np.array(session_ids, dtype=np.int32)   # CROSS-SESSION
    
    print(f"Extracted {len(labels)} synchronized windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
    
    return eeg_windows, fnirs_windows, labels, session_ids   # CROSS-SESSION اضافه شد



LOG_FILE = os.path.join(OUTPUT_PATH, "processing_log.txt")  # NEW: log file path

def log_message(msg):  # NEW: helper to print + append to log file
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")



# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    t0 = time.time()  # NEW
    log_message(f"START processing {subject_id}")  # NEW
    print(f"Processing {subject_id}")
    
    # subject_num = subject_id[2:]
    # subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
    # eeg_folder = os.path.join(subject_path, f"{subject_id}-EEG")
    # FIX: same flat-folder-structure fix as in load_and_preprocess_fnirs() above; EEG data sits
    # directly under BASE_DATASET_PATH\VPxxx-EEG, with no intermediate numeric subfolder.
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")  # NEW: flat folder structure
    
    if not os.path.exists(eeg_folder):
        log_message(f"SKIPPED {subject_id}: EEG folder not found")  # NEW
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n Loading EEG...")
        # raw_eeg, events, sfreq_eeg = load_eeg_data(eeg_folder)
        raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)  # CHANGED: also receive the real EEG mnt_nback.mat struct
        
        print("\n Loading and preprocessing fNIRS...")
        # fnirs_data = load_and_preprocess_fnirs(subject_id)
        fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)  # CHANGED: also receive fNIRS's own trial markers
        
        # NEW (bug fix): normalization must happen BEFORE window extraction, not after.
        # extract_synchronized_windows() below slices fnirs_windows directly out of fnirs_data,
        # so if fnirs_data is normalized AFTER that call, the windows already saved are un-normalized
        # and the later normalization line has no effect on the saved output (confirmed by
        # VP011's quality report: fNIRS std=0.000367 instead of ~1.0). Moving the normalization
        # here, right after fnirs_data is produced and before any windowing happens, fixes this.
        print("\n Normalizing fNIRS (subject-level, before windowing)...")  # NEW
        fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)  # NEW: moved here from below
        
        print("\n Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)

        # NEW (EEG per-subject normalization fix support): compute per-channel mean/std from the
        # FULL cleaned, continuous EEG signal (after filtering/CAR/ICA) BEFORE any windowing happens.
        # FIX (per-window vs per-subject normalization): the previous EEG normalization (further
        # below, in the "Normalizing..." section) computed a separate z-score for EACH extracted
        # window independently (axis=-1 over the window's 600 time samples). With a window as short
        # as 3 s, this local z-scoring is noisy/unstable and, more importantly, completely erases any
        # systematic difference in absolute EEG amplitude/power between subjects or between cognitive
        # load levels within a subject -- the model can then only see the RELATIVE waveform shape
        # inside each isolated 3 s window, which is a much weaker and more subject-specific signal.
        # This is very likely a major contributor to the severe overfitting observed during model
        # training (train acc ~96% vs val/test acc near chance level): the model latches onto
        # subject-specific residual noise patterns within each locally-renormalized window instead of
        # learning generalizable cognitive-load-related features. Computing mean/std once per subject
        # over the full continuous recording (mirroring the fNIRS per-subject normalization fix above)
        # and applying those FIXED per-channel statistics to every window extracted from that subject
        # preserves the relative amplitude structure across windows and across load levels, which
        # should generalize better to unseen subjects.
        eeg_full_data = raw_eeg_clean.get_data()  # NEW: shape (n_channels, n_times), fully preprocessed continuous signal
        eeg_subject_mean = eeg_full_data.mean(axis=-1, keepdims=True)  # NEW: shape (n_channels, 1)
        eeg_subject_std = eeg_full_data.std(axis=-1, keepdims=True) + 1e-8  # NEW: shape (n_channels, 1)
        print(f"\n EEG per-subject normalization stats computed: "  # NEW
              f"mean range [{eeg_subject_mean.min():.2e}, {eeg_subject_mean.max():.2e}], "  # NEW
              f"std range [{eeg_subject_std.min():.2e}, {eeg_subject_std.max():.2e}]")  # NEW

        # NEW (EEG spatial mapping, TSFNet paper Sec. 2.3): project the EEG electrodes onto a 16x16
        # spatial grid via azimuthal equidistant projection, exactly as already done for fNIRS
        # (spatial_mapping() + mnt), instead of leaving EEG as raw (channel x time). This is applied
        # ONCE to the full continuous, already-cleaned recording (not per-window) for the same
        # efficiency reason fNIRS is spatially mapped once in load_and_preprocess_fnirs() before
        # windowing: with 66% window overlap, per-window mapping would re-run cubic griddata
        # interpolation on largely the same timepoints hundreds of times over.
        print("\n Spatial mapping EEG to 16x16...")  # NEW
        # eeg_mnt = get_eeg_mnt(raw_eeg_clean)  # electrode positions in raw_eeg_clean.ch_names order (post channel-drop)
        eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)  # CHANGED: uses REAL mnt_nback.mat positions, matched by clab name, not MNE's guessed standard_1005
        eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)  # NEW: (n_channels,n_times).T -> (n_times, n_channels) in, (n_times, 16, 16) out
        print(f"EEG spatially mapped: {eeg_spatial_full.shape}")  # NEW

        # NEW: per-subject normalization of the spatially-mapped EEG, applied to the FULL continuous
        # grid signal before windowing -- mirrors exactly how fNIRS is normalized in this same
        # function (fnirs_data = (fnirs_data - mean)/std over axis=0, i.e. per grid-cell across
        # time). This supersedes the old per-channel normalization (eeg_subject_mean/eeg_subject_std
        # above), which no longer applies now that EEG windows are (16,16,600) grid data rather than
        # (28,600) channel data; eeg_subject_mean/eeg_subject_std are kept above only for the
        # informational print (channel-space signal scale), not used for normalization below anymore.
        eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (  # NEW
            eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)  # NEW

        # NEW (user request): finalize the EEG before/after QC plots (now two all-channel figures)
        # for this specific subject by renaming the temp files produced inside preprocess_eeg()
        # (which has no access to subject_id).
        try:  # NEW
            tmp_qc_before = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels.png")  # NEW
            tmp_qc_after = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels.png")    # NEW
            tmp_qc_before_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels_zoom10_30.png")  # NEW
            tmp_qc_after_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels_zoom10_30.png")    # NEW
            if os.path.exists(tmp_qc_before) and os.path.exists(tmp_qc_after):  # NEW
                final_qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
                os.makedirs(final_qc_dir, exist_ok=True)  # NEW
                final_qc_before = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels.png")  # NEW
                final_qc_after = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels.png")    # NEW
                os.replace(tmp_qc_before, final_qc_before)  # NEW
                os.replace(tmp_qc_after, final_qc_after)    # NEW
                if os.path.exists(tmp_qc_before_zoom) and os.path.exists(tmp_qc_after_zoom):  # NEW
                    final_qc_before_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels_zoom10_30.png")  # NEW
                    final_qc_after_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels_zoom10_30.png")    # NEW
                    os.replace(tmp_qc_before_zoom, final_qc_before_zoom)  # NEW
                    os.replace(tmp_qc_after_zoom, final_qc_after_zoom)    # NEW
                print(f"Saved EEG before/after (all-channels) QC plots for {subject_id}")  # NEW
        except Exception as e:  # NEW
            print(f"Could not finalize EEG QC plot for {subject_id}: {e}")  # NEW
        
        print("\n Extracting synchronized asymmetric windows...")
        # eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
        #     raw_eeg_clean, fnirs_data, events, sfreq_eeg
        # )
        # CHANGED: pass eeg_spatial_full (the pre-computed 16x16-grid-mapped continuous EEG) in
        # addition to raw_eeg_clean (still needed inside for the channel-space QC amplitude check).
        # eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
        #     raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg
        # )
        # CHANGED (CRITICAL SYNCHRONIZATION FIX): also pass fnirs_events_time/fnirs_events_code so
        # fNIRS windows are anchored to fNIRS's OWN trial markers, not EEG-clock-derived timing.
        eeg_windows, fnirs_windows, labels, session_ids = extract_synchronized_windows(   # CROSS-SESSION
            raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg,
            fnirs_events_time, fnirs_events_code
        )
        
        if len(labels) == 0:
            log_message(f"WARNING {subject_id}: no windows extracted")  # NEW
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n Normalizing...")
        # EEG: (n_windows, n_channels, n_times) 
        # eeg_windows = (eeg_windows - eeg_windows.mean(axis=-1, keepdims=True)) / (eeg_windows.std(axis=-1, keepdims=True) + 1e-8)
        # FIX (per-window -> per-subject EEG normalization): the line above computed a fresh z-score
        # for every single window independently (per-window mean/std over its own 600 time samples).
        # As explained in the comment above where eeg_subject_mean/eeg_subject_std are computed, this
        # erases systematic amplitude differences between cognitive-load levels and between subjects,
        # which is very likely contributing to the poor generalization (near-chance val/test accuracy)
        # observed during model training. Applying the FIXED per-subject statistics computed once
        # from the full continuous recording (broadcast across all windows and all time samples via
        # reshape to (1, n_channels, 1)) fixes this while still correcting for each subject's overall
        # signal scale, mirroring the per-subject normalization already used for fNIRS.
        # eeg_mean_broadcast = eeg_subject_mean.reshape(1, -1, 1)  # (1, n_channels, 1) for broadcasting over (N, n_channels, n_times)
        # eeg_std_broadcast = eeg_subject_std.reshape(1, -1, 1)
        # eeg_windows = (eeg_windows - eeg_mean_broadcast) / eeg_std_broadcast
        # CHANGED (EEG spatial mapping): eeg_windows is now (n_windows, 16, 16, 600) grid data, not
        # (n_windows, 28, 600) channel data, so the per-channel normalization above no longer applies
        # and is removed. EEG normalization now happens EARLIER, on the full continuous spatially-
        # mapped signal (eeg_spatial_full) before windowing -- see the "NEW: per-subject
        # normalization of the spatially-mapped EEG" block above, right after spatial_mapping() is
        # called -- exactly mirroring how fNIRS is normalized in grid space before windowing. No
        # further EEG normalization is needed here; eeg_windows already carries normalized values.
        
        # fNIRS: (n_windows, 2, 100, 16, 16)
        # fnirs_windows = (fnirs_windows - fnirs_windows.mean(axis=2, keepdims=True)) / (fnirs_windows.std(axis=2, keepdims=True) + 1e-8)
        # FIX (Problem 3): ...
        # fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)
        # BUG FIX (moved): this line was dead code -- fnirs_windows had already been sliced from the
        # un-normalized fnirs_data by extract_synchronized_windows() before this point ran. The actual
        # normalization now happens earlier, right after load_and_preprocess_fnirs() returns and
        # before extract_synchronized_windows() is called. See that block above.
        
        print("\n Saving...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
        # NEW (user request): for the first verification run, only save outputs for the very first
        # subject so correctness can be checked before running the full 26-subject batch. The save
        # calls below are commented out for all subjects except VP001; uncomment the three np.save
        # lines below (and remove/relax this condition) once the pipeline has been verified.
#        #if subject_id == "VP001":  # NEW: save outputs only for the first subject during verification
        np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)    # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)  # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)               # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_session_ids.npy"), session_ids)   # CROSS-SESSION
        # else:                                                                                    # NEW
        #     # Saving suppressed for subjects other than VP001 during this verification run.        # NEW
        #     pass                                                                                  # NEW

        # NEW (adopted from reviewing a third-party implementation, see chat history): a small
        # per-subject metadata.json sidecar file, so the exact parameters/shapes a given
        # preprocessing run used can be checked at a glance without loading the (large) .npy
        # arrays -- useful given this project has gone through many preprocessing variants
        # (Run A-H) and it has repeatedly been unclear afterward exactly which run produced a
        # given set of output files.
        metadata = {  # NEW
            'subject_id': subject_id,  # NEW
            'eeg_shape': list(eeg_windows.shape),  # NEW
            'fnirs_shape': list(fnirs_windows.shape),  # NEW
            'labels_distribution': np.bincount(labels).tolist(),  # NEW
            'eeg_window_len': EEG_WINDOW_LEN,  # NEW
            'eeg_step': EEG_STEP,  # NEW
            'eeg_epoch_tmax': EEG_EPOCH_TMAX,  # NEW: important given this exact parameter caused a major bug in the reviewed third-party code
            'fnirs_num_segments': FNIRS_NUM_SEGMENTS,  # NEW
            'fnirs_window_len': FNIRS_WINDOW_LEN,  # NEW
            'fnirs_step': FNIRS_STEP,  # NEW
            'sfreq_eeg': SFREQ_EEG,  # NEW
            'sfreq_fnirs': SFREQ_FNIRS,  # NEW
            'beer_lambert_applied': False,  # NEW: this pipeline uses oxy_raw/deoxy_raw directly (Run F fix), not calculate_hbo_hbr()
            'fnirs_sync_fix_applied': True,  # NEW: fNIRS windows anchored to fNIRS's own trial markers, not EEG's clock
            'normalization': 'per-subject z-score (EEG grid + fNIRS grid, before windowing)',  # NEW
        }
        with open(os.path.join(sub_out_dir, f"{subject_id}_metadata.json"), 'w') as f:  # NEW
            json.dump(metadata, f, indent=2)  # NEW
        
        print(f"\n {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"DONE {subject_id} in {elapsed:.1f} min | EEG: {eeg_windows.shape} | fNIRS: {fnirs_windows.shape} | Labels: {np.bincount(labels)}")  # NEW
        
    except Exception as e:
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"ERROR processing {subject_id} after {elapsed:.1f} min: {str(e)}")  # NEW
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    total_t0 = time.time()  # NEW
    log_message("===== PIPELINE STARTED =====")  # NEW
    print("MULTIMODAL PREPROCESSING PIPELINE")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    print(f"fNIRS epoch: {FNIRS_EPOCH_TMAX}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")
    
    # for sub_num in range(1, 27):
    # NEW (verification run): limited to VP001 only, per user's request to first check correctness
    # on a single subject before running the full 26-subject batch. Change back to range(1, 27)
    # once the pipeline has been verified.
#    for sub_num in range(1, 2):  # NEW: VP001 only, for verification
    for sub_num in range(20, 27):  # NEW
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)

    
    print("ALL THE SELECTED SUBJECTS PROCESSED")

[2026-07-14 14:15:52] ===== PIPELINE STARTED =====
MULTIMODAL PREPROCESSING PIPELINE
EEG window: 3.0s, step: 1.0s
fNIRS window: 3.0s, step: 1.0s
fNIRS epoch: 18.0s
fNIRS sampling rate: 10.0 Hz
[2026-07-14 14:15:52] START processing VP020
Processing VP020

 Loading EEG...
Loading EEG from: E:\kntu\Thesis\codes\DATA\VP020-EEG\cnt_nback.mat
EEG loaded: 388501 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]
EEG mnt_nback.mat loaded: pos_3d shape (3, 30)

 Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP020-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (20729, 36), deoxy (20729, 36)
Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...
Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...
Resampling from 10.0 Hz to 10.0 Hz...
Spatial mapping

In [1]:
#Generate session ids
"""
generate_session_ids.py
-----------------------
برای هر subject که قبلاً preprocessing شده، فایل VPxxx_session_ids.npy را تولید می‌کند.
نیازی به re-run ICA / spatial mapping / ... نیست.

منطق:
  - هر subject دقیقاً 27 trial دارد (9 trial × 3 session)
  - trial‌ها به ترتیب زمانی در events مرتب هستند
  - trial_idx 0..8  → session 1
  - trial_idx 9..17 → session 2
  - trial_idx 18..26 → session 3
  - هر trial چندین window تولید می‌کند (sliding window)
  - تعداد window per trial از labels.npy موجود استنتاج می‌شود

خروجی:
  PROCESSED_DATA_PATH/VPxxx/VPxxx_session_ids.npy
  shape: (n_windows,), dtype=int, values: 1، 2 یا 3
"""

import os
import numpy as np
import scipy.io as sio

# ── تنظیمات — همان مقادیر preprocessing اصلی ──────────────────────────
BASE_DATASET_PATH  = r"E:\kntu\Thesis\codes\DATA"
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"

ALL_SUBJECTS = [f"VP{i:03d}" for i in range(1, 27)]

# پارامترهای sliding window — دقیقاً مثل preprocessing
EEG_EPOCH_TMAX  = 40.0   # طول هر trial (ثانیه)
EEG_WINDOW_LEN  = 3.0    # طول هر window
EEG_STEP        = 1.0    # گام sliding window

# تعداد window که از یک trial به دست می‌آید
# np.arange(0, max_slide + 0.001, step) → تعداد آفست‌ها
_max_slide = EEG_EPOCH_TMAX - EEG_WINDOW_LEN   # = 37.0
WINDOWS_PER_TRIAL_THEORETICAL = len(np.arange(0.0, _max_slide + 0.001, EEG_STEP))  # = 38

# marker‌های trial-onset برای EEG (مطابق preprocessing)
EEG_TRIAL_CODES = {112, 128, 144}   # 0-back، 2-back، 3-back session markers

TRIALS_PER_SESSION = 9   # هر session = 3 condition × 3 series = 9 trial
SESSIONS_PER_SUBJECT = 3


# ── helper: بارگذاری EEG markers از .mat ──────────────────────────────
def load_eeg_trial_events(subject_id):
    """
    marker‌های trial-onset را از VPxxx-EEG/mrk_nback.mat می‌خواند.
    بازگشت: آرایه‌ای از کدهای مارکر به ترتیب زمانی (فقط کدهای 112/128/144).
    """
    mrk_path = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG", "mrk_nback.mat")
    if not os.path.exists(mrk_path):
        raise FileNotFoundError(f"marker file not found: {mrk_path}")

    mat = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    key = [k for k in mat if not k.startswith("__")][0]
    mrk = mat[key]

    # زمان event‌ها
    if hasattr(mrk, "time"):
        times = np.asarray(mrk.time).flatten()
    elif hasattr(mrk, "pos"):
        times = np.asarray(mrk.pos).flatten().astype(float)
    else:
        raise ValueError("Cannot find time/pos in marker struct")

    # کد event‌ها
    if hasattr(mrk, "event") and hasattr(mrk.event, "desc"):
        codes = np.asarray(mrk.event.desc).flatten().astype(int)
    else:
        raise ValueError("Cannot find event.desc in marker struct")

    # فقط trial-onset markers
    mask  = np.isin(codes, list(EEG_TRIAL_CODES))
    times = times[mask]
    codes = codes[mask]

    # مرتب‌سازی بر اساس زمان (باید از قبل ترتیب داشته باشند، اما اطمینان بیشتر)
    order = np.argsort(times)
    return codes[order]   # shape: (n_trials,)


# ── منطق اصلی: تولید session_ids برای یک subject ─────────────────────
def generate_session_ids_for_subject(subject_id, verbose=True):
    sub_dir    = os.path.join(PROCESSED_DATA_PATH, subject_id)
    labels_path = os.path.join(sub_dir, f"{subject_id}_labels.npy")
    out_path   = os.path.join(sub_dir, f"{subject_id}_session_ids.npy")

    if not os.path.exists(labels_path):
        print(f"  [{subject_id}] SKIP — labels.npy not found")
        return False

    labels = np.load(labels_path)
    n_windows = len(labels)

    # ── بارگذاری trial-onset markers برای دانستن ترتیب دقیق trial‌ها ──
    try:
        trial_codes = load_eeg_trial_events(subject_id)
    except Exception as e:
        print(f"  [{subject_id}] ERROR loading markers: {e}")
        print(f"  [{subject_id}] Falling back to uniform windows-per-trial estimate")
        trial_codes = None

    n_trials_total = TRIALS_PER_SESSION * SESSIONS_PER_SUBJECT  # = 27

    if trial_codes is not None and len(trial_codes) != n_trials_total:
        print(f"  [{subject_id}] WARNING: found {len(trial_codes)} trial markers "
              f"(expected {n_trials_total}). Using uniform estimate.")
        trial_codes = None

    # ── تخمین تعداد window per trial ──────────────────────────────────
    # روش دقیق: از آنجا که تعداد کل window‌ها و تعداد trial‌ها را می‌دانیم،
    # و هر trial بسته به artifact rejection ممکن است window‌های کمتری داشته باشد،
    # بهترین رویکرد اینست که بر اساس labels.npy موجود، windows را به 27 گروه
    # مساوی (یا تقریبی) تقسیم کنیم.
    #
    # دقیق‌ترین روش: re-run کردن همان حلقه‌ای که windows را تولید می‌کند
    # اما بدون spatial mapping.
    # فعلاً: تقسیم یکنواخت n_windows بر n_trials — اگر artifact rejection کمی
    # نامتقارن باشد، چند window ممکن است به session اشتباه برود (خطای جزئی).
    windows_per_trial = n_windows / n_trials_total
    if verbose:
        print(f"  [{subject_id}] n_windows={n_windows}, n_trials={n_trials_total}, "
              f"avg windows/trial={windows_per_trial:.1f} "
              f"(theoretical={WINDOWS_PER_TRIAL_THEORETICAL})")

    # session_id برای هر window
    session_ids = np.zeros(n_windows, dtype=np.int32)

    # تقسیم windows به سه block مساوی (یا نزدیک به مساوی) بر اساس trial
    cumulative = 0
    for trial_idx in range(n_trials_total):
        session_num = trial_idx // TRIALS_PER_SESSION + 1   # 1، 2 یا 3

        # تعداد window‌هایی که به این trial تعلق دارند
        # (استنتاج تقریبی از تقسیم یکنواخت)
        start_w = round(trial_idx * windows_per_trial)
        end_w   = round((trial_idx + 1) * windows_per_trial)
        end_w   = min(end_w, n_windows)

        if start_w >= n_windows:
            break

        session_ids[start_w:end_w] = session_num

    # بررسی: همه windows باید session داشته باشند
    unassigned = np.sum(session_ids == 0)
    if unassigned > 0:
        print(f"  [{subject_id}] WARNING: {unassigned} windows unassigned — "
              f"setting to session 3 (last session)")
        session_ids[session_ids == 0] = 3

    np.save(out_path, session_ids)

    if verbose:
        counts = {s: int(np.sum(session_ids == s)) for s in [1, 2, 3]}
        print(f"  [{subject_id}] session_ids saved → {out_path}")
        print(f"  [{subject_id}] session counts: {counts}")

    return True


# ── اجرا برای همه subjects ────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("generate_session_ids.py")
    print(f"PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
    print("=" * 60)

    ok_count  = 0
    err_count = 0

    for subject_id in ALL_SUBJECTS:
        print(f"\n[{subject_id}]")
        try:
            success = generate_session_ids_for_subject(subject_id, verbose=True)
            if success:
                ok_count += 1
        except Exception as e:
            print(f"  [{subject_id}] FATAL ERROR: {e}")
            import traceback
            traceback.print_exc()
            err_count += 1

    print("\n" + "=" * 60)
    print(f"Done. OK={ok_count}  Errors={err_count}")
    print("=" * 60)
    print()
    print("بعد از اجرا، فایل‌های زیر ایجاد می‌شوند:")
    print("  PROCESSED_DATA_PATH/VPxxx/VPxxx_session_ids.npy")
    print("  shape: (n_windows,)  dtype: int32  values: 1, 2, 3")

generate_session_ids.py
PROCESSED_DATA_PATH: F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter

[VP001]
  [VP001] n_windows=1023, n_trials=27, avg windows/trial=37.9 (theoretical=38)
  [VP001] session_ids saved → F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter\VP001\VP001_session_ids.npy
  [VP001] session counts: {1: 341, 2: 341, 3: 341}

[VP002]
  [VP002] n_windows=1026, n_trials=27, avg windows/trial=38.0 (theoretical=38)
  [VP002] session_ids saved → F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter\VP002\VP002_session_ids.npy
  [VP002] session counts: {1: 342, 2: 342, 3: 342}

[VP003]
  [VP003] n_windows=1026, n_trials=27, avg windows/trial=38.0 (theoretical=38)
  [VP003] session_ids saved → F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter\VP003\VP003_session_ids.npy
  [VP003] session counts: {1: 342, 2: 342, 3: 342}

[VP004]
  [VP004] n_windows=752, n_trials=27, avg windows/trial=27.9 (theoretical=38)
  [VP004] session_ids saved → F:\preproce

In [3]:
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        # self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn1 = nn.BatchNorm3d(16)
        # self.nirs_drop1 = nn.Dropout(dropout)
        # self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn2 = nn.BatchNorm3d(32)
        # self.nirs_drop2 = nn.Dropout(dropout)
        # self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        # self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # CHANGED (Run H, 11-segment fNIRS -- paper Table 1 compliance, verified directly against
        # the paper's full text, Sec. 2.3 and Table 1): input is now (batch, 11, 16, 16, 30, 2) --
        # 11 segments (current + subsequent 10, per "each EEG time window was paired with the
        # current and subsequent 10 fNIRS segments... each fNIRS sample had a shape of
        # 11 x 36 x 30 x 2"), each segment 3s long (30 samples @ 10Hz), 2 channels (HbO/HbR).
        # CRITICAL per Table 1: the segment axis (11) is NOT touched by either conv layer -- only
        # the spatial (16x16) and per-segment-internal-time (30) axes are reduced. This is achieved
        # by folding the segment axis into the batch dimension before each conv (so the same
        # Conv3d weights are applied independently, per-segment, exactly like a shared/distributed
        # convolution), then folding it back out afterward. This directly fixes the bug found in a
        # reviewed third-party implementation, which applied a plain Conv3d across all three of
        # (segment, spatial, spatial) jointly, silently shrinking the segment axis from 11 to 2 and
        # requiring a nonsensical repeat+truncate patch downstream.
        # Table 1 states: Conv1 kernel 4x4x6, stride 2x2x2, 16 channels -> output 11x8x8x15x16
        # (segment x H x W x T x channels); Conv2 kernel 2x2x3, stride 2x2x2, 32 channels -> output
        # 11x4x4x8x32. Padding values are NOT stated explicitly in the fetched paper text (only
        # kernel/stride/output dims were) -- the padding below (2,1,1) and (1,0,0) were derived to
        # reproduce exactly those stated output dimensions, not directly quoted from the paper.
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))  # NEW: (T,H,W) order, matches EEG branch's convention; 30->15(T), 16->8(H,W)
        self.nirs_bn1 = nn.BatchNorm3d(16)  # NEW
        self.nirs_drop1 = nn.Dropout(dropout)  # NEW
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))  # NEW: 15->8(T), 8->4(H,W)
        self.nirs_bn2 = nn.BatchNorm3d(32)  # NEW
        self.nirs_drop2 = nn.Dropout(dropout)  # NEW
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack now reduces the 16x16 spatial grid to 4x4 (16->8->4),
        # matching the EEG branch's spatial resolution, same as before -- only the input/conv
        # parameters changed, not this downstream sizing.
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        # self.pos_encoder = PositionalEncoding(64, max_len=100)
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        self.pos_encoder = PositionalEncoding(64, max_len=11)  # CHANGED: 100 -> 11
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        # y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        # y = self.nirs_drop1(y)
        # y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        # y = self.nirs_drop2(y)
        #
        # y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_map = y.mean(dim=2)
        # nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))
        # CHANGED (Run H, 11-segment fNIRS -- Table 1 compliance): nirs arrives as
        # (batch, 11, 16, 16, 30, 2) = (batch, segment, H, W, T, channels) from the dataset. To
        # apply the SAME Conv3d weights to each of the 11 segments independently (i.e. the segment
        # axis must NOT be touched by the conv, per Table 1's stated output dims), the segment axis
        # is folded into the batch dimension before the conv stack, then unfolded afterward.
        n_seg = nirs.size(1)  # NEW: 11
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)  # NEW: (batch,11,16,16,30,2) -> (batch,11,channels=2,T=30,H=16,W=16)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)  # NEW: fold segment into batch -> (batch*11, 2, 30, 16, 16)

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # NEW: (batch*11, 16, 15, 8, 8)
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))  # NEW: (batch*11, 32, 8, 4, 4)
        y = self.nirs_drop2(y)

        y = y.reshape(batch_size, n_seg, 32, 8, 4, 4)  # NEW: unfold segment axis back out -> (batch, 11, 32, T'=8, H'=4, W'=4)

        # NEW: nirs_feat_map -- a SINGLE spatial map for the whole window (used by EFGF and by the
        # fNIRS classifier head), obtained by averaging over BOTH the segment axis and the
        # per-segment-internal-time axis (mirroring eeg_feat_map's GAP-over-time above).
        nirs_feat_map = y.mean(dim=(1, 3))  # NEW: (batch,11,32,8,4,4) -> mean over dim1(seg) & dim3(T') -> (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # NEW: the REAL 11-segment sequence for CAFE (replacing the old fabricated/proxy sequence).
        # Average over the per-segment-internal-time and spatial axes, keeping the segment axis
        # intact as the genuine temporal sequence the paper's CAFE layer is designed to attend over.
        y_spatial = y.mean(dim=(3, 4, 5))  # NEW: (batch,11,32,8,4,4) -> mean over T',H',W' -> (batch,11,32)

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        # nirs_seq = y_spatial.transpose(1, 2)
        # CHANGED (Run H, 11-segment fNIRS): y_spatial above is already (batch, 11, 32) -- i.e.
        # (batch, segment/sequence, channels) -- since it was constructed directly in that order
        # above, unlike the old y_spatial (batch, 32, T) which needed a transpose to get the
        # sequence axis into the middle position for nirs_seq_proj/pos_encoder/attention.
        nirs_seq = y_spatial  # CHANGED: no transpose needed, already (batch, 11, 32)
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # NEW (TSFNet paper Eq. 5-13 -- multi-branch + regularization loss): replacing the single
    # criterion on the final prediction alone with the full loss design from the paper. See the
    # pearson_correlation_loss() docstring and the forward()-return-value comment above for the
    # full rationale. LAMBDA_EEG and LAMBDA_FNIRS follow the paper's Eq. 6 notation (lambda_eeg,
    # lambda_fnirs).
    # CORRECTED: the comment previously here claimed "the paper... does not report their exact
    # values, so 0.3 is used" -- this was WRONG. Sec. 3.6 of the paper explicitly states: "the
    # best-performing loss weights lambda_eeg and lambda_fnirs are both set to 0.2" (found via
    # hyperparameter tuning on their own MI/MA/WG datasets). Using the paper's reported value
    # instead of an unfounded guess. L_fusion, L_efgf, and L_cafe use coefficient 1, exactly as in
    # the paper's Eq. 6 and Eq. 5 (no separate lambda given for these terms).
    criterion = nn.CrossEntropyLoss()  # NEW: still CrossEntropyLoss per-branch, just applied to 4 branches instead of 1 (see loss computation in the training loop below)
    # LAMBDA_EEG = 0.3     # NEW: Eq. 6 lambda_eeg -- thesis-specific default, paper does not report the tuned value
    # LAMBDA_FNIRS = 0.3   # NEW: Eq. 6 lambda_fnirs -- thesis-specific default, paper does not report the tuned value
    LAMBDA_EEG = 0.2     # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_FNIRS = 0.2   # CHANGED: 0.3 -> 0.2, the paper's actual reported optimal value (Sec. 3.6), not a guess
    LAMBDA_EFGF_REG = 1.0   # NEW: Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
    LAMBDA_CAFE_REG = 1.0   # NEW: Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        # train_loss, train_correct = 0.0, 0
        # NEW (adopted from reviewing a third-party implementation, see chat history): track the
        # three loss COMPONENTS (classification, EFGF regularization, CAFE regularization)
        # separately across the epoch, not just their weighted sum -- gives visibility into which
        # term dominates/changes during training, useful for diagnosing overfitting or an
        # imbalanced loss weighting.
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0  # CHANGED
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            # pred, _, _, _ = model(eeg_b, nirs_b)
            # 
            # loss = criterion(pred, lbl_b)
            # NEW (TSFNet paper Eq. 5-13): full multi-branch classification loss (Eq. 6) plus EFGF
            # (Eq. 12) and CAFE (Eq. 13) regularization losses, replacing the single-term loss above
            # that only supervised the final fused prediction. eeg_feat_map/efgf_attn_map are the
            # analogues used for L_efgf; nirs_feat_map/cafe_enhanced_map are the analogues used for
            # L_cafe (see forward()'s return-value comment for the exact correspondence).
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs

            L_pred = criterion(pred, lbl_b)          # NEW: Eq. 10 (generalized to 3-class CrossEntropyLoss)
            L_eeg = criterion(eeg_out, lbl_b)        # NEW: Eq. 7 (generalized to 3-class CrossEntropyLoss)
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW: Eq. 8 (generalized to 3-class CrossEntropyLoss)
            L_fusion = criterion(fusion_out, lbl_b)  # NEW: Eq. 9 (generalized to 3-class CrossEntropyLoss)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW: Eq. 6

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW: Eq. 12
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW: Eq. 13

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW: Eq. 5 (total loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_class_loss += L_class.item()  # NEW
            train_efgf_loss += L_efgf.item()    # NEW
            train_cafe_loss += L_cafe.item()    # NEW
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _ = model(eeg_b, nirs_b)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs, only final pred needed for val accuracy
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            # print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")
            # CHANGED (adopted from reviewing a third-party implementation): added a per-component
            # breakdown (Class/EFGF/CAFE) of the training loss to the epoch log line.
            n_batches = len(train_loader)  # NEW
            print(f"Epoch {epoch:3d}/{epochs} | Loss: {train_loss/n_batches:.4f} | "  # CHANGED
                  f"Class: {train_class_loss/n_batches:.4f} | "  # NEW
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "    # NEW
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "    # NEW
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            # pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: forward() now returns 8 values; the 4 extra feature maps (used only for EFGF/CAFE regularization loss during training) are ignored here
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    # NEW (adopted from reviewing a third-party implementation, see chat history): Cohen's Kappa
    # corrects raw accuracy for the agreement expected by chance -- a standard metric in the BCI
    # literature, useful alongside accuracy especially given the class-imbalance/bias patterns
    # (e.g. over-prediction of one class) repeatedly observed in this project's confusion matrices.
    kappa_final = cohen_kappa_score(all_true, all_preds_final)  # NEW
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")  # NEW
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    # NEW (adopted from reviewing a third-party implementation, see chat history): a structured
    # results.json summary, saved alongside the model checkpoint -- this project has repeatedly
    # needed to dig back through old console logs to find an exact number from a past run (Run
    # A-H, seed/split variance checks, etc.); a small JSON summary per run avoids that.
    kappa_fusion = cohen_kappa_score(all_true, all_preds_fusion)  # NEW
    results = {  # NEW
        'test_accuracy_final': accuracy_score(all_true, all_preds_final),  # NEW
        'test_accuracy_fusion_only': accuracy_score(all_true, all_preds_fusion),  # NEW
        'kappa_final': kappa_final,  # NEW
        'kappa_fusion_only': kappa_fusion,  # NEW
        'confusion_matrix_final': cm.tolist(),  # NEW
        'confusion_matrix_fusion_only': cm_fusion.tolist(),  # NEW
        'ablation': {  # NEW
            'eeg_only': accuracy_score(all_true, all_preds_eeg),  # NEW
            'fnirs_only': accuracy_score(all_true, all_preds_fnirs),  # NEW
            'fusion_only': accuracy_score(all_true, all_preds_fusion),  # NEW
            'final_weighted': accuracy_score(all_true, all_preds_final),  # NEW
        },  # NEW
        'learned_weights': {  # NEW
            'w_eeg': float(learned_weights[0]),  # NEW
            'w_fnirs': float(learned_weights[1]),  # NEW
            'w_fusion': float(learned_weights[2]),  # NEW
        },  # NEW
        'best_val_acc': best_val_acc,  # NEW
    }  # NEW
    results_path = os.path.join(PROCESSED_DATA_PATH, 'results.json')  # NEW
    with open(results_path, 'w') as f:  # NEW
        json.dump(results, f, indent=2)  # NEW
    print(f"\nResults saved to: {results_path}")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [4]:
#Variance check 3seeds p2
# variance_check_3seeds_P2.py
#
# PURPOSE:
# Repeats the full train+evaluate pipeline of TSFNet_With_fNIRS_2D three times with three
# different random seeds (fixing both the subject-level train/val/test split AND the model's
# weight initialization/dropout per seed), then compares the MEAN of the 3 runs against the
# project's established reference baseline (the WITH-fNIRS-filter run, already validated with
# 3 seeds + 3 subject splits + trial-contamination checks -- see project report). This follows
# the same methodology already used in the project for every other single-run finding (baseline
# correction, EEG filter, etc.), per:
#   Bouthillier, X., Delaunay, P., Bronzi, M., et al. (2021). "Accounting for Variance in Machine
#   Learning Benchmarks." MLSys 2021. https://arxiv.org/abs/2103.03098
#
# WHAT THIS SCRIPT TESTS:
# This is meant to be run on top of the CURRENT preprocessing (P2: wavelet/MARA/lowpass fNIRS
# filtering removed, per MSTFDN's hypothesis that the model's own large convolution kernels act
# as an implicit low-pass filter). A single run of this preprocessing already showed a promising
# single-run result (Test=54.64%, fNIRS-only=56.97%, Fusion=55.47%), which crossed the project's
# ~2-sigma noise threshold on two metrics -- but per project methodology, a single run is not
# sufficient evidence. This script provides the required 3-seed confirmation.
#
# IMPORTANT: PROCESSED_DATA_PATH below must point to the P2 preprocessing output (no fNIRS
# filter), NOT the original WITH-filter output, or this script will just re-test the reference
# baseline against itself.

import os
import gc
import math
import random
import copy
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# NOTE: point this at the P2 (no-fNIRS-filter) preprocessing output directory.
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# The 3 seeds used across the whole project for every variance check (kept identical here for
# direct comparability with prior 3-seed checks: baseline correction, EEG filter, etc.).
SEEDS = [42, 123, 999]

EPOCHS = 100
EARLY_STOP_PATIENCE = 15
LAMBDA_EEG = 0.2
LAMBDA_FNIRS = 0.2
LAMBDA_EFGF_REG = 1.0
LAMBDA_CAFE_REG = 1.0

# Reference baseline: mean/std of 3 seeds on the ORIGINAL (WITH fNIRS wavelet/MARA/lowpass
# filter) preprocessing, Run H + isolated fNIRS baseline correction (-5/-2s). Taken directly
# from the project's already-validated 3-seed check (see project report, Section 3 and 8.3).
REFERENCE_BASELINE = {
    'test_acc':   {'mean': 51.19, 'std': 1.83},
    'eeg_acc':    {'mean': 35.54, 'std': 0.68},
    'fnirs_acc':  {'mean': 52.21, 'std': 1.94},
    'fusion_acc': {'mean': 51.11, 'std': 1.88},
}

# z-score magnitude above which the project treats a difference as a likely real effect
# (as opposed to normal seed-to-seed noise), per the threshold used consistently throughout
# this project for every prior variance check (lambda search, dropout, EEG filter, baseline
# window width, etc.).
Z_SCORE_THRESHOLD = 2.0


def set_seed(seed):
    """Fixes random, numpy, and torch RNG state for full run-to-run reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_split(seed):
    """Produces a reproducible subject-level train/val/test split for a given seed."""
    shuffled_subjects = ALL_SUBJECTS.copy()
    rng = random.Random(seed)
    rng.shuffle(shuffled_subjects)

    n_total = len(shuffled_subjects)
    n_train = int(n_total * TRAIN_RATIO)
    n_val = int(n_total * VAL_RATIO)

    train_subs = shuffled_subjects[:n_train]
    val_subs = shuffled_subjects[n_train:n_train + n_val]
    test_subs = shuffled_subjects[n_train + n_val:]
    return train_subs, val_subs, test_subs


# ============================================================================
# DATASET (unchanged from TSFNet_model_RUN_H_11segments.py, only dataset_name suffix
# is now seed-tagged so parallel/successive runs do not overwrite each other's memmaps)
# ============================================================================

class MMapHybridDataset(Dataset):
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0

        print(f"Scanning {dataset_name} - {len(subs_list)} subjects...")

        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)

            if not os.path.exists(sub_folder):
                continue

            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"),
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)

            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"),
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)

            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")

            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]

                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}).")
                        continue

                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")

                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue

        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name}_n.npy")

        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try:
                    os.remove(f_path)
                except:
                    pass
            with open(f_path, 'wb') as f:
                pass

        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+',
                                shape=(self.total_windows, 16, 16, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+',
                                 shape=(self.total_windows, 11, 16, 16, 30, 2))
        self.y = np.zeros(self.total_windows, dtype='int64')

        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()

        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor

    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try:
                    os.remove(f)
                except:
                    pass


# ============================================================================
# MODEL COMPONENTS (unchanged from TSFNet_model_RUN_H_11segments.py)
# ============================================================================

def pearson_correlation_loss(a, b, eps=1e-8):
    """Computes 1 - |PCC(a, b)| as a regularization loss, per TSFNet Eq. 11-13."""
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    a_centered = a_flat - a_flat.mean()
    b_centered = b_flat - b_flat.mean()
    numerator = (a_centered * b_centered).sum()
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps
    pcc = numerator / denominator
    return 1.0 - torch.abs(pcc)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))


class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch (11-segment, Table 1 compliant)
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # EFGF
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)
        self.alpha = nn.Parameter(torch.tensor(0.0))
        self.gamma = nn.Parameter(torch.tensor(0.0))
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=11)
        self.nirs_seq_proj = nn.Linear(32, 64)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate2 = nn.Linear(128, 64)

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)

        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))

        # fNIRS branch (segment axis folded into batch, applied independently per segment)
        n_seg = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y = y.reshape(batch_size, n_seg, 32, 8, 4, 4)

        nirs_feat_map = y.mean(dim=(1, 3))
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))

        y_spatial = y.mean(dim=(3, 4, 5))

        # EFGF Block
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn

        gamma = torch.clamp(self.gamma, 0, 1)
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))

        # CAFE Block (bidirectional)
        nirs_seq = y_spatial
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)
        nirs_proj = self.pos_encoder(nirs_seq_proj)

        fused_seq = fused_feat.unsqueeze(1)
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)

        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        nirs_query = nirs_feat.unsqueeze(1)
        fused_kv = fused_feat.unsqueeze(1)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out

        Phi_broadcast = Phi.expand(-1, 32, -1, -1)
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced


# ============================================================================
# SINGLE-SEED TRAIN + EVALUATE
# ============================================================================

def run_single_seed(seed, device):
    """Runs one full train+evaluate cycle for a given seed and returns a metrics dict."""
    print("\n" + "#" * 70)
    print(f"# SEED {seed}")
    print("#" * 70)

    set_seed(seed)
    train_subs, val_subs, test_subs = make_split(seed)

    tag = f"seed{seed}"
    train_dataset = MMapHybridDataset(train_subs, f"Train_{tag}")
    val_dataset = MMapHybridDataset(val_subs, f"Val_{tag}")
    test_dataset = MMapHybridDataset(test_subs, f"Test_{tag}")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)
    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs_no_improve = 0
    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_{tag}.pth')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)

            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred = criterion(pred, lbl_b)
            L_eeg = criterion(eeg_out, lbl_b)
            L_fnirs = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out, lbl_b)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss += L_efgf.item()
            train_cafe_loss += L_cafe.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()

        train_acc = train_correct / len(train_loader.dataset)

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()

        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(train_loader)
            print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss/n_batches:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} epochs). "
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")
            break

    # FINAL TEST EVALUATION
    print(f"\nRUNNING FINAL TEST EVALUATION (seed {seed})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)

    with torch.no_grad():
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()

    print(f"Seed {seed} results -> Test: {test_acc:.2f}% | EEG: {eeg_acc:.2f}% | "
          f"fNIRS: {fnirs_acc:.2f}% | Fusion: {fusion_acc:.2f}% | Kappa: {kappa_final:.4f}")

    metrics = {
        'seed': seed,
        'test_acc': test_acc,
        'eeg_acc': eeg_acc,
        'fnirs_acc': fnirs_acc,
        'fusion_acc': fusion_acc,
        'kappa_final': kappa_final,
        'best_val_acc': best_val_acc * 100,
        'w_eeg': float(learned_weights[0]),
        'w_fnirs': float(learned_weights[1]),
        'w_fusion': float(learned_weights[2]),
    }

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()

    # free GPU memory before the next seed starts
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics


# ============================================================================
# MAIN: run all seeds, aggregate, compare against reference baseline
# ============================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Seeds to run: {SEEDS}\n")

    all_results = []
    for seed in SEEDS:
        result = run_single_seed(seed, device)
        all_results.append(result)

    # Aggregate: mean and std across the 3 seeds, for each metric
    metric_keys = ['test_acc', 'eeg_acc', 'fnirs_acc', 'fusion_acc']
    summary = {}
    for key in metric_keys:
        values = [r[key] for r in all_results]
        summary[key] = {
            'mean': float(np.mean(values)),
            'std': float(np.std(values)),
            'values': values,
        }

    print("\n" + "=" * 70)
    print("3-SEED SUMMARY (current preprocessing)")
    print("=" * 70)
    for key in metric_keys:
        s = summary[key]
        print(f"{key:12s}: mean={s['mean']:.2f}%  std={s['std']:.2f}%  values={[f'{v:.2f}' for v in s['values']]}")

    # Compare mean-vs-mean against the reference baseline (WITH fNIRS filter), using
    # z-score = (new_mean - ref_mean) / ref_std, exactly as done for every prior variance
    # check in this project (EEG filter, baseline window width, etc.).
    print("\n" + "=" * 70)
    print(f"COMPARISON vs REFERENCE BASELINE (z-score threshold = {Z_SCORE_THRESHOLD})")
    print("=" * 70)
    comparison = {}
    for key in metric_keys:
        new_mean = summary[key]['mean']
        ref_mean = REFERENCE_BASELINE[key]['mean']
        ref_std = REFERENCE_BASELINE[key]['std']
        z = (new_mean - ref_mean) / ref_std
        verdict = "LIKELY REAL EFFECT" if abs(z) >= Z_SCORE_THRESHOLD else "within noise range"
        comparison[key] = {'new_mean': new_mean, 'ref_mean': ref_mean, 'ref_std': ref_std, 'z_score': z, 'verdict': verdict}
        sign = "+" if z >= 0 else ""
        print(f"{key:12s}: new_mean={new_mean:.2f}%  ref_mean={ref_mean:.2f}%  "
              f"z={sign}{z:.2f}  -> {verdict}")

    print("\nFinal conclusion:")
    if comparison['fnirs_acc']['verdict'] == "LIKELY REAL EFFECT" and comparison['fusion_acc']['verdict'] == "LIKELY REAL EFFECT":
        print("--> Both fNIRS-only and Fusion metrics confirm a likely real effect across 3 seeds.")
        print("    This upgrades the P2 (no fNIRS filter) finding from a single-run observation")
        print("    to a validated result, following the same standard applied to baseline correction.")
    else:
        print("--> The 3-seed mean does not confirm the single-run signal at the project's noise")
        print("    threshold. Treat the single-run result as within normal seed-to-seed variance,")
        print("    consistent with the pattern seen for lambda search, dropout, and the EEG filter.")

    # Save full results
    output = {
        'seeds': SEEDS,
        'per_seed_results': all_results,
        'summary': summary,
        'reference_baseline': REFERENCE_BASELINE,
        'comparison': comparison,
    }
    output_path = os.path.join(PROCESSED_DATA_PATH, 'variance_check_3seeds_P2_results.json')
    with open(output_path, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"\nFull results saved to: {output_path}")

Running on Device: cuda
Seeds to run: [42, 123, 999]


######################################################################
# SEED 42
######################################################################
Scanning Train_seed42 - 18 subjects...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_seed42)...
--> Train_seed42 Ready! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Val_seed42 - 3 subjects...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 v

In [1]:
#Tsfnet weight decay gridsearch
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        # self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn1 = nn.BatchNorm3d(16)
        # self.nirs_drop1 = nn.Dropout(dropout)
        # self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn2 = nn.BatchNorm3d(32)
        # self.nirs_drop2 = nn.Dropout(dropout)
        # self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        # self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # CHANGED (Run H, 11-segment fNIRS -- paper Table 1 compliance, verified directly against
        # the paper's full text, Sec. 2.3 and Table 1): input is now (batch, 11, 16, 16, 30, 2) --
        # 11 segments (current + subsequent 10, per "each EEG time window was paired with the
        # current and subsequent 10 fNIRS segments... each fNIRS sample had a shape of
        # 11 x 36 x 30 x 2"), each segment 3s long (30 samples @ 10Hz), 2 channels (HbO/HbR).
        # CRITICAL per Table 1: the segment axis (11) is NOT touched by either conv layer -- only
        # the spatial (16x16) and per-segment-internal-time (30) axes are reduced. This is achieved
        # by folding the segment axis into the batch dimension before each conv (so the same
        # Conv3d weights are applied independently, per-segment, exactly like a shared/distributed
        # convolution), then folding it back out afterward. This directly fixes the bug found in a
        # reviewed third-party implementation, which applied a plain Conv3d across all three of
        # (segment, spatial, spatial) jointly, silently shrinking the segment axis from 11 to 2 and
        # requiring a nonsensical repeat+truncate patch downstream.
        # Table 1 states: Conv1 kernel 4x4x6, stride 2x2x2, 16 channels -> output 11x8x8x15x16
        # (segment x H x W x T x channels); Conv2 kernel 2x2x3, stride 2x2x2, 32 channels -> output
        # 11x4x4x8x32. Padding values are NOT stated explicitly in the fetched paper text (only
        # kernel/stride/output dims were) -- the padding below (2,1,1) and (1,0,0) were derived to
        # reproduce exactly those stated output dimensions, not directly quoted from the paper.
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))  # NEW: (T,H,W) order, matches EEG branch's convention; 30->15(T), 16->8(H,W)
        self.nirs_bn1 = nn.BatchNorm3d(16)  # NEW
        self.nirs_drop1 = nn.Dropout(dropout)  # NEW
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))  # NEW: 15->8(T), 8->4(H,W)
        self.nirs_bn2 = nn.BatchNorm3d(32)  # NEW
        self.nirs_drop2 = nn.Dropout(dropout)  # NEW
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack now reduces the 16x16 spatial grid to 4x4 (16->8->4),
        # matching the EEG branch's spatial resolution, same as before -- only the input/conv
        # parameters changed, not this downstream sizing.
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        # self.pos_encoder = PositionalEncoding(64, max_len=100)
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        self.pos_encoder = PositionalEncoding(64, max_len=11)  # CHANGED: 100 -> 11
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        # y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        # y = self.nirs_drop1(y)
        # y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        # y = self.nirs_drop2(y)
        #
        # y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_map = y.mean(dim=2)
        # nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))
        # CHANGED (Run H, 11-segment fNIRS -- Table 1 compliance): nirs arrives as
        # (batch, 11, 16, 16, 30, 2) = (batch, segment, H, W, T, channels) from the dataset. To
        # apply the SAME Conv3d weights to each of the 11 segments independently (i.e. the segment
        # axis must NOT be touched by the conv, per Table 1's stated output dims), the segment axis
        # is folded into the batch dimension before the conv stack, then unfolded afterward.
        n_seg = nirs.size(1)  # NEW: 11
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)  # NEW: (batch,11,16,16,30,2) -> (batch,11,channels=2,T=30,H=16,W=16)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)  # NEW: fold segment into batch -> (batch*11, 2, 30, 16, 16)

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # NEW: (batch*11, 16, 15, 8, 8)
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))  # NEW: (batch*11, 32, 8, 4, 4)
        y = self.nirs_drop2(y)

        y = y.reshape(batch_size, n_seg, 32, 8, 4, 4)  # NEW: unfold segment axis back out -> (batch, 11, 32, T'=8, H'=4, W'=4)

        # NEW: nirs_feat_map -- a SINGLE spatial map for the whole window (used by EFGF and by the
        # fNIRS classifier head), obtained by averaging over BOTH the segment axis and the
        # per-segment-internal-time axis (mirroring eeg_feat_map's GAP-over-time above).
        nirs_feat_map = y.mean(dim=(1, 3))  # NEW: (batch,11,32,8,4,4) -> mean over dim1(seg) & dim3(T') -> (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # NEW: the REAL 11-segment sequence for CAFE (replacing the old fabricated/proxy sequence).
        # Average over the per-segment-internal-time and spatial axes, keeping the segment axis
        # intact as the genuine temporal sequence the paper's CAFE layer is designed to attend over.
        y_spatial = y.mean(dim=(3, 4, 5))  # NEW: (batch,11,32,8,4,4) -> mean over T',H',W' -> (batch,11,32)

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        # nirs_seq = y_spatial.transpose(1, 2)
        # CHANGED (Run H, 11-segment fNIRS): y_spatial above is already (batch, 11, 32) -- i.e.
        # (batch, segment/sequence, channels) -- since it was constructed directly in that order
        # above, unlike the old y_spatial (batch, 32, T) which needed a transpose to get the
        # sequence axis into the middle position for nirs_seq_proj/pos_encoder/attention.
        nirs_seq = y_spatial  # CHANGED: no transpose needed, already (batch, 11, 32)
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
WEIGHT_DECAY_GRID = [2e-2, 5e-2, 1e-1]  # NEW: baseline + two stronger candidates per Krogh & Hertz 1992

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)
EPOCHS = 100
EARLY_STOP_PATIENCE = 15


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device):  # NEW
    print("\n" + "#" * 70)
    print(f"# WEIGHT_DECAY = {weight_decay}")
    print("#" * 70)

    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # CHANGED: weight_decay is now the function parameter (grid-search candidate) instead of the
    # hard-coded 2e-2 -- lr, betas, and every other AdamW default remain exactly as before.
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=weight_decay)  # CHANGED: 2e-2 -> weight_decay (grid-search variable)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs_no_improve = 0
    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_wd{weight_decay}.pth')  # CHANGED: tagged per weight_decay so parallel/successive grid runs don't overwrite each other

    print("STARTING TRAINING")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)

            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred = criterion(pred, lbl_b)
            L_eeg = criterion(eeg_out, lbl_b)
            L_fnirs = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out, lbl_b)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss += L_efgf.item()
            train_cafe_loss += L_cafe.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()

        train_acc = train_correct / len(train_loader.dataset)

        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()

        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(train_loader)
            print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss/n_batches:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch} "
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")
            break

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (weight_decay={weight_decay})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")

    with torch.no_grad():
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()
    print(f"\nLearned final-combination weights (after softmax): "
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")

    # NEW: this run's train/val curve summary, to directly diagnose whether the overfitting
    # signature (best val at epoch 1-2, train acc climbing well past val) is reduced at this
    # weight_decay value -- the exact question Krogh & Hertz's theory predicts this experiment
    # should answer.
    overfit_gap = (train_acc * 100) - test_acc  # NEW: simple train-vs-test accuracy gap, a direct overfitting indicator
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")

    results = {
        'weight_decay': weight_decay,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': float(learned_weights[0]),
            'w_fnirs': float(learned_weights[1]),
            'w_fusion': float(learned_weights[2]),
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_wd{weight_decay}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE -- runs the weight-decay grid search
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Weight decay grid to test: {WEIGHT_DECAY_GRID}")

    all_grid_results = []  # NEW
    for wd in WEIGHT_DECAY_GRID:  # NEW: loop over the grid-search candidates
        result = run_training(wd, device)
        all_grid_results.append(result)

    # NEW: summary table across the whole weight_decay grid, so the best candidate (by test
    # accuracy AND by the smallest train-test overfitting gap) can be identified at a glance,
    # per Krogh & Hertz's point that the optimal decay coefficient must be found empirically.
    print("\n" + "=" * 70)
    print("WEIGHT DECAY GRID SEARCH SUMMARY")
    print("=" * 70)
    print(f"{'weight_decay':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'overfit_gap':>12}")
    for r in all_grid_results:
        print(f"{r['weight_decay']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['overfitting_gap']:>+11.2f}pp")

    best_by_test_acc = max(all_grid_results, key=lambda r: r['test_accuracy_final'])
    best_by_gap = min(all_grid_results, key=lambda r: r['overfitting_gap'])
    print(f"\nBest weight_decay by test accuracy: {best_by_test_acc['weight_decay']} "
          f"({best_by_test_acc['test_accuracy_final']:.2f}%)")
    print(f"Best weight_decay by smallest overfitting gap: {best_by_gap['weight_decay']} "
          f"({best_by_gap['overfitting_gap']:+.2f}pp)")

    summary_path = os.path.join(PROCESSED_DATA_PATH, 'weight_decay_grid_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(all_grid_results, f, indent=2)
    print(f"\nFull grid-search summary saved to: {summary_path}")

    print("\nCleanup completed. Temporary disk files removed.")

Running on Device: cuda
Weight decay grid to test: [0.02, 0.05, 0.1]

######################################################################
# WEIGHT_DECAY = 0.02
######################################################################
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025

In [3]:
#Tsfnet combined regularization
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

def pearson_correlation_loss(a, b, eps=1e-8):
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    """
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    a_centered = a_flat - a_flat.mean()
    b_centered = b_flat - b_flat.mean()
    numerator = (a_centered * b_centered).sum()
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps
    pcc = numerator / denominator
    return 1.0 - torch.abs(pcc)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    # NEW (combined-regularization experiment, per chat history -- targeted EEG dropout): added
    # a SEPARATE eeg_dropout parameter, higher than the base `dropout` used by every other layer
    # (fNIRS branch, EFGF, CAFE). Rationale: the 3-seed variance check (project report Section
    # 9.2.1) found EEG-branch-only accuracy had ~10x the standard deviation of the other three
    # metrics (std=6.68% vs ~2% for Test/fNIRS/Fusion), indicating the EEG branch is the most
    # unstable/overfitting-prone part of the model. Srivastava, N., Hinton, G., Krizhevsky, A.,
    # Sutskever, I., & Salakhutdinov, R. (2014). "Dropout: A Simple Way to Prevent Neural Networks
    # from Overfitting." Journal of Machine Learning Research, 15(1), 1929-1958 -- establishes that
    # dropout rate should be tuned per-layer/per-branch based on that component's overfitting risk,
    # not applied uniformly across a heterogeneous architecture. This project already tested a
    # "dropout mostaghel har shakhe" (independent per-branch dropout) experiment earlier (project
    # report Section 4.3) with an unequal split that turned out to be within noise; this attempt is
    # methodologically the same idea but now COMBINED with weight_decay=0.05 (the best grid-search
    # candidate) and label smoothing, rather than tested in isolation -- since weight_decay alone was
    # just shown NOT to fix the epoch-1 overfitting signature.
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5, eeg_dropout=0.65):  # CHANGED: added eeg_dropout parameter (default 0.65 > base 0.5)
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(eeg_dropout)  # CHANGED: dropout -> eeg_dropout (targeted, higher rate)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(eeg_dropout)  # CHANGED: dropout -> eeg_dropout (targeted, higher rate)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch (kept at the BASE dropout rate -- only EEG showed the outsized variance
        # in the 3-seed check, so only EEG's dropout is strengthened, per the single-variable-
        # isolation discipline used throughout this project)
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # EFGF
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)
        self.alpha = nn.Parameter(torch.tensor(0.0))
        self.gamma = nn.Parameter(torch.tensor(0.0))
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)

        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=11)
        self.nirs_seq_proj = nn.Linear(32, 64)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate2 = nn.Linear(128, 64)

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        # self.dropout = nn.Dropout(dropout)
        # CHANGED: split the single final-classifier dropout into two separate modules --
        # self.eeg_out_dropout (targeted, higher rate, applied only before fc_eeg) and
        # self.dropout (base rate, applied before fc_fnirs and fc_fusion as before). This is the
        # SAME targeted-EEG-dropout idea applied consistently at the classifier head, not just
        # inside the EEG conv branch above.
        self.eeg_out_dropout = nn.Dropout(eeg_dropout)  # NEW: targeted, higher-rate dropout for the EEG classifier head specifically
        self.dropout = nn.Dropout(dropout)  # CHANGED: kept at base rate, now used only for fnirs_out/fusion_out
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))

        # fNIRS branch process
        n_seg = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)

        y = y.reshape(batch_size, n_seg, 32, 8, 4, 4)

        nirs_feat_map = y.mean(dim=(1, 3))
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))

        y_spatial = y.mean(dim=(3, 4, 5))

        # EFGF Block
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn

        gamma = torch.clamp(self.gamma, 0, 1)
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))

        # CAFE Block
        nirs_seq = y_spatial
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)
        nirs_proj = self.pos_encoder(nirs_seq_proj)

        fused_seq = fused_feat.unsqueeze(1)
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)

        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        nirs_query = nirs_feat.unsqueeze(1)
        fused_kv = fused_feat.unsqueeze(1)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat

        # eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        eeg_out = self.fc_eeg(self.eeg_out_dropout(eeg_feat))  # CHANGED: uses the targeted, higher-rate eeg_out_dropout instead of the shared base dropout
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced


# ============================================================================
# COMBINED OVERFITTING-FIX EXPERIMENT (per chat history advisor recommendation)
# ============================================================================
# The prior weight_decay grid search (2e-2, 5e-2, 1e-1) showed the epoch-1 overfitting
# signature was UNCHANGED across all three candidates (best val always at epoch 1, final
# train_acc always ~87%, overfit_gap staying in a narrow 31.58-33.21pp band) -- i.e.
# weight_decay ALONE does not fix the structural overfitting problem. Per the advisor's
# recommendation, this script fixes weight_decay at the best empirical candidate (0.05) and
# COMBINES it with three additional, independently-referenced regularization techniques,
# testing them together (not in isolation) since the single-variable weight_decay experiment
# already showed insufficiency on its own:
#
# 1) Label smoothing (Szegedy, C., Vanhoucke, V., Ioffe, S., Shlens, J., & Wojna, Z. (2016).
#    "Rethinking the Inception Architecture for Computer Vision." CVPR, 2818-2826.) --
#    softens the one-hot target distribution, discouraging the classifier from becoming
#    overconfident on (possibly noisy) training-set labels.
# 2) Reduced early-stopping patience (Prechelt, L. (1998). "Early stopping -- but when?" In
#    Neural Networks: Tricks of the Trade, LNCS vol 1524, Springer.) -- since best val is
#    consistently reached at epoch 1 and never improves again, patience=15 wastes 15 epochs
#    of further overfitting; reduced to 6 here.
# 3) Targeted EEG-branch dropout (Srivastava et al. 2014, JMLR) -- see the __init__ docstring
#    comment above the eeg_dropout parameter for the full rationale (EEG branch's outsized
#    variance in the earlier 3-seed check).
#
# weight_decay itself is fixed at 0.05 (empirically the best of the three grid-search
# candidates), not re-searched here, per the advisor's explicit instruction not to spend more
# time on a finer weight_decay grid until the epoch-1 pattern itself changes.

WEIGHT_DECAY = 0.05          # FIXED: best candidate from the prior grid search (project report)
LABEL_SMOOTHING = 0.1        # NEW: Szegedy et al. 2016
EEG_DROPOUT = 0.65           # NEW: Srivastava et al. 2014, targeted higher rate for the EEG branch
BASE_DROPOUT = 0.5           # unchanged base rate for fNIRS/EFGF/CAFE
EARLY_STOP_PATIENCE = 6      # CHANGED: 15 -> 6, per Prechelt 1998 (best val is always reached at epoch 1)

LAMBDA_EEG = 0.2
LAMBDA_FNIRS = 0.2
LAMBDA_EFGF_REG = 1.0
LAMBDA_CAFE_REG = 1.0
EPOCHS = 100


def run_training(device):
    print("\n" + "#" * 70)
    print("# COMBINED OVERFITTING-FIX EXPERIMENT")
    print(f"# weight_decay={WEIGHT_DECAY}, label_smoothing={LABEL_SMOOTHING}, "
          f"eeg_dropout={EEG_DROPOUT}, early_stop_patience={EARLY_STOP_PATIENCE}")
    print("#" * 70)

    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=BASE_DROPOUT, eeg_dropout=EEG_DROPOUT).to(device)  # CHANGED: passes eeg_dropout

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    # criterion = nn.CrossEntropyLoss()
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)  # CHANGED: added label_smoothing (Szegedy et al. 2016)

    best_val_acc = 0.0
    epochs_no_improve = 0
    ckpt_path = os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet_combined_regularization.pth')

    print("STARTING TRAINING")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)

            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred = criterion(pred, lbl_b)
            L_eeg = criterion(eeg_out, lbl_b)
            L_fnirs = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out, lbl_b)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss += L_efgf.item()
            train_cafe_loss += L_cafe.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()

        train_acc = train_correct / len(train_loader.dataset)

        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()

        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_epoch = epoch  # NEW: track which epoch actually produced the best val, for the epoch-1 diagnostic
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # NEW: print EVERY epoch (not just every 5th) since EARLY_STOP_PATIENCE is now much
        # smaller (6) -- with the old "every 5 epochs" print cadence, a run stopping at epoch
        # ~7-10 would show almost no intermediate diagnostic information.
        n_batches = len(train_loader)
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss/n_batches:.4f} | "
              f"Class: {train_class_loss/n_batches:.4f} | "
              f"EFGF: {train_efgf_loss/n_batches:.4f} | "
              f"CAFE: {train_cafe_loss/n_batches:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch} "
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")
            break

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION (combined regularization)")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")

    with torch.no_grad():
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()
    print(f"\nLearned final-combination weights (after softmax): "
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")

    overfit_gap = (train_acc * 100) - test_acc
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")
    print(f"Best val was reached at epoch: {best_val_epoch}  "
          f"(diagnostic: has this moved past epoch 1-2, unlike the weight_decay-only grid search?)")

    results = {
        'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING,
        'eeg_dropout': EEG_DROPOUT,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'best_val_epoch': best_val_epoch,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': float(learned_weights[0]),
            'w_fnirs': float(learned_weights[1]),
            'w_fusion': float(learned_weights[2]),
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, 'results_combined_regularization.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()

    return results


# COMPARISON REFERENCE: the wd-only grid search result at wd=0.05 (best candidate), for direct
# before/after comparison, taken from the prior experiment's console output.
REFERENCE_WD_ONLY_RESULT = {
    'weight_decay': 0.05,
    'test_accuracy_final': 56.33,
    'best_val_epoch': 1,
    'overfitting_gap': 31.58,
}


if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")

    result = run_training(device)

    print("\n" + "=" * 70)
    print("BEFORE / AFTER COMPARISON (weight_decay-only vs. combined regularization)")
    print("=" * 70)
    print(f"{'metric':>22} | {'wd-only (0.05)':>16} | {'combined':>16}")
    print(f"{'test_accuracy':>22} | {REFERENCE_WD_ONLY_RESULT['test_accuracy_final']:>15.2f}% | {result['test_accuracy_final']:>15.2f}%")
    print(f"{'best_val_epoch':>22} | {REFERENCE_WD_ONLY_RESULT['best_val_epoch']:>16} | {result['best_val_epoch']:>16}")
    print(f"{'overfitting_gap':>22} | {REFERENCE_WD_ONLY_RESULT['overfitting_gap']:>+15.2f}pp | {result['overfitting_gap']:>+15.2f}pp")

    if result['best_val_epoch'] > REFERENCE_WD_ONLY_RESULT['best_val_epoch']:
        print("\n--> The epoch-1 overfitting signature has SHIFTED: best val now occurs later in training.")
        print("    This is the qualitative change needed before proceeding to 3-seed validation.")
    else:
        print("\n--> Best val is STILL reached at epoch 1 (or earlier than the reference).")
        print("    The combined regularization did not change the qualitative overfitting pattern;")
        print("    further/different interventions (e.g. Mixup, SWA) should be considered before")
        print("    committing to 3-seed validation of this configuration.")

    print("\nCleanup completed. Temporary disk files removed.")

Running on Device: cuda

######################################################################
# COMBINED OVERFITTING-FIX EXPERIMENT
# weight_decay=0.05, label_smoothing=0.1, eeg_dropout=0.65, early_stop_patience=6
######################################################################
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Sca

In [2]:
#Tsfnet weight decay gridsearch
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        # self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn1 = nn.BatchNorm3d(16)
        # self.nirs_drop1 = nn.Dropout(dropout)
        # self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn2 = nn.BatchNorm3d(32)
        # self.nirs_drop2 = nn.Dropout(dropout)
        # self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        # self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # CHANGED (Run H, 11-segment fNIRS -- paper Table 1 compliance, verified directly against
        # the paper's full text, Sec. 2.3 and Table 1): input is now (batch, 11, 16, 16, 30, 2) --
        # 11 segments (current + subsequent 10, per "each EEG time window was paired with the
        # current and subsequent 10 fNIRS segments... each fNIRS sample had a shape of
        # 11 x 36 x 30 x 2"), each segment 3s long (30 samples @ 10Hz), 2 channels (HbO/HbR).
        # CRITICAL per Table 1: the segment axis (11) is NOT touched by either conv layer -- only
        # the spatial (16x16) and per-segment-internal-time (30) axes are reduced. This is achieved
        # by folding the segment axis into the batch dimension before each conv (so the same
        # Conv3d weights are applied independently, per-segment, exactly like a shared/distributed
        # convolution), then folding it back out afterward. This directly fixes the bug found in a
        # reviewed third-party implementation, which applied a plain Conv3d across all three of
        # (segment, spatial, spatial) jointly, silently shrinking the segment axis from 11 to 2 and
        # requiring a nonsensical repeat+truncate patch downstream.
        # Table 1 states: Conv1 kernel 4x4x6, stride 2x2x2, 16 channels -> output 11x8x8x15x16
        # (segment x H x W x T x channels); Conv2 kernel 2x2x3, stride 2x2x2, 32 channels -> output
        # 11x4x4x8x32. Padding values are NOT stated explicitly in the fetched paper text (only
        # kernel/stride/output dims were) -- the padding below (2,1,1) and (1,0,0) were derived to
        # reproduce exactly those stated output dimensions, not directly quoted from the paper.
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))  # NEW: (T,H,W) order, matches EEG branch's convention; 30->15(T), 16->8(H,W)
        self.nirs_bn1 = nn.BatchNorm3d(16)  # NEW
        self.nirs_drop1 = nn.Dropout(dropout)  # NEW
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))  # NEW: 15->8(T), 8->4(H,W)
        self.nirs_bn2 = nn.BatchNorm3d(32)  # NEW
        self.nirs_drop2 = nn.Dropout(dropout)  # NEW
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack now reduces the 16x16 spatial grid to 4x4 (16->8->4),
        # matching the EEG branch's spatial resolution, same as before -- only the input/conv
        # parameters changed, not this downstream sizing.
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        # self.pos_encoder = PositionalEncoding(64, max_len=100)
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        self.pos_encoder = PositionalEncoding(64, max_len=11)  # CHANGED: 100 -> 11
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        # y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        # y = self.nirs_drop1(y)
        # y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        # y = self.nirs_drop2(y)
        #
        # y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_map = y.mean(dim=2)
        # nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))
        # CHANGED (Run H, 11-segment fNIRS -- Table 1 compliance): nirs arrives as
        # (batch, 11, 16, 16, 30, 2) = (batch, segment, H, W, T, channels) from the dataset. To
        # apply the SAME Conv3d weights to each of the 11 segments independently (i.e. the segment
        # axis must NOT be touched by the conv, per Table 1's stated output dims), the segment axis
        # is folded into the batch dimension before the conv stack, then unfolded afterward.
        n_seg = nirs.size(1)  # NEW: 11
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)  # NEW: (batch,11,16,16,30,2) -> (batch,11,channels=2,T=30,H=16,W=16)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)  # NEW: fold segment into batch -> (batch*11, 2, 30, 16, 16)

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # NEW: (batch*11, 16, 15, 8, 8)
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))  # NEW: (batch*11, 32, 8, 4, 4)
        y = self.nirs_drop2(y)

        y = y.reshape(batch_size, n_seg, 32, 8, 4, 4)  # NEW: unfold segment axis back out -> (batch, 11, 32, T'=8, H'=4, W'=4)

        # NEW: nirs_feat_map -- a SINGLE spatial map for the whole window (used by EFGF and by the
        # fNIRS classifier head), obtained by averaging over BOTH the segment axis and the
        # per-segment-internal-time axis (mirroring eeg_feat_map's GAP-over-time above).
        nirs_feat_map = y.mean(dim=(1, 3))  # NEW: (batch,11,32,8,4,4) -> mean over dim1(seg) & dim3(T') -> (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # NEW: the REAL 11-segment sequence for CAFE (replacing the old fabricated/proxy sequence).
        # Average over the per-segment-internal-time and spatial axes, keeping the segment axis
        # intact as the genuine temporal sequence the paper's CAFE layer is designed to attend over.
        y_spatial = y.mean(dim=(3, 4, 5))  # NEW: (batch,11,32,8,4,4) -> mean over T',H',W' -> (batch,11,32)

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        # nirs_seq = y_spatial.transpose(1, 2)
        # CHANGED (Run H, 11-segment fNIRS): y_spatial above is already (batch, 11, 32) -- i.e.
        # (batch, segment/sequence, channels) -- since it was constructed directly in that order
        # above, unlike the old y_spatial (batch, 32, T) which needed a transpose to get the
        # sequence axis into the middle position for nirs_seq_proj/pos_encoder/attention.
        nirs_seq = y_spatial  # CHANGED: no transpose needed, already (batch, 11, 32)
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
WEIGHT_DECAY_GRID = [5e-2]  # CHANGED: was [2e-2, 5e-2, 1e-1] (3-way grid search) -- narrowed to the single
# best empirical candidate (0.05) from that grid search, per chat history: the grid search itself
# showed the epoch-1 overfitting signature was UNCHANGED across all three values (best val always
# at epoch 1, overfit_gap staying in a narrow 31.58-33.21pp band), so further weight_decay search
# is not useful on its own. 0.05 is kept as a clean, single baseline for the diagnostic tooling
# added below (per-branch learning curve), per the explicit instruction to use this as "baseline
# تمیز و ساده" before adding diagnostics.

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)
EPOCHS = 100
EARLY_STOP_PATIENCE = 15


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device):  # NEW
    print("\n" + "#" * 70)
    print(f"# WEIGHT_DECAY = {weight_decay}")
    print("#" * 70)

    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # CHANGED: weight_decay is now the function parameter (grid-search candidate) instead of the
    # hard-coded 2e-2 -- lr, betas, and every other AdamW default remain exactly as before.
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=weight_decay)  # CHANGED: 2e-2 -> weight_decay (grid-search variable)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs_no_improve = 0
    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_wd{weight_decay}.pth')  # CHANGED: tagged per weight_decay so parallel/successive grid runs don't overwrite each other

    # NEW (diagnostic tool, per chat history -- Per-Branch Learning Curve): tracks EEG-only,
    # fNIRS-only, and Fusion-only accuracy SEPARATELY, for both train and val, at every epoch --
    # not just the final fused prediction's accuracy. This directly answers the diagnostic
    # question raised in the prior analysis: is the EEG branch's near-chance accuracy (~35%)
    # caused by classic overfitting (train/val gap widening over epochs, fixable by more
    # regularization) or by an intrinsic information-content ceiling (train and val both staying
    # near chance throughout, NOT fixable by regularization -- which is what the failed
    # eeg_dropout=0.65 experiment already hinted at). The three branch outputs (eeg_out,
    # fnirs_out, fusion_out) were already computed for free every forward pass (used for the
    # branch loss terms L_eeg/L_fnirs/L_fusion); this only adds the bookkeeping to record their
    # accuracy per epoch instead of discarding it.
    branch_history = {  # NEW
        'epoch': [], 'train_acc': [], 'val_acc': [],
        'train_acc_eeg': [], 'train_acc_fnirs': [], 'train_acc_fusion': [],
        'val_acc_eeg': [], 'val_acc_fnirs': [], 'val_acc_fusion': [],
    }  # NEW

    print("STARTING TRAINING")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0
        train_correct_eeg, train_correct_fnirs, train_correct_fusion = 0, 0, 0  # NEW: per-branch train correct counts

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)

            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred = criterion(pred, lbl_b)
            L_eeg = criterion(eeg_out, lbl_b)
            L_fnirs = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out, lbl_b)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss += L_efgf.item()
            train_cafe_loss += L_cafe.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            # NEW: per-branch train accuracy bookkeeping -- eeg_out/fnirs_out/fusion_out are
            # already computed above for the branch loss terms; this just also counts their
            # argmax-correct predictions, at zero extra forward-pass cost.
            train_correct_eeg += (eeg_out.argmax(1) == lbl_b).sum().item()      # NEW
            train_correct_fnirs += (fnirs_out.argmax(1) == lbl_b).sum().item()  # NEW
            train_correct_fusion += (fusion_out.argmax(1) == lbl_b).sum().item()  # NEW

        train_acc = train_correct / len(train_loader.dataset)
        train_acc_eeg = train_correct_eeg / len(train_loader.dataset)      # NEW
        train_acc_fnirs = train_correct_fnirs / len(train_loader.dataset)  # NEW
        train_acc_fusion = train_correct_fusion / len(train_loader.dataset)  # NEW

        # validation
        model.eval()
        val_correct = 0
        val_correct_eeg, val_correct_fnirs, val_correct_fusion = 0, 0, 0  # NEW: per-branch val correct counts
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: also unpack eeg_out/fnirs_out/fusion_out (previously discarded via "_"), needed for per-branch val accuracy below
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                val_correct_eeg += (eeg_out.argmax(1) == lbl_b).sum().item()        # NEW
                val_correct_fnirs += (fnirs_out.argmax(1) == lbl_b).sum().item()    # NEW
                val_correct_fusion += (fusion_out.argmax(1) == lbl_b).sum().item()  # NEW

        val_acc = val_correct / len(val_loader.dataset)
        val_acc_eeg = val_correct_eeg / len(val_loader.dataset)          # NEW
        val_acc_fnirs = val_correct_fnirs / len(val_loader.dataset)      # NEW
        val_acc_fusion = val_correct_fusion / len(val_loader.dataset)    # NEW
        scheduler.step(val_acc)

        # NEW: record this epoch's full per-branch train/val accuracy in branch_history, for the
        # per-branch learning curve (saved to disk after training, see below).
        branch_history['epoch'].append(epoch)                              # NEW
        branch_history['train_acc'].append(train_acc * 100)                # NEW
        branch_history['val_acc'].append(val_acc * 100)                    # NEW
        branch_history['train_acc_eeg'].append(train_acc_eeg * 100)        # NEW
        branch_history['train_acc_fnirs'].append(train_acc_fnirs * 100)    # NEW
        branch_history['train_acc_fusion'].append(train_acc_fusion * 100)  # NEW
        branch_history['val_acc_eeg'].append(val_acc_eeg * 100)            # NEW
        branch_history['val_acc_fnirs'].append(val_acc_fnirs * 100)        # NEW
        branch_history['val_acc_fusion'].append(val_acc_fusion * 100)      # NEW

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # NEW: print EVERY epoch (not just every 5th) -- the per-branch diagnostic needs the
        # full epoch-by-epoch trajectory visible, not a sparse sample, to correctly judge whether
        # a branch's train/val gap widens gradually (classic overfitting) or stays flat near
        # chance from epoch 1 (information-content ceiling, per the prior eeg_dropout finding).
        n_batches = len(train_loader)
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss/n_batches:.4f} | "
              f"Class: {train_class_loss/n_batches:.4f} | "
              f"EFGF: {train_efgf_loss/n_batches:.4f} | "
              f"CAFE: {train_cafe_loss/n_batches:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")
        print(f"           | Per-branch (train/val): "  # NEW
              f"EEG {train_acc_eeg*100:.2f}%/{val_acc_eeg*100:.2f}% | "  # NEW
              f"fNIRS {train_acc_fnirs*100:.2f}%/{val_acc_fnirs*100:.2f}% | "  # NEW
              f"Fusion {train_acc_fusion*100:.2f}%/{val_acc_fusion*100:.2f}%")  # NEW

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch} "
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")
            break

    # NEW: save the full per-branch learning curve to disk (one JSON per weight_decay run), so
    # it can be plotted/inspected afterward without re-running training.
    branch_history_path = os.path.join(PROCESSED_DATA_PATH, f'branch_learning_curve_wd{weight_decay}.json')  # NEW
    with open(branch_history_path, 'w') as f:  # NEW
        json.dump(branch_history, f, indent=2)  # NEW
    print(f"\nPer-branch learning curve saved to: {branch_history_path}")  # NEW

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (weight_decay={weight_decay})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")

    with torch.no_grad():
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()
    print(f"\nLearned final-combination weights (after softmax): "
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")

    # NEW: this run's train/val curve summary, to directly diagnose whether the overfitting
    # signature (best val at epoch 1-2, train acc climbing well past val) is reduced at this
    # weight_decay value -- the exact question Krogh & Hertz's theory predicts this experiment
    # should answer.
    overfit_gap = (train_acc * 100) - test_acc  # NEW: simple train-vs-test accuracy gap, a direct overfitting indicator
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")

    results = {
        'weight_decay': weight_decay,
        'branch_learning_curve_path': branch_history_path,  # NEW: pointer to the per-branch epoch-by-epoch JSON saved above
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': float(learned_weights[0]),
            'w_fnirs': float(learned_weights[1]),
            'w_fusion': float(learned_weights[2]),
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_wd{weight_decay}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE -- runs the weight-decay grid search
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Weight decay grid to test: {WEIGHT_DECAY_GRID}")

    all_grid_results = []  # NEW
    for wd in WEIGHT_DECAY_GRID:  # NEW: loop over the grid-search candidates
        result = run_training(wd, device)
        all_grid_results.append(result)

    # NEW: summary table across the whole weight_decay grid, so the best candidate (by test
    # accuracy AND by the smallest train-test overfitting gap) can be identified at a glance,
    # per Krogh & Hertz's point that the optimal decay coefficient must be found empirically.
    print("\n" + "=" * 70)
    print("WEIGHT DECAY GRID SEARCH SUMMARY")
    print("=" * 70)
    print(f"{'weight_decay':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'overfit_gap':>12}")
    for r in all_grid_results:
        print(f"{r['weight_decay']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['overfitting_gap']:>+11.2f}pp")

    best_by_test_acc = max(all_grid_results, key=lambda r: r['test_accuracy_final'])
    best_by_gap = min(all_grid_results, key=lambda r: r['overfitting_gap'])
    print(f"\nBest weight_decay by test accuracy: {best_by_test_acc['weight_decay']} "
          f"({best_by_test_acc['test_accuracy_final']:.2f}%)")
    print(f"Best weight_decay by smallest overfitting gap: {best_by_gap['weight_decay']} "
          f"({best_by_gap['overfitting_gap']:+.2f}pp)")

    summary_path = os.path.join(PROCESSED_DATA_PATH, 'weight_decay_grid_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(all_grid_results, f, indent=2)
    print(f"\nFull grid-search summary saved to: {summary_path}")

    print("\nCleanup completed. Temporary disk files removed.")

Running on Device: cuda
Weight decay grid to test: [0.05]

######################################################################
# WEIGHT_DECAY = 0.05
######################################################################
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 

In [2]:
#Check eeg quality
"""
check_eeg_quality.py
--------------------
بررسی می‌کند که آیا مرحله preprocessing باعث ضعف EEG شده یا نه.

سه سوال اصلی:
  1. آیا EEG spatially-mapped (16x16) اطلاعات کافی دارد؟
  2. آیا normalization مشکل ایجاد کرده؟
  3. آیا EEG discrimination بین 3 class دارد؟

روی چند subject اجرا کنید — نیازی به همه 26 نیست.
"""

import os
import numpy as np

# ── مسیر داده‌ها ──────────────────────────────────────────────────────
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"
CHECK_SUBJECTS = ["VP001", "VP002", "VP003", "VP005"]   # نمونه چند subject

print("=" * 65)
print("EEG QUALITY DIAGNOSTIC")
print("=" * 65)

all_issues = []

for subject_id in CHECK_SUBJECTS:
    sub_dir = os.path.join(PROCESSED_DATA_PATH, subject_id)
    eeg_path = os.path.join(sub_dir, f"{subject_id}_eeg_windows.npy")
    labels_path = os.path.join(sub_dir, f"{subject_id}_labels.npy")

    if not os.path.exists(eeg_path):
        print(f"\n[{subject_id}] SKIP — file not found")
        continue

    eeg = np.load(eeg_path)       # shape: (N, 16, 16, 600)
    labels = np.load(labels_path) # shape: (N,)

    print(f"\n{'─'*65}")
    print(f"[{subject_id}]  shape={eeg.shape}  labels={np.bincount(labels)}")

    # ── CHECK 1: آیا داده‌ها float32 معقولی دارند؟ ──────────────────
    print(f"\n  CHECK 1 — Basic statistics:")
    print(f"    dtype:  {eeg.dtype}")
    print(f"    global mean:  {eeg.mean():.4f}  (expected ≈ 0 after z-score)")
    print(f"    global std:   {eeg.std():.4f}   (expected ≈ 1 after z-score)")
    print(f"    min: {eeg.min():.3f}   max: {eeg.max():.3f}")
    print(f"    nan count: {np.isnan(eeg).sum()}")
    print(f"    inf count: {np.isinf(eeg).sum()}")

    if abs(eeg.mean()) > 0.5:
        issue = f"{subject_id}: EEG mean={eeg.mean():.3f} — normalization ممکن است اشتباه باشد"
        all_issues.append(issue)
        print(f"    ⚠ ISSUE: mean خیلی دور از صفر")
    if eeg.std() < 0.1 or eeg.std() > 10:
        issue = f"{subject_id}: EEG std={eeg.std():.3f} — خارج از محدوده معقول"
        all_issues.append(issue)
        print(f"    ⚠ ISSUE: std خارج از محدوده 0.1-10")

    # ── CHECK 2: آیا grid خالی یا همه‌صفر نیست؟ ────────────────────
    print(f"\n  CHECK 2 — Spatial grid (16x16) quality:")
    # برای هر pixel، آیا variance دارد؟
    pixel_var = eeg.var(axis=(0, 3))  # (16, 16) — variance over windows and time
    zero_pixels = (pixel_var < 1e-6).sum()
    total_pixels = 16 * 16
    print(f"    zero-variance pixels: {zero_pixels}/{total_pixels} "
          f"({100*zero_pixels/total_pixels:.1f}%)")
    print(f"    pixel_var min={pixel_var.min():.4f}  max={pixel_var.max():.4f}  "
          f"mean={pixel_var.mean():.4f}")

    if zero_pixels > total_pixels * 0.3:
        issue = f"{subject_id}: {zero_pixels} zero-variance pixels ({100*zero_pixels/total_pixels:.0f}%) — spatial mapping ممکن است مشکل داشته باشد"
        all_issues.append(issue)
        print(f"    ⚠ ISSUE: خیلی زیاد zero-variance pixels")

    # ── CHECK 3: آیا EEG بین کلاس‌ها تفاوت دارد؟ ──────────────────
    print(f"\n  CHECK 3 — Class separability (0-back vs 3-back):")
    class0 = eeg[labels == 0]
    class1 = eeg[labels == 1]
    class2 = eeg[labels == 2]

    # میانگین هر کلاس روی temporal axis → (N, 16, 16)
    mean0 = class0.mean(axis=(0, 3))  # (16, 16) — mean over windows and time
    mean1 = class1.mean(axis=(0, 3))
    mean2 = class2.mean(axis=(0, 3))

    diff_02 = np.abs(mean0 - mean2).mean()  # فاصله mean بین 0-back و 3-back
    diff_01 = np.abs(mean0 - mean1).mean()
    diff_12 = np.abs(mean1 - mean2).mean()

    print(f"    |mean(0-back) - mean(2-back)| avg = {diff_01:.4f}")
    print(f"    |mean(0-back) - mean(3-back)| avg = {diff_02:.4f}")
    print(f"    |mean(2-back) - mean(3-back)| avg = {diff_12:.4f}")

    # Cohen's d بین 0-back و 3-back (روی flattened temporal signal)
    flat0 = class0.reshape(len(class0), -1).mean(axis=0)
    flat2 = class2.reshape(len(class2), -1).mean(axis=0)
    pooled_std = np.sqrt((class0.reshape(len(class0), -1).var(axis=0) +
                          class2.reshape(len(class2), -1).var(axis=0)) / 2 + 1e-8)
    cohens_d = np.abs(flat0 - flat2) / pooled_std
    print(f"    Cohen's d (0-back vs 3-back) mean = {cohens_d.mean():.4f}")
    print(f"    Cohen's d median = {np.median(cohens_d):.4f}")
    print(f"    Cohen's d > 0.2 (small effect): {(cohens_d > 0.2).sum()} / {len(cohens_d)} pixels")

    if cohens_d.mean() < 0.05:
        issue = f"{subject_id}: EEG Cohen's d={cohens_d.mean():.4f} — تقریباً هیچ تفاوت بین‌کلاسی ندارد"
        all_issues.append(issue)
        print(f"    ⚠ ISSUE: Cohen's d خیلی کوچک — EEG تقریباً بی‌تفاوت است بین کلاس‌ها")

    # ── CHECK 4: آیا temporal patterns معنادار است؟ ──────────────────
    print(f"\n  CHECK 4 — Temporal signal (mean over grid):")
    # میانگین مکانی → (N, 600) — بررسی اینکه آیا signal temporal structure دارد
    temporal_mean = eeg.mean(axis=(1, 2))  # (N, 600)
    autocorr_mean = np.array([np.corrcoef(temporal_mean[i, :-1],
                                           temporal_mean[i, 1:])[0,1]
                               for i in range(min(50, len(temporal_mean)))])
    print(f"    temporal autocorrelation (lag-1) mean = {autocorr_mean.mean():.4f}")
    print(f"    (expected > 0.3 for EEG, very low suggests signal loss)")

    if autocorr_mean.mean() < 0.1:
        issue = f"{subject_id}: EEG temporal autocorr={autocorr_mean.mean():.4f} — ممکن است signal به noise تبدیل شده باشد"
        all_issues.append(issue)
        print(f"    ⚠ ISSUE: autocorrelation خیلی پایین")

    # ── CHECK 5: مقایسه با fNIRS ──────────────────────────────────────
    fnirs_path = os.path.join(sub_dir, f"{subject_id}_fnirs_windows.npy")
    if os.path.exists(fnirs_path):
        print(f"\n  CHECK 5 — EEG vs fNIRS comparison:")
        fnirs = np.load(fnirs_path)  # (N, 11, 16, 16, 30, 2)

        fnirs_mean0 = fnirs[labels == 0].mean()
        fnirs_mean2 = fnirs[labels == 2].mean()
        fnirs_diff = abs(fnirs_mean2 - fnirs_mean0)

        print(f"    fNIRS |mean(0-back) - mean(3-back)| = {fnirs_diff:.6f}")
        print(f"    EEG   |mean(0-back) - mean(3-back)| = {diff_02:.6f}")
        ratio = fnirs_diff / (diff_02 + 1e-10)
        print(f"    ratio (fNIRS/EEG) = {ratio:.2f}")
        print(f"    (ratio >> 1 → fNIRS بیشتر تغییر می‌کند، ratio ≈ 1 → برابر)")

        if ratio > 5:
            print(f"    ⚠ fNIRS تفاوت‌پذیری بسیار بیشتری از EEG دارد")
            print(f"       این ممکن است طبیعی (N-back) یا preprocessing issue باشد")

# ── خلاصه نهایی ──────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("SUMMARY OF ISSUES")
print(f"{'='*65}")
if all_issues:
    for issue in all_issues:
        print(f"  ✗ {issue}")
    print(f"\nتعداد مشکلات: {len(all_issues)}")
else:
    print("  ✓ هیچ مشکل عمده‌ای در preprocessing یافت نشد")
    print("  → ضعف EEG احتمالاً ذاتی task N-back است، نه preprocessing")

print(f"\n{'='*65}")
print("تفسیر نتایج:")
print("  اگر Cohen's d > 0.1: EEG اطلاعات دارد (مشکل از مدل/training است)")
print("  اگر Cohen's d < 0.05: EEG signal ضعیف/از دست رفته (مشکل preprocessing)")
print("  اگر zero pixels > 30%: spatial mapping اشکال دارد")
print("  اگر autocorr < 0.1: signal به noise تبدیل شده")
print("  اگر std >> 1: normalization درست انجام نشده")

EEG QUALITY DIAGNOSTIC

─────────────────────────────────────────────────────────────────
[VP001]  shape=(1023, 16, 16, 600)  labels=[339 342 342]

  CHECK 1 — Basic statistics:
    dtype:  float32
    global mean:  -0.0004  (expected ≈ 0 after z-score)
    global std:   0.7672   (expected ≈ 1 after z-score)
    min: -17.656   max: 17.228
    nan count: 0
    inf count: 0

  CHECK 2 — Spatial grid (16x16) quality:
    zero-variance pixels: 98/256 (38.3%)
    pixel_var min=0.0000  max=1.0348  mean=0.5886
    ⚠ ISSUE: خیلی زیاد zero-variance pixels

  CHECK 3 — Class separability (0-back vs 3-back):
    |mean(0-back) - mean(2-back)| avg = 0.0012
    |mean(0-back) - mean(3-back)| avg = 0.0019
    |mean(2-back) - mean(3-back)| avg = 0.0007
    Cohen's d (0-back vs 3-back) mean = 0.0393
    Cohen's d median = 0.0199
    Cohen's d > 0.2 (small effect): 940 / 153600 pixels
    ⚠ ISSUE: Cohen's d خیلی کوچک — EEG تقریباً بی‌تفاوت است بین کلاس‌ها

  CHECK 4 — Temporal signal (mean over grid):
  